# Burned-area mapping: sampling, model training and evaluation

Create an event-level sample split; compare Random Forest, U-Net, Siamese and Prithvi models and summarize evaluation results.

Run the configuration cell first. See the repository README for data, dependencies and setup.


In [ ]:
# Shared project configuration (adjust for your environment).
import os
from pathlib import Path

# For Colab: mount Google Drive using the next cell, if needed.
# For local use: set FIRE_PROJECT_DIR to your own data directory.
PROJECT_DIR = Path(os.environ.get(
    "FIRE_PROJECT_DIR", "/content/drive/MyDrive/Fire detection"
)).expanduser()
INPUT_GPKG = PROJECT_DIR / "spain-portugal-topfires-2020-2024.gpkg"
OUTPUT_GPKG = PROJECT_DIR / "fire_collection_areas.gpkg"
EO_DIR = PROJECT_DIR / "eo_data"

print(f"Project directory: {PROJECT_DIR}")


In [ ]:
# Optional for Google Colab only; no existing directories are removed.
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


## Environment
Install dependencies once from `requirements.txt` and, for Prithvi, `requirements-prithvi.txt` in a fresh runtime. Avoid changing compiled packages midway through a running notebook.


# Read the geopackage top fire

# Load the saved burne/unburned geopackage

In [ ]:
import geopandas as gpd
from pathlib import Path

OUTPUT_GPKG = PROJECT_DIR / 'fire_collection_areas.gpkg'

# Load the layers from the GeoPackage
fire_polygons_loaded = gpd.read_file(OUTPUT_GPKG, layer="fire_polygons")
unburned_rings_loaded = gpd.read_file(OUTPUT_GPKG, layer="candidate_unburned_rings")
collection_extents_loaded = gpd.read_file(OUTPUT_GPKG, layer="collection_extents")

print("Loaded GeoPackage layers:")
print(f"  - fire_polygons: {len(fire_polygons_loaded)} features")
print(f"  - candidate_unburned_rings: {len(unburned_rings_loaded)} features")
print(f"  - collection_extents: {len(collection_extents_loaded)} features")

# Load saved S1/S2 rasters

## Install and import

In [ ]:
# ============================================================
# RASTERIO IMPORT CHECK
#
# The notebook intentionally does not run an unconstrained
# ``pip install rasterio`` because that can modify NumPy/SciPy
# dependencies in the active Colab runtime.
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

try:
    import rasterio
    from rasterio.merge import merge
except ImportError as exc:
    raise ImportError(
        "rasterio is not available in this runtime. Install it in a "
        "fresh environment before running the notebook; do not upgrade "
        "NumPy/SciPy in the middle of this workflow."
    ) from exc

print("Rasterio version:", rasterio.__version__)


## Find the exported rasters

In [ ]:

EO_DIR = PROJECT_DIR / 'eo_data'

if not EO_DIR.exists():
    raise FileNotFoundError(f"Folder not found: {EO_DIR}")

PRODUCTS = ["S2_pre", "S2_post", "S1_pre", "S1_post"]

BAND_NAMES = {
    "S2_pre": [
        "B2", "B3", "B4", "B5", "B6",
        "B7", "B8", "B8A", "B11", "B12",
        "SCL", "valid",
    ],
    "S2_post": [
        "B2", "B3", "B4", "B5", "B6",
        "B7", "B8", "B8A", "B11", "B12",
        "SCL", "valid",
    ],
    "S1_pre": ["VV", "VH", "n_obs", "valid"],
    "S1_post": ["VV", "VH", "n_obs", "valid"],
}

pattern = re.compile(
    r"^(fire_\d+)_(S2_pre|S2_post|S1_pre|S1_post)"
    r"_20m(?:-\d+-\d+)?\.tiff?$",
    re.IGNORECASE,
)

# catalog[fire_id][product] = list of raster paths
catalog = {}
unrecognized = []

for path in sorted(EO_DIR.iterdir()):
    if path.suffix.lower() not in {".tif", ".tiff"}:
        continue

    match = pattern.match(path.name)

    if not match:
        unrecognized.append(path.name)
        continue

    fire_id = match.group(1).lower()
    product = next(
        p for p in PRODUCTS
        if p.lower() == match.group(2).lower()
    )

    catalog.setdefault(fire_id, {}).setdefault(
        product, []
    ).append(path)

if not catalog:
    raise FileNotFoundError("No matching exported rasters found.")

summary = pd.DataFrame([
    {
        "fire_id": fire_id,
        **{
            product: len(files.get(product, []))
            for product in PRODUCTS
        },
        "complete": all(
            product in files for product in PRODUCTS
        ),
    }
    for fire_id, files in sorted(catalog.items())
])

display(summary)

if unrecognized:
    print("Unrecognized TIFF filenames:", unrecognized)

## Inspect metadata without loading pixel arrays

In [ ]:
metadata_rows = []

for fire_id, products in sorted(catalog.items()):
    for product, paths in products.items():
        for path in paths:
            with rasterio.open(path) as src:
                metadata_rows.append({
                    "fire_id": fire_id,
                    "product": product,
                    "filename": path.name,
                    "bands": src.count,
                    "width": src.width,
                    "height": src.height,
                    "crs": str(src.crs),
                    "pixel_size": src.res,
                    "nodata": src.nodata,
                    "size_MB": round(
                        path.stat().st_size / 1024**2, 2
                    ),
                })

raster_metadata = pd.DataFrame(metadata_rows)
display(raster_metadata)

## Define the raster loader

In [ ]:
from contextlib import ExitStack

def load_product(fire_id, product):
    paths = catalog.get(fire_id, {}).get(product)

    if not paths:
        raise FileNotFoundError(
            f"Missing {product} for {fire_id}"
        )

    with ExitStack() as stack:
        sources = [
            stack.enter_context(rasterio.open(path))
            for path in paths
        ]

        reference = sources[0]
        expected_bands = BAND_NAMES[product]

        for src in sources:
            if src.count != len(expected_bands):
                raise ValueError(
                    f"{src.name}: expected "
                    f"{len(expected_bands)} bands, "
                    f"found {src.count}"
                )

            if (
                src.crs != reference.crs
                or src.res != reference.res
                or src.nodata != reference.nodata
            ):
                raise ValueError(
                    f"Inconsistent raster pieces for "
                    f"{fire_id}, {product}"
                )

        if len(sources) == 1:
            data = reference.read(
                masked=True
            ).astype("float32").filled(np.nan)

            transform = reference.transform

        else:
            nodata = reference.nodata
            if nodata is None:
                nodata = -9999.0

            data, transform = merge(
                sources,
                nodata=nodata,
                dtype="float32",
            )

            data[data == nodata] = np.nan

        return {
            "data": data,
            "band_names": expected_bands,
            "crs": reference.crs,
            "transform": transform,
            "height": data.shape[1],
            "width": data.shape[2],
            "paths": [str(path) for path in paths],
        }


def load_fire(fire_id):
    rasters = {
        product: load_product(fire_id, product)
        for product in PRODUCTS
    }

    reference = rasters["S2_pre"]

    for product, raster in rasters.items():
        if (
            raster["crs"] != reference["crs"]
            or raster["transform"] != reference["transform"]
            or raster["data"].shape[1:] != reference["data"].shape[1:]
        ):
            raise ValueError(
                f"{fire_id}: {product} does not match "
                "the S2_pre grid."
            )

    print(f"{fire_id}: all four grids match.")

    return rasters


# PRE/POST IMAGE AND REFERENCE-MASK ILLUSTRATION

In [ ]:
# ============================================================
# BLOCK 16 — COMPACT PRE/POST IMAGE COMPARISON
# Requires Blocks 11–14: EO_DIR, catalog and load_product.
# ============================================================

import json
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from pathlib import Path
from rasterio.features import rasterize
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

FIGURE_DIR = PROJECT_DIR / 'print_layouts'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_GPKG = (
    str(PROJECT_DIR) + "/"
    "spain-portugal-topfires-2020-2024.gpkg"
)
REFERENCE_LAYER = "top_fires_2020_2024"

MANIFEST_FILE = EO_DIR / "satellite_selection_manifest.json"

FIRE_IDS = None                  # None = all available S2 pairs
FIRES_PER_FIGURE = 6
MAX_DISPLAY_PIXELS = 800
DPI = 300

TRUE_COLOR = ["B4", "B3", "B2"]
FALSE_COLOR = ["B12", "B8", "B4"]

# Fixed display ranges for all fires and both dates.
TRUE_RANGE = (0.0, 0.30)
FALSE_RANGE = (0.0, 0.50)

GAMMA = 1.2
SATURATION = 0.80

# Font sizes.
COLUMN_FONT_SIZE = 14
ROW_FONT_SIZE = 12
LEGEND_FONT_SIZE = 11
DATE_FONT_SIZE = 9

# Layout dimensions in inches.
PANEL_WIDTH = 2.0
COLUMN_GAP = 0.05
ROW_GAP = 0.18
LEFT_MARGIN = 1.0
RIGHT_MARGIN = 0.15
TOP_MARGIN = 0.85
BOTTOM_MARGIN = 0.85

COLUMN_TITLES = [
    "Pre-fire\nTrue color",
    "Pre-fire\nFalse color",
    "Post-fire\nTrue color",
    "Post-fire\nFalse color",
    "Reference\nperimeter mask",
]

INVALID_COLOR = "#cccccc"


# ------------------------------------------------------------
# 2. READ MANIFEST AND REFERENCE POLYGONS
# ------------------------------------------------------------

manifest = json.loads(MANIFEST_FILE.read_text())

site_info = {
    record["site_key"]: record
    for record in manifest["selected_sites"]
}

reference_polygons = gpd.read_file(
    REFERENCE_GPKG,
    layer=REFERENCE_LAYER,
)

if reference_polygons.crs is None:
    raise ValueError("Reference polygons have no CRS.")

if FIRE_IDS is None:
    FIRE_IDS = [
        fire_id
        for fire_id in sorted(catalog)
        if "S2_pre" in catalog[fire_id]
        and "S2_post" in catalog[fire_id]
    ]

if not FIRE_IDS:
    raise ValueError("No complete Sentinel-2 pairs were found.")


def get_reference_polygon(fire_id, target_crs):
    info = site_info[fire_id]

    matches = reference_polygons[
        (
            reference_polygons["area_name"].astype(str)
            == info["area_name"]
        )
        & (
            reference_polygons["country"].astype(str)
            == info["country"]
        )
    ]

    if len(matches) != 1:
        raise ValueError(
            f"{fire_id}: expected one reference polygon, "
            f"found {len(matches)}."
        )

    geometry = matches.to_crs(target_crs).geometry.iloc[0]

    if (
        geometry is None
        or geometry.is_empty
        or not geometry.is_valid
    ):
        raise ValueError(f"{fire_id}: invalid reference geometry.")

    return geometry


# ------------------------------------------------------------
# 3. DISPLAY FUNCTIONS
# ------------------------------------------------------------

def get_display_valid(raster, stride):
    valid_index = raster["band_names"].index("valid")

    valid = (
        raster["data"][valid_index, ::stride, ::stride] > 0.5
    )

    spectral = raster["data"][:10, ::stride, ::stride]

    return valid & np.isfinite(spectral).all(axis=0)


def make_rgb(raster, bands, stride, display_range):
    indexes = [
        raster["band_names"].index(band)
        for band in bands
    ]

    rgb = raster["data"][
        indexes, ::stride, ::stride
    ].transpose(1, 2, 0).copy()

    valid = get_display_valid(raster, stride)

    low, high = display_range

    if high <= low:
        raise ValueError("Display range must have high > low.")

    rgb = np.clip((rgb - low) / (high - low), 0, 1)
    rgb = np.nan_to_num(rgb, nan=0.0)
    rgb = rgb ** (1.0 / GAMMA)

    gray = (
        0.2126 * rgb[..., 0]
        + 0.7152 * rgb[..., 1]
        + 0.0722 * rgb[..., 2]
    )[..., None]

    rgb = gray + SATURATION * (rgb - gray)
    rgb = np.clip(rgb, 0, 1).astype("float32")

    rgb[~valid] = [0.8, 0.8, 0.8]

    return rgb


mask_cmap = ListedColormap(["black", "#b30000"])
mask_cmap.set_bad(INVALID_COLOR)


def prepare_fire_panels(fire_id):
    pre = load_product(fire_id, "S2_pre")
    post = load_product(fire_id, "S2_post")

    if (
        pre["crs"] != post["crs"]
        or pre["transform"] != post["transform"]
        or pre["data"].shape[1:] != post["data"].shape[1:]
    ):
        raise ValueError(
            f"{fire_id}: Sentinel-2 pre/post grids do not match."
        )

    stride = max(
        1,
        int(np.ceil(
            max(pre["height"], pre["width"])
            / MAX_DISPLAY_PIXELS
        )),
    )

    panels = [
        make_rgb(pre, TRUE_COLOR, stride, TRUE_RANGE),
        make_rgb(pre, FALSE_COLOR, stride, FALSE_RANGE),
        make_rgb(post, TRUE_COLOR, stride, TRUE_RANGE),
        make_rgb(post, FALSE_COLOR, stride, FALSE_RANGE),
    ]

    geometry = get_reference_polygon(fire_id, pre["crs"])

    reference_mask = rasterize(
        [(geometry, 1)],
        out_shape=(pre["height"], pre["width"]),
        transform=pre["transform"],
        fill=0,
        all_touched=False,
        dtype="uint8",
    )[::stride, ::stride]

    common_valid = (
        get_display_valid(pre, stride)
        & get_display_valid(post, stride)
    )

    panels.append(
        np.ma.array(
            reference_mask,
            mask=~common_valid,
        )
    )

    info = site_info[fire_id]
    display_height, display_width = panels[0].shape[:2]

    return {
        "fire_id": fire_id,
        "name": info["area_name"],
        "country": info["country"],
        "pre_date": info["acquisitions"]["S2_pre"]["date"],
        "post_date": info["acquisitions"]["S2_post"]["date"],
        "aspect_ratio": display_height / display_width,
        "panels": panels,
    }


# ------------------------------------------------------------
# 4. CREATE FIGURES
# ------------------------------------------------------------

for page_start in range(0, len(FIRE_IDS), FIRES_PER_FIGURE):
    page_ids = FIRE_IDS[
        page_start:page_start + FIRES_PER_FIGURE
    ]

    page_number = page_start // FIRES_PER_FIGURE + 1

    rows = []

    for fire_id in page_ids:
        print("Preparing:", fire_id)
        rows.append(prepare_fire_panels(fire_id))

    row_heights = [
        PANEL_WIDTH * row["aspect_ratio"]
        for row in rows
    ]

    figure_width = (
        LEFT_MARGIN
        + 5 * PANEL_WIDTH
        + 4 * COLUMN_GAP
        + RIGHT_MARGIN
    )

    figure_height = (
        TOP_MARGIN
        + sum(row_heights)
        + (len(rows) - 1) * ROW_GAP
        + BOTTOM_MARGIN
    )

    fig = plt.figure(
        figsize=(figure_width, figure_height),
        facecolor="white",
    )

    top_position = figure_height - TOP_MARGIN

    for row_number, (row, row_height) in enumerate(
        zip(rows, row_heights)
    ):
        bottom_position = top_position - row_height

        # Vertical label, centered beside the row.
        row_label = fig.text(
            (LEFT_MARGIN - 0.30) / figure_width,
            (bottom_position + row_height / 2) / figure_height,
            f"{row['name']}\n{row['country']}",
            rotation=90,
            rotation_mode="anchor",
            ha="center",
            va="center",
            fontsize=ROW_FONT_SIZE,
            fontweight="bold",
        )

        for column, panel in enumerate(row["panels"]):
            left_position = (
                LEFT_MARGIN
                + column * (PANEL_WIDTH + COLUMN_GAP)
            )

            ax = fig.add_axes([
                left_position / figure_width,
                bottom_position / figure_height,
                PANEL_WIDTH / figure_width,
                row_height / figure_height,
            ])

            if column < 4:
                ax.imshow(
                    panel,
                    interpolation="nearest",
                    aspect="equal",
                )

                date = (
                    row["pre_date"]
                    if column < 2
                    else row["post_date"]
                )

                ax.text(
                    0.02, 0.02, date,
                    transform=ax.transAxes,
                    fontsize=DATE_FONT_SIZE,
                    color="white",
                    ha="left",
                    va="bottom",
                    bbox={
                        "facecolor": "black",
                        "alpha": 0.65,
                        "edgecolor": "none",
                        "pad": 2,
                    },
                )

            else:
                ax.imshow(
                    panel,
                    cmap=mask_cmap,
                    vmin=0,
                    vmax=1,
                    interpolation="nearest",
                    aspect="equal",
                )

            if row_number == 0:
                ax.set_title(
                    COLUMN_TITLES[column],
                    fontsize=COLUMN_FONT_SIZE,
                    fontweight="bold",
                    pad=12,
                )

            ax.set_axis_off()

        # Fit long vertical labels within short rows.
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()

        available_height_px = (
            row_height * fig.dpi * 0.95
        )

        while (
            row_label.get_window_extent(renderer).height
            > available_height_px
            and row_label.get_fontsize() > 8
        ):
            row_label.set_fontsize(
                row_label.get_fontsize() - 0.5
            )
            fig.canvas.draw()

        top_position = bottom_position - ROW_GAP

    fig.legend(
        handles=[
            Patch(
                facecolor="#b30000",
                label="Inside fire perimeter",
            ),
            Patch(
                facecolor="black",
                label="Outside fire perimeter",
            ),
            Patch(
                facecolor=INVALID_COLOR,
                label="Invalid optical coverage",
            ),
        ],
        loc="lower center",
        bbox_to_anchor=(0.54, 0.015),
        ncol=3,
        frameon=False,
        prop={
            "size": LEGEND_FONT_SIZE,
            "weight": "bold",
        },
        handlelength=1.8,
        handleheight=1.0,
        columnspacing=1.5,
    )

    prefix = (
        FIGURE_DIR
        / f"fire_pre_post_panels_vertical_{page_number:02d}"
    )

    for extension in [".png", ".pdf"]:
        output_path = prefix.with_suffix(extension)

        fig.savefig(
            output_path,
            dpi=DPI,
            bbox_inches="tight",
            pad_inches=0.08,
            facecolor="white",
        )

        print("Saved:", output_path)

    plt.show()
    plt.close(fig)

    del rows

# COMMON SAMPLING AND EVENT-LEVEL VALIDATION DESIGN


In [ ]:
# ============================================================
# COMMON SAMPLING AND EVENT-LEVEL VALIDATION DESIGN
# ULTRA MEMORY-SAFE VERSION
#
# KEY FEATURES
# ------------
# - 8 train / 2 validation / 2 test event-level split
# - same sampling locations for all models
# - 240 samples per train/validation fire
# - 35% burned interior
# - 35% boundary
# - 30% unburned
# - 64 x 64 patches
# - 300 m minimum center spacing
# - minimum 90% valid EO pixels per patch
# - previous fire scars from preceding 2 years excluded
# - historical fire database queried locally from disk
# - NO loading of all 19,844 historical polygons
# - NO unary_union
# - NO np.argwhere over millions of pixels
# - each completed fire saved immediately
# - automatically resumes after Colab crash
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import re
import gc
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import pyogrio

from rasterio.features import rasterize
from rasterio.transform import xy
from rasterio.merge import merge
from rasterio.io import MemoryFile

from shapely import make_valid


# ============================================================
# 2. PATHS
# ============================================================

PROJECT_DIR = PROJECT_DIR

REFERENCE_GPKG = (
    PROJECT_DIR /
    "fire_collection_areas.gpkg"
)

ALL_FIRES_GPKG = (
    PROJECT_DIR /
    "spain-portugal-morethan100ha.gpkg"
)

SAMPLING_OUTPUT_DIR = (
    PROJECT_DIR /
    "sampling"
)

SAMPLING_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PER_FIRE_DIR = (
    SAMPLING_OUTPUT_DIR /
    "per_fire"
)

PER_FIRE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SPLIT_CSV = (
    SAMPLING_OUTPUT_DIR /
    "fire_event_split.csv"
)

SAMPLE_CSV = (
    SAMPLING_OUTPUT_DIR /
    "common_training_samples.csv"
)


# ============================================================
# 3. LAYER NAMES
# ============================================================

FIRE_LAYER = "fire_polygons"

UNBURNED_LAYER = (
    "candidate_unburned_rings"
)

EXTENT_LAYER = (
    "collection_extents"
)


# ============================================================
# 4. SAMPLING PARAMETERS
# ============================================================

RANDOM_SEED = 42

N_SAMPLES_PER_FIRE = 240

SAMPLING_FRACTIONS = {
    "burned_interior": 0.35,
    "boundary": 0.35,
    "unburned": 0.30,
}

PATCH_SIZE_PIXELS = 64

BOUNDARY_WIDTH_M = 100

MIN_CENTER_SPACING_M = 300

MIN_PATCH_VALID_FRACTION = 0.90

PREVIOUS_FIRE_YEARS = 2


# ============================================================
# 5. NORMALIZE FIRE ID
# ============================================================

def normalize_fire_id(value):

    value = str(
        value
    ).strip().lower()

    match = re.search(
        r"fire[_\- ]*0*(\d+)",
        value,
        flags=re.IGNORECASE
    )

    if match is None:

        match = re.search(
            r"(\d+)",
            value
        )

    if match is None:

        return value

    number = int(
        match.group(1)
    )

    return f"fire_{number:02d}"


# ============================================================
# 6. GEOMETRY REPAIR
# ============================================================

def repair_geometry(geom):

    if (
        geom is None
        or
        geom.is_empty
    ):
        return geom

    if geom.is_valid:
        return geom

    try:

        geom = make_valid(
            geom
        )

    except Exception:

        geom = geom.buffer(
            0
        )

    if (
        geom is None
        or
        geom.is_empty
    ):
        return None

    return geom


# ============================================================
# 7. CHECK INPUT FILES
# ============================================================

if not REFERENCE_GPKG.exists():

    raise FileNotFoundError(
        REFERENCE_GPKG
    )


if not ALL_FIRES_GPKG.exists():

    raise FileNotFoundError(
        ALL_FIRES_GPKG
    )


if not np.isclose(
    sum(
        SAMPLING_FRACTIONS.values()
    ),
    1.0
):

    raise ValueError(
        "Sampling fractions must sum to 1."
    )


if PATCH_SIZE_PIXELS % 2 != 0:

    raise ValueError(
        "PATCH_SIZE_PIXELS must be even."
    )


# ============================================================
# 8. LOAD ONLY TARGET-FIRE REFERENCE DATA
# ============================================================

fire_polygons = gpd.read_file(
    REFERENCE_GPKG,
    layer=FIRE_LAYER
)

candidate_unburned = gpd.read_file(
    REFERENCE_GPKG,
    layer=UNBURNED_LAYER
)

collection_extents = gpd.read_file(
    REFERENCE_GPKG,
    layer=EXTENT_LAYER
)


for frame_name, frame in [
    (
        "fire_polygons",
        fire_polygons
    ),
    (
        "candidate_unburned",
        candidate_unburned
    ),
    (
        "collection_extents",
        collection_extents
    ),
]:

    if frame.crs is None:

        raise ValueError(
            f"{frame_name} has no CRS."
        )


# ============================================================
# 9. NORMALIZE VECTOR IDS
# ============================================================

for frame in [
    fire_polygons,
    candidate_unburned,
    collection_extents,
]:

    frame[
        "fire_id"
    ] = (
        frame[
            "fire_id"
        ]
        .apply(
            normalize_fire_id
        )
    )


fire_polygons[
    "detected_datetime"
] = pd.to_datetime(
    fire_polygons[
        "detected_datetime"
    ],
    errors="raise"
)


# ============================================================
# 10. NORMALIZE RASTER CATALOG IDS
#
# catalog must already exist from EO preprocessing/catalog cell
# ============================================================

if "catalog" not in globals():

    raise NameError(
        "catalog is not defined. "
        "Run the EO raster catalog block first."
    )


catalog = {
    normalize_fire_id(
        fire_id
    ):
        products
    for fire_id, products
    in catalog.items()
}


# ============================================================
# 11. MATCH FIRE IDS
# ============================================================

raster_ids = set(
    catalog.keys()
)

vector_ids = set(
    fire_polygons[
        "fire_id"
    ]
)

available_fire_ids = sorted(
    raster_ids &
    vector_ids
)


print(
    "Common fire IDs:",
    len(
        available_fire_ids
    ),
    available_fire_ids
)


if len(
    available_fire_ids
) == 0:

    raise RuntimeError(
        "No common fire IDs."
    )


fire_polygons = (
    fire_polygons[
        fire_polygons[
            "fire_id"
        ].isin(
            available_fire_ids
        )
    ]
    .copy()
)


candidate_unburned = (
    candidate_unburned[
        candidate_unburned[
            "fire_id"
        ].isin(
            available_fire_ids
        )
    ]
    .copy()
)


collection_extents = (
    collection_extents[
        collection_extents[
            "fire_id"
        ].isin(
            available_fire_ids
        )
    ]
    .copy()
)


# ============================================================
# 12. HISTORICAL FIRE LAYER METADATA ONLY
#
# IMPORTANT:
# We DO NOT load the full historical GeoPackage here.
# ============================================================

layers = pyogrio.list_layers(
    ALL_FIRES_GPKG
)

polygon_layers = [
    layer_name
    for layer_name, geom_type
    in layers
    if (
        geom_type is not None
        and
        "Polygon" in geom_type
    )
]


if len(
    polygon_layers
) == 0:

    raise ValueError(
        "No polygon layer found in "
        "historical-fire database."
    )


ALL_FIRES_LAYER = (
    polygon_layers[0]
)


print(
    "Using all-fire layer:",
    ALL_FIRES_LAYER
)


all_fires_info = pyogrio.read_info(
    ALL_FIRES_GPKG,
    layer=ALL_FIRES_LAYER
)


ALL_FIRES_CRS = (
    all_fires_info[
        "crs"
    ]
)


ALL_FIRES_FIELDS = set(
    all_fires_info[
        "fields"
    ]
)


candidate_date_fields = [
    "detected_datetime",
    "detected_date",
    "start_datetime",
    "start_date",
    "fire_date",
    "date",
]


ALL_FIRES_DATE_FIELD = None


for field in candidate_date_fields:

    if field in ALL_FIRES_FIELDS:

        ALL_FIRES_DATE_FIELD = (
            field
        )

        break


if ALL_FIRES_DATE_FIELD is None:

    raise ValueError(
        "Could not identify historical-fire date field."
    )


print(
    "All-fire date field:",
    ALL_FIRES_DATE_FIELD
)


# ============================================================
# 13. EVENT-LEVEL SPLIT
# ============================================================

def create_country_balanced_split(
    fires,
    seed=42
):

    rng = np.random.default_rng(
        seed
    )

    records = []


    for country, group in (
        fires.groupby(
            "country"
        )
    ):

        ids = list(
            rng.permutation(
                group[
                    "fire_id"
                ].tolist()
            )
        )


        if len(
            ids
        ) != 6:

            raise ValueError(
                f"Expected 6 fires for {country}, "
                f"found {len(ids)}."
            )


        train_ids = ids[:4]

        val_ids = ids[4:5]

        test_ids = ids[5:6]


        for fire_id in train_ids:

            records.append({
                "fire_id":
                    fire_id,

                "country":
                    country,

                "split":
                    "train",
            })


        for fire_id in val_ids:

            records.append({
                "fire_id":
                    fire_id,

                "country":
                    country,

                "split":
                    "validation",
            })


        for fire_id in test_ids:

            records.append({
                "fire_id":
                    fire_id,

                "country":
                    country,

                "split":
                    "test",
            })


    return pd.DataFrame(
        records
    )


event_split = (
    create_country_balanced_split(
        fire_polygons,
        RANDOM_SEED
    )
)


event_split.to_csv(
    SPLIT_CSV,
    index=False
)


print(
    "\nEVENT SPLIT"
)

display(
    event_split
)


print(
    "\nCounts by country and split"
)

display(
    pd.crosstab(
        event_split[
            "country"
        ],
        event_split[
            "split"
        ]
    )
)


# ============================================================
# 14. GET TARGET GEOMETRY
# ============================================================

def get_event_geometry(
    frame,
    fire_id,
    target_crs
):

    fire_id = (
        normalize_fire_id(
            fire_id
        )
    )


    subset = frame[
        frame[
            "fire_id"
        ]
        ==
        fire_id
    ]


    if len(
        subset
    ) != 1:

        raise ValueError(
            f"{fire_id}: expected one geometry, "
            f"found {len(subset)}."
        )


    geom = (
        subset
        .to_crs(
            target_crs
        )
        .geometry
        .iloc[0]
    )


    return repair_geometry(
        geom
    )


# ============================================================
# 15. GET TARGET METADATA
# ============================================================

def get_target_fire_row(
    fire_id
):

    fire_id = (
        normalize_fire_id(
            fire_id
        )
    )


    subset = fire_polygons[
        fire_polygons[
            "fire_id"
        ]
        ==
        fire_id
    ]


    if len(
        subset
    ) != 1:

        raise ValueError(
            f"{fire_id}: target metadata not unique."
        )


    return subset.iloc[
        0
    ]


# ============================================================
# 16. GET RASTER PRODUCT PATHS
# ============================================================

def get_product_paths(
    fire_id,
    product
):

    fire_id = (
        normalize_fire_id(
            fire_id
        )
    )


    if fire_id not in catalog:

        raise KeyError(
            f"{fire_id} missing from catalog."
        )


    if product not in catalog[
        fire_id
    ]:

        raise KeyError(
            f"{product} missing for {fire_id}."
        )


    item = (
        catalog[
            fire_id
        ][
            product
        ]
    )


    if isinstance(
        item,
        (str, Path)
    ):

        paths = [
            item
        ]


    elif isinstance(
        item,
        dict
    ):

        paths = None

        for key in [
            "paths",
            "path",
            "files",
            "file",
            "filepath",
        ]:

            if key in item:

                paths = (
                    item[
                        key
                    ]
                )

                break


        if paths is None:

            raise ValueError(
                f"Cannot determine paths for "
                f"{fire_id}, {product}."
            )


        if isinstance(
            paths,
            (str, Path)
        ):

            paths = [
                paths
            ]


    else:

        paths = list(
            item
        )


    paths = [
        Path(
            p
        )
        for p in paths
    ]


    return paths


# ============================================================
# 17. BAND DEFINITIONS
# ============================================================

BAND_NAMES = {

    "S2_pre": [
        "B2",
        "B3",
        "B4",
        "B5",
        "B6",
        "B7",
        "B8",
        "B8A",
        "B11",
        "B12",
        "valid",
    ],

    "S2_post": [
        "B2",
        "B3",
        "B4",
        "B5",
        "B6",
        "B7",
        "B8",
        "B8A",
        "B11",
        "B12",
        "valid",
    ],

    "S1_pre": [
        "VV",
        "VH",
        "valid",
    ],

    "S1_post": [
        "VV",
        "VH",
        "valid",
    ],
}


# ============================================================
# 18. READ ONLY VALID BAND
# ============================================================

def read_product_valid_mask(
    fire_id,
    product
):

    paths = get_product_paths(
        fire_id,
        product
    )


    band_index = (
        BAND_NAMES[
            product
        ].index(
            "valid"
        )
        +
        1
    )


    # --------------------------------------------------------
    # Single GeoTIFF
    # --------------------------------------------------------

    if len(
        paths
    ) == 1:

        with rasterio.open(
            paths[0]
        ) as src:

            valid = (
                src.read(
                    band_index
                )
                >
                0.5
            )


            meta = {
                "transform":
                    src.transform,

                "crs":
                    src.crs,

                "height":
                    src.height,

                "width":
                    src.width,
            }


        return (
            valid,
            meta
        )


    # --------------------------------------------------------
    # Multiple raster pieces
    # --------------------------------------------------------

    memory_files = []

    datasets = []


    try:

        for path in paths:

            with rasterio.open(
                path
            ) as src:

                valid_array = (
                    src.read(
                        band_index
                    )
                    >
                    0.5
                ).astype(
                    np.uint8
                )


                profile = (
                    src.profile.copy()
                )


                profile.update({
                    "count":
                        1,

                    "dtype":
                        "uint8",

                    "nodata":
                        0,
                })


                memfile = (
                    MemoryFile()
                )


                ds = memfile.open(
                    **profile
                )


                ds.write(
                    valid_array,
                    1
                )


                memory_files.append(
                    memfile
                )

                datasets.append(
                    ds
                )


                del valid_array

                gc.collect()


        merged, transform = merge(
            datasets,
            nodata=0,
            dtype="uint8"
        )


        valid = (
            merged[
                0
            ]
            >
            0
        )


        meta = {
            "transform":
                transform,

            "crs":
                datasets[
                    0
                ].crs,

            "height":
                valid.shape[
                    0
                ],

            "width":
                valid.shape[
                    1
                ],
        }


        del merged


        return (
            valid,
            meta
        )


    finally:

        for ds in datasets:

            try:

                ds.close()

            except Exception:

                pass


        for memfile in memory_files:

            try:

                memfile.close()

            except Exception:

                pass


        gc.collect()


# ============================================================
# 19. GRID CHECK
# ============================================================

def grids_match(
    a,
    b
):

    return (
        a[
            "crs"
        ]
        ==
        b[
            "crs"
        ]

        and

        a[
            "transform"
        ]
        ==
        b[
            "transform"
        ]

        and

        a[
            "height"
        ]
        ==
        b[
            "height"
        ]

        and

        a[
            "width"
        ]
        ==
        b[
            "width"
        ]
    )


# ============================================================
# 20. COMMON VALIDITY MASK
# ============================================================

def build_common_valid_mask_light(
    fire_id
):

    products = [
        "S2_pre",
        "S2_post",
        "S1_pre",
        "S1_post",
    ]


    common_valid = None

    reference_meta = None


    for product in products:

        valid, meta = (
            read_product_valid_mask(
                fire_id,
                product
            )
        )


        if common_valid is None:

            common_valid = (
                valid.copy()
            )

            reference_meta = (
                meta
            )


        else:

            if not grids_match(
                reference_meta,
                meta
            ):

                raise ValueError(
                    f"{fire_id}: grid mismatch "
                    f"for {product}."
                )


            common_valid &= (
                valid
            )


        del valid

        gc.collect()


    print(
        f"{fire_id}: "
        "all four validity grids match."
    )


    print(
        f"{fire_id}: common valid pixels = "
        f"{common_valid.sum():,} "
        f"({100 * common_valid.mean():.1f}%)"
    )


    return (
        common_valid,
        reference_meta
    )


# ============================================================
# 21. GENERIC GEOMETRY RASTERIZER
# ============================================================

def rasterize_geometry(
    geom,
    shape,
    transform
):

    if (
        geom is None
        or
        geom.is_empty
    ):

        return np.zeros(
            shape,
            dtype=np.bool_
        )


    return (
        rasterize(
            [
                (
                    geom,
                    1
                )
            ],

            out_shape=
                shape,

            transform=
                transform,

            fill=
                0,

            dtype=
                "uint8",

            all_touched=
                False,
        )
        >
        0
    )


# ============================================================
# 22. READ ONLY LOCAL HISTORICAL FIRES FROM DISK
# ============================================================

def get_recent_previous_fires_subset(
    fire_id,
    target_crs
):

    fire_id = (
        normalize_fire_id(
            fire_id
        )
    )


    target = (
        get_target_fire_row(
            fire_id
        )
    )


    target_date = pd.Timestamp(
        target[
            "detected_datetime"
        ]
    )


    start_date = (
        target_date
        -
        pd.DateOffset(
            years=
                PREVIOUS_FIRE_YEARS
        )
    )


    # --------------------------------------------------------
    # Local collection extent in raster CRS
    # --------------------------------------------------------

    extent_geom = (
        get_event_geometry(
            collection_extents,
            fire_id,
            target_crs
        )
    )


    # --------------------------------------------------------
    # Convert one geometry to historical database CRS
    # --------------------------------------------------------

    extent_frame = gpd.GeoDataFrame(
        {
            "geometry": [
                extent_geom
            ]
        },
        crs=target_crs
    )


    extent_historical = (
        extent_frame
        .to_crs(
            ALL_FIRES_CRS
        )
        .geometry
        .iloc[
            0
        ]
    )


    bbox = (
        extent_historical.bounds
    )


    del extent_frame
    del extent_geom

    gc.collect()


    # --------------------------------------------------------
    # Read ONLY local polygons from GeoPackage
    # --------------------------------------------------------

    recent = (
        pyogrio.read_dataframe(
            ALL_FIRES_GPKG,

            layer=
                ALL_FIRES_LAYER,

            columns=[
                ALL_FIRES_DATE_FIELD
            ],

            bbox=
                bbox,
        )
    )


    if recent.empty:

        print(
            f"{fire_id}: "
            "no historical polygons in local bbox."
        )

        return recent


    print(
        f"{fire_id}: "
        f"{len(recent)} historical polygon(s) "
        "read from local bbox."
    )


    # --------------------------------------------------------
    # Date filter
    # --------------------------------------------------------

    recent[
        "_fire_date"
    ] = pd.to_datetime(
        recent[
            ALL_FIRES_DATE_FIELD
        ],
        errors="coerce"
    )


    recent = (
        recent.loc[
            (
                recent[
                    "_fire_date"
                ]
                >=
                start_date
            )
            &
            (
                recent[
                    "_fire_date"
                ]
                <
                target_date
            )
        ]
        .copy()
    )


    if recent.empty:

        print(
            f"{fire_id}: "
            "no historical fires within "
            f"previous {PREVIOUS_FIRE_YEARS} years."
        )

        return recent


    # --------------------------------------------------------
    # Exact spatial intersection
    # --------------------------------------------------------

    keep = (
        recent.geometry.intersects(
            extent_historical
        )
    )


    recent = (
        recent.loc[
            keep
        ]
        .copy()
    )


    del keep
    del extent_historical

    gc.collect()


    if recent.empty:

        print(
            f"{fire_id}: "
            "no recent scars intersect exact extent."
        )

        return recent


    # --------------------------------------------------------
    # Repair only these local polygons
    # --------------------------------------------------------

    recent[
        "geometry"
    ] = (
        recent.geometry.apply(
            repair_geometry
        )
    )


    recent = (
        recent[
            recent.geometry.notna()
            &
            ~recent.geometry.is_empty
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Reproject only selected polygons
    # --------------------------------------------------------

    recent = (
        recent.to_crs(
            target_crs
        )
    )


    print(
        f"{fire_id}: "
        f"{len(recent)} previous fire polygon(s) "
        f"within {PREVIOUS_FIRE_YEARS} years."
    )


    return recent


# ============================================================
# 23. DIRECTLY RASTERIZE HISTORICAL SCARS
# ============================================================

def rasterize_previous_fire_scars(
    fire_id,
    target_crs,
    shape,
    transform
):

    recent = (
        get_recent_previous_fires_subset(
            fire_id,
            target_crs
        )
    )


    if recent.empty:

        return np.zeros(
            shape,
            dtype=np.bool_
        )


    scar_mask = (
        rasterize(
            shapes=(
                (
                    geom,
                    1
                )

                for geom
                in recent.geometry

                if (
                    geom is not None
                    and
                    not geom.is_empty
                )
            ),

            out_shape=
                shape,

            transform=
                transform,

            fill=
                0,

            dtype=
                "uint8",

            all_touched=
                False,
        )
        >
        0
    )


    print(
        f"{fire_id}: "
        f"{scar_mask.sum():,} pixels "
        "excluded as previous fire scars."
    )


    del recent

    gc.collect()


    return scar_mask


# ============================================================
# 24. CREATE SAMPLING STRATA
# ============================================================

def create_sampling_strata(
    fire_id,
    meta
):

    fire_id = (
        normalize_fire_id(
            fire_id
        )
    )


    crs = (
        meta[
            "crs"
        ]
    )

    transform = (
        meta[
            "transform"
        ]
    )

    shape = (
        meta[
            "height"
        ],
        meta[
            "width"
        ],
    )


    # --------------------------------------------------------
    # Fire geometry
    # --------------------------------------------------------

    fire_geom = (
        get_event_geometry(
            fire_polygons,
            fire_id,
            crs
        )
    )


    fire_mask = (
        rasterize_geometry(
            fire_geom,
            shape,
            transform
        )
    )


    # --------------------------------------------------------
    # Boundary
    # --------------------------------------------------------

    boundary_geom = (
        fire_geom
        .boundary
        .buffer(
            BOUNDARY_WIDTH_M
        )
    )


    boundary_geom = (
        repair_geometry(
            boundary_geom
        )
    )


    boundary_mask = (
        rasterize_geometry(
            boundary_geom,
            shape,
            transform
        )
    )


    # --------------------------------------------------------
    # Burned interior
    # --------------------------------------------------------

    burned_mask = (
        fire_mask
        &
        ~boundary_mask
    )


    if not burned_mask.any():

        warnings.warn(
            f"{fire_id}: no burned interior "
            "after boundary exclusion."
        )

        burned_mask = (
            fire_mask.copy()
        )


    # --------------------------------------------------------
    # Candidate unburned area
    # --------------------------------------------------------

    unburned_geom = (
        get_event_geometry(
            candidate_unburned,
            fire_id,
            crs
        )
    )


    unburned_mask = (
        rasterize_geometry(
            unburned_geom,
            shape,
            transform
        )
    )


    # --------------------------------------------------------
    # Previous fire scars
    # --------------------------------------------------------

    scar_mask = (
        rasterize_previous_fire_scars(
            fire_id,
            crs,
            shape,
            transform
        )
    )


    # --------------------------------------------------------
    # Mutually exclusive masks
    # --------------------------------------------------------

    unburned_mask &= (
        ~fire_mask
    )

    unburned_mask &= (
        ~boundary_mask
    )

    unburned_mask &= (
        ~scar_mask
    )


    boundary_mask &= (
        ~burned_mask
    )


    print(
        f"{fire_id}: candidate pixels — "
        f"burned={burned_mask.sum():,}, "
        f"boundary={boundary_mask.sum():,}, "
        f"unburned={unburned_mask.sum():,}"
    )


    del fire_geom
    del fire_mask
    del boundary_geom
    del unburned_geom
    del scar_mask

    gc.collect()


    return {
        "burned_interior":
            burned_mask,

        "boundary":
            boundary_mask,

        "unburned":
            unburned_mask,
    }


# ============================================================
# 25. PATCH VALID FRACTION
# ============================================================

def get_patch_valid_fraction(
    valid_mask,
    row,
    col,
    patch_size
):

    half = (
        patch_size // 2
    )


    r0 = (
        row - half
    )

    c0 = (
        col - half
    )

    r1 = (
        r0 + patch_size
    )

    c1 = (
        c0 + patch_size
    )


    if (
        r0 < 0
        or
        c0 < 0
        or
        r1 > valid_mask.shape[
            0
        ]
        or
        c1 > valid_mask.shape[
            1
        ]
    ):

        return 0.0


    return float(
        valid_mask[
            r0:r1,
            c0:c1
        ].mean()
    )


# ============================================================
# 26. MEMORY-SAFE RANDOM STRATUM SAMPLER
# ============================================================

def sample_stratum(
    stratum_mask,
    valid_mask,
    requested,
    patch_size,
    minimum_distance_pixels,
    rng,
    max_attempt_factor=5000,
):

    height, width = (
        stratum_mask.shape
    )


    half = (
        patch_size // 2
    )


    row_min = (
        half
    )

    row_max = (
        height - half
    )

    col_min = (
        half
    )

    col_max = (
        width - half
    )


    selected = []

    selected_rows = []

    selected_cols = []


    min_dist_sq = (
        minimum_distance_pixels
        **
        2
    )


    max_attempts = max(
        requested
        *
        max_attempt_factor,
        10000
    )


    attempts = 0


    while (
        len(
            selected
        )
        <
        requested
        and
        attempts
        <
        max_attempts
    ):

        attempts += 1


        row = int(
            rng.integers(
                row_min,
                row_max
            )
        )


        col = int(
            rng.integers(
                col_min,
                col_max
            )
        )


        if not stratum_mask[
            row,
            col
        ]:

            continue


        if not valid_mask[
            row,
            col
        ]:

            continue


        valid_fraction = (
            get_patch_valid_fraction(
                valid_mask,
                row,
                col,
                patch_size
            )
        )


        if (
            valid_fraction
            <
            MIN_PATCH_VALID_FRACTION
        ):

            continue


        too_close = False


        for rr, cc in zip(
            selected_rows,
            selected_cols
        ):

            dr = (
                rr - row
            )

            dc = (
                cc - col
            )


            if (
                dr * dr
                +
                dc * dc
                <
                min_dist_sq
            ):

                too_close = True

                break


        if too_close:

            continue


        selected.append(
            (
                row,
                col,
                valid_fraction
            )
        )


        selected_rows.append(
            row
        )

        selected_cols.append(
            col
        )


    if len(
        selected
    ) < requested:

        warnings.warn(
            f"Requested {requested} samples "
            f"but selected only {len(selected)}."
        )


    return selected


# ============================================================
# 27. SAMPLE ONE FIRE
# ============================================================

def sample_fire(
    fire_id,
    split,
    seed
):

    fire_id = (
        normalize_fire_id(
            fire_id
        )
    )


    print(
        "\n"
        +
        "=" * 70
    )


    print(
        f"Sampling {fire_id} [{split}]"
    )


    common_valid, meta = (
        build_common_valid_mask_light(
            fire_id
        )
    )


    transform = (
        meta[
            "transform"
        ]
    )


    pixel_size = (
        abs(
            transform.a
        )
        +
        abs(
            transform.e
        )
    ) / 2


    minimum_distance_pixels = max(
        1,
        int(
            np.ceil(
                MIN_CENTER_SPACING_M
                /
                pixel_size
            )
        )
    )


    strata = (
        create_sampling_strata(
            fire_id,
            meta
        )
    )


    gc.collect()


    fire_number = int(
        re.search(
            r"(\d+)",
            fire_id
        )
        .group(
            1
        )
    )


    rng = np.random.default_rng(
        seed
        +
        fire_number
    )


    requested_counts = {
        key:
            int(
                round(
                    N_SAMPLES_PER_FIRE
                    *
                    fraction
                )
            )

        for key, fraction
        in SAMPLING_FRACTIONS.items()
    }


    requested_counts[
        "boundary"
    ] += (
        N_SAMPLES_PER_FIRE
        -
        sum(
            requested_counts.values()
        )
    )


    records = []


    for (
        stratum_name,
        stratum_mask
    ) in strata.items():

        requested = (
            requested_counts[
                stratum_name
            ]
        )


        selected = sample_stratum(
            stratum_mask=
                stratum_mask,

            valid_mask=
                common_valid,

            requested=
                requested,

            patch_size=
                PATCH_SIZE_PIXELS,

            minimum_distance_pixels=
                minimum_distance_pixels,

            rng=
                rng,
        )


        print(
            f"{stratum_name:18s} | "
            f"requested={requested:3d} | "
            f"selected={len(selected):3d}"
        )


        for (
            sample_number,
            (
                row,
                col,
                valid_fraction
            )
        ) in enumerate(
            selected,
            start=1
        ):

            x_coord, y_coord = xy(
                transform,
                row,
                col,
                offset="center"
            )


            half = (
                PATCH_SIZE_PIXELS
                //
                2
            )


            records.append({

                "fire_id":
                    fire_id,

                "split":
                    split,

                "stratum":
                    stratum_name,

                "sample_number":
                    sample_number,

                "row":
                    int(
                        row
                    ),

                "col":
                    int(
                        col
                    ),

                "row_start":
                    int(
                        row - half
                    ),

                "row_stop":
                    int(
                        row - half
                        +
                        PATCH_SIZE_PIXELS
                    ),

                "col_start":
                    int(
                        col - half
                    ),

                "col_stop":
                    int(
                        col - half
                        +
                        PATCH_SIZE_PIXELS
                    ),

                "patch_size_pixels":
                    PATCH_SIZE_PIXELS,

                "pixel_size_m":
                    float(
                        pixel_size
                    ),

                "patch_width_m":
                    float(
                        PATCH_SIZE_PIXELS
                        *
                        pixel_size
                    ),

                "valid_fraction":
                    float(
                        valid_fraction
                    ),

                "x":
                    float(
                        x_coord
                    ),

                "y":
                    float(
                        y_coord
                    ),

                "crs":
                    str(
                        meta[
                            "crs"
                        ]
                    ),
            })


        del selected

        gc.collect()


    del common_valid

    for key in list(
        strata.keys()
    ):

        del strata[
            key
        ]


    del strata

    gc.collect()


    return pd.DataFrame(
        records
    )


# ============================================================
# 28. RESUMABLE SAMPLING LOOP
# ============================================================

for _, split_row in (
    event_split.iterrows()
):

    fire_id = (
        normalize_fire_id(
            split_row[
                "fire_id"
            ]
        )
    )


    split = (
        split_row[
            "split"
        ]
    )


    # --------------------------------------------------------
    # Test events remain intact
    # --------------------------------------------------------

    if split == "test":

        print(
            f"\n{fire_id}: TEST fire — "
            "no sampling; complete evaluation extent retained."
        )

        continue


    output_csv = (
        PER_FIRE_DIR
        /
        f"{fire_id}_{split}_samples.csv"
    )


    # --------------------------------------------------------
    # Resume
    # --------------------------------------------------------

    if output_csv.exists():

        print(
            f"\n{fire_id}: "
            "already sampled — skipped."
        )

        continue


    samples = sample_fire(
        fire_id=
            fire_id,

        split=
            split,

        seed=
            RANDOM_SEED,
    )


    if samples.empty:

        warnings.warn(
            f"{fire_id}: no samples generated."
        )

        continue


    target = (
        get_target_fire_row(
            fire_id
        )
    )


    samples[
        "area_name"
    ] = (
        target[
            "area_name"
        ]
    )


    samples[
        "country"
    ] = (
        target[
            "country"
        ]
    )


    # --------------------------------------------------------
    # SAVE IMMEDIATELY
    # --------------------------------------------------------

    samples.to_csv(
        output_csv,
        index=False
    )


    print(
        f"{fire_id}: "
        f"saved {len(samples)} samples."
    )


    print(
        output_csv
    )


    del samples

    gc.collect()


# ============================================================
# 29. REBUILD MASTER SAMPLE TABLE
# ============================================================

sample_files = sorted(
    PER_FIRE_DIR.glob(
        "*_samples.csv"
    )
)


print(
    f"\nFound {len(sample_files)} "
    "per-fire sample files:"
)


for path in sample_files:

    print(
        " -",
        path.name
    )


if len(
    sample_files
) == 0:

    raise RuntimeError(
        "No sample CSV files found."
    )


master_samples = pd.concat(
    [
        pd.read_csv(
            path
        )
        for path in sample_files
    ],
    ignore_index=True
)


master_samples.to_csv(
    SAMPLE_CSV,
    index=False
)


# ============================================================
# 30. FINAL SUMMARY
# ============================================================

print(
    "\n"
    +
    "=" * 70
)

print(
    "FINAL SAMPLING SUMMARY"
)

print(
    "=" * 70
)


print(
    "\nTotal samples:",
    len(
        master_samples
    )
)


print(
    "\nSamples by split:"
)

print(
    master_samples[
        "split"
    ].value_counts()
)


print(
    "\nSamples by fire:"
)

print(
    master_samples
    .groupby(
        [
            "split",
            "fire_id"
        ]
    )
    .size()
)


print(
    "\nSamples by stratum:"
)

print(
    master_samples[
        "stratum"
    ].value_counts()
)


# ============================================================
# 31. CHECK COMPLETENESS
# ============================================================

TRAIN_FIRE_IDS = (
    event_split.loc[
        event_split[
            "split"
        ]
        ==
        "train",
        "fire_id"
    ]
    .tolist()
)


VAL_FIRE_IDS = (
    event_split.loc[
        event_split[
            "split"
        ]
        ==
        "validation",
        "fire_id"
    ]
    .tolist()
)


TEST_FIRE_IDS = (
    event_split.loc[
        event_split[
            "split"
        ]
        ==
        "test",
        "fire_id"
    ]
    .tolist()
)


actual_train = set(
    master_samples.loc[
        master_samples[
            "split"
        ]
        ==
        "train",
        "fire_id"
    ]
)


actual_val = set(
    master_samples.loc[
        master_samples[
            "split"
        ]
        ==
        "validation",
        "fire_id"
    ]
)


missing_train = sorted(
    set(
        TRAIN_FIRE_IDS
    )
    -
    actual_train
)


missing_val = sorted(
    set(
        VAL_FIRE_IDS
    )
    -
    actual_val
)


print(
    "\nMissing training fires:",
    missing_train
)

print(
    "Missing validation fires:",
    missing_val
)


expected_total = (
    (
        len(
            TRAIN_FIRE_IDS
        )
        +
        len(
            VAL_FIRE_IDS
        )
    )
    *
    N_SAMPLES_PER_FIRE
)


print(
    "\nExpected total samples:",
    expected_total
)


print(
    "Actual total samples:",
    len(
        master_samples
    )
)


if (
    len(
        missing_train
    )
    ==
    0

    and

    len(
        missing_val
    )
    ==
    0

    and

    len(
        master_samples
    )
    ==
    expected_total
):

    print(
        "\nSUCCESS: "
        "sampling dataset is complete."
    )


else:

    print(
        "\nSampling is still incomplete."
    )


print(
    "\nTRAIN:"
)

print(
    TRAIN_FIRE_IDS
)


print(
    "\nVALIDATION:"
)

print(
    VAL_FIRE_IDS
)


print(
    "\nTEST:"
)

print(
    TEST_FIRE_IDS
)


print(
    "\nMaster sample file:"
)

print(
    SAMPLE_CSV
)

# Modeling

In [ ]:
# ============================================================
# REQUIRED GEOMETRY HELPERS FOR MODELING
# IDEMPOTENT / HEALTH-CHECKED VERSION
# ============================================================

from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely import make_valid
from shapely.ops import unary_union
import pyogrio


# ============================================================
# 1. PATHS / LAYERS
# ============================================================

PROJECT_DIR = PROJECT_DIR

REFERENCE_GPKG = (
    PROJECT_DIR /
    "fire_collection_areas.gpkg"
)

ALL_FIRES_GPKG = (
    PROJECT_DIR /
    "spain-portugal-morethan100ha.gpkg"
)

FIRE_LAYER = "fire_polygons"
UNBURNED_LAYER = "candidate_unburned_rings"
EXTENT_LAYER = "collection_extents"

PREVIOUS_FIRE_YEARS = 2


# ============================================================
# 2. HEALTH-CHECK HELPERS
# ============================================================

def healthy_geodataframe(
    obj,
    required_columns=None,
    require_nonempty=True,
):
    """
    Basic health check for an existing GeoDataFrame.
    """

    if not isinstance(
        obj,
        gpd.GeoDataFrame
    ):
        return False

    if require_nonempty and len(obj) == 0:
        return False

    if obj.geometry.name not in obj.columns:
        return False

    if required_columns is not None:

        missing = [
            col
            for col in required_columns
            if col not in obj.columns
        ]

        if missing:
            return False

    return True


def variable_is_healthy(
    name,
    required_columns=None,
    require_nonempty=True,
):
    """
    Check whether a named GeoDataFrame already exists
    and looks usable.
    """

    if name not in globals():
        return False

    return healthy_geodataframe(
        globals()[name],
        required_columns=required_columns,
        require_nonempty=require_nonempty,
    )


# ============================================================
# 3. LOAD REFERENCE LAYERS ONLY IF NEEDED
# ============================================================

if variable_is_healthy(
    "fire_polygons",
    required_columns=[
        "fire_id",
        "detected_datetime",
    ],
):

    print(
        "fire_polygons already loaded and healthy."
    )

else:

    print(
        "Loading fire_polygons..."
    )

    fire_polygons = gpd.read_file(
        REFERENCE_GPKG,
        layer=FIRE_LAYER
    )


if variable_is_healthy(
    "candidate_unburned",
    required_columns=[
        "fire_id",
    ],
):

    print(
        "candidate_unburned already loaded and healthy."
    )

else:

    print(
        "Loading candidate_unburned..."
    )

    candidate_unburned = gpd.read_file(
        REFERENCE_GPKG,
        layer=UNBURNED_LAYER
    )


if variable_is_healthy(
    "collection_extents",
    required_columns=[
        "fire_id",
    ],
):

    print(
        "collection_extents already loaded and healthy."
    )

else:

    print(
        "Loading collection_extents..."
    )

    collection_extents = gpd.read_file(
        REFERENCE_GPKG,
        layer=EXTENT_LAYER
    )


# ============================================================
# 4. NORMALIZE FIRE IDS ONLY IF NEEDED
# ============================================================

for frame_name in [
    "fire_polygons",
    "candidate_unburned",
    "collection_extents",
]:

    frame = globals()[
        frame_name
    ]

    normalized = (
        frame["fire_id"]
        .apply(
            normalize_fire_id
        )
    )

    if not frame[
        "fire_id"
    ].equals(
        normalized
    ):

        print(
            f"Normalizing fire IDs in {frame_name}..."
        )

        frame[
            "fire_id"
        ] = normalized

    else:

        print(
            f"{frame_name}: fire IDs already normalized."
        )


# ============================================================
# 5. TARGET FIRE DATES
# ============================================================

if not pd.api.types.is_datetime64_any_dtype(
    fire_polygons[
        "detected_datetime"
    ]
):

    print(
        "Parsing target-fire dates..."
    )

    fire_polygons[
        "detected_datetime"
    ] = pd.to_datetime(
        fire_polygons[
            "detected_datetime"
        ],
        errors="raise"
    )

else:

    print(
        "Target-fire dates already parsed."
    )


# ============================================================
# 6. GEOMETRY REPAIR FUNCTION
# ============================================================

if (
    "repair_geometry" not in globals()
    or
    not callable(
        globals()[
            "repair_geometry"
        ]
    )
):

    def repair_geometry(
        geom
    ):

        if (
            geom is None
            or
            geom.is_empty
        ):
            return geom

        if geom.is_valid:
            return geom

        try:

            repaired = make_valid(
                geom
            )

        except Exception:

            repaired = geom.buffer(
                0
            )

        if (
            repaired is None
            or
            repaired.is_empty
        ):

            repaired = geom.buffer(
                0
            )

        return repaired


    print(
        "Defined repair_geometry()."
    )

else:

    print(
        "repair_geometry() already defined."
    )


# ============================================================
# 7. GET ONE EVENT GEOMETRY
# ============================================================

if (
    "get_event_geometry" not in globals()
    or
    not callable(
        globals()[
            "get_event_geometry"
        ]
    )
):

    def get_event_geometry(
        frame,
        fire_id,
        target_crs
    ):

        fire_id = (
            normalize_fire_id(
                fire_id
            )
        )

        subset = frame[
            frame[
                "fire_id"
            ]
            ==
            fire_id
        ]

        if len(
            subset
        ) != 1:

            raise ValueError(
                f"{fire_id}: expected exactly one geometry "
                f"but found {len(subset)}."
            )

        subset = (
            subset.to_crs(
                target_crs
            )
        )

        geom = (
            subset.geometry
            .iloc[0]
        )

        return repair_geometry(
            geom
        )


    print(
        "Defined get_event_geometry()."
    )

else:

    print(
        "get_event_geometry() already defined."
    )


# ============================================================
# 8. GET TARGET-FIRE METADATA ROW
# ============================================================

if (
    "get_target_fire_row" not in globals()
    or
    not callable(
        globals()[
            "get_target_fire_row"
        ]
    )
):

    def get_target_fire_row(
        fire_id
    ):

        fire_id = (
            normalize_fire_id(
                fire_id
            )
        )

        subset = fire_polygons[
            fire_polygons[
                "fire_id"
            ]
            ==
            fire_id
        ]

        if len(
            subset
        ) != 1:

            raise ValueError(
                f"{fire_id}: expected exactly one target-fire row "
                f"but found {len(subset)}."
            )

        return subset.iloc[
            0
        ]


    print(
        "Defined get_target_fire_row()."
    )

else:

    print(
        "get_target_fire_row() already defined."
    )


# ============================================================
# 9. IDENTIFY ALL-FIRES POLYGON LAYER
# ============================================================

if (
    "ALL_FIRES_LAYER" in globals()
    and
    isinstance(
        ALL_FIRES_LAYER,
        str
    )
    and
    len(
        ALL_FIRES_LAYER
    ) > 0
):

    print(
        "Using existing all-fire layer:",
        ALL_FIRES_LAYER
    )

else:

    layers = (
        pyogrio.list_layers(
            ALL_FIRES_GPKG
        )
    )

    polygon_layers = [
        layer_name
        for layer_name, geom_type
        in layers
        if (
            geom_type is not None
            and
            "Polygon" in geom_type
        )
    ]

    if len(
        polygon_layers
    ) == 0:

        raise ValueError(
            "No polygon layer found in "
            "spain-portugal-morethan100ha.gpkg."
        )

    ALL_FIRES_LAYER = (
        polygon_layers[0]
    )

    print(
        "Detected all-fire layer:",
        ALL_FIRES_LAYER
    )


# ============================================================
# 10. LOAD ALL-FIRES DATABASE ONLY IF NEEDED
# ============================================================

if variable_is_healthy(
    "all_fires",
):

    print(
        "all_fires already loaded and healthy."
    )

else:

    print(
        "Loading all-fires database..."
    )

    all_fires = gpd.read_file(
        ALL_FIRES_GPKG,
        layer=ALL_FIRES_LAYER
    )


# ============================================================
# 11. REPAIR ALL-FIRE GEOMETRIES
# ONLY IF INVALID GEOMETRIES EXIST
# ============================================================

valid_geometry_mask = (
    all_fires.geometry.notna()
    &
    ~all_fires.geometry.is_empty
)


invalid_geometry_count = (
    ~all_fires.loc[
        valid_geometry_mask,
        "geometry"
    ].is_valid
).sum()


if (
    invalid_geometry_count > 0
    or
    not valid_geometry_mask.all()
):

    print(
        "Repairing/filtering all-fire geometries..."
    )

    all_fires[
        "geometry"
    ] = (
        all_fires.geometry
        .apply(
            repair_geometry
        )
    )

    all_fires = (
        all_fires[
            all_fires.geometry.notna()
            &
            ~all_fires.geometry.is_empty
        ]
        .copy()
    )

else:

    print(
        "All-fire geometries already healthy."
    )


# ============================================================
# 12. IDENTIFY / PREPARE DATE FIELD ONLY IF NEEDED
# ============================================================

candidate_date_fields = [
    "detected_datetime",
    "detected_date",
    "start_datetime",
    "start_date",
    "fire_date",
    "date",
]


existing_date_field_healthy = (
    "ALL_FIRES_DATE_FIELD" in globals()
    and
    ALL_FIRES_DATE_FIELD in all_fires.columns
    and
    "_fire_date" in all_fires.columns
    and
    pd.api.types.is_datetime64_any_dtype(
        all_fires[
            "_fire_date"
        ]
    )
    and
    all_fires[
        "_fire_date"
    ].notna().any()
)


if existing_date_field_healthy:

    print(
        "All-fire date field already prepared:",
        ALL_FIRES_DATE_FIELD
    )

else:

    ALL_FIRES_DATE_FIELD = None

    for field in candidate_date_fields:

        if field not in all_fires.columns:
            continue

        parsed = pd.to_datetime(
            all_fires[
                field
            ],
            errors="coerce"
        )

        if parsed.notna().any():

            ALL_FIRES_DATE_FIELD = (
                field
            )

            all_fires[
                "_fire_date"
            ] = parsed

            break

    if ALL_FIRES_DATE_FIELD is None:

        raise ValueError(
            "Could not identify date field "
            "in all-fires database."
        )

    all_fires = (
        all_fires[
            all_fires[
                "_fire_date"
            ].notna()
        ]
        .copy()
    )

    print(
        "Prepared all-fire date field:",
        ALL_FIRES_DATE_FIELD
    )


# ============================================================
# 13. PREVIOUS TWO-YEAR FIRE-SCAR GEOMETRY
# ============================================================

if (
    "get_recent_previous_fire_geometry" not in globals()
    or
    not callable(
        globals()[
            "get_recent_previous_fire_geometry"
        ]
    )
):

    def get_recent_previous_fire_geometry(
        fire_id,
        target_crs
    ):

        target = (
            get_target_fire_row(
                fire_id
            )
        )

        target_date = pd.Timestamp(
            target[
                "detected_datetime"
            ]
        )

        start_date = (
            target_date
            -
            pd.DateOffset(
                years=
                    PREVIOUS_FIRE_YEARS
            )
        )

        recent = all_fires[
            (
                all_fires[
                    "_fire_date"
                ]
                >=
                start_date
            )
            &
            (
                all_fires[
                    "_fire_date"
                ]
                <
                target_date
            )
        ].copy()

        if recent.empty:

            return None

        recent = (
            recent.to_crs(
                target_crs
            )
        )

        geometries = []

        for geom in recent.geometry:

            geom = repair_geometry(
                geom
            )

            if (
                geom is not None
                and
                not geom.is_empty
            ):

                geometries.append(
                    geom
                )

        if len(
            geometries
        ) == 0:

            return None

        try:

            merged = unary_union(
                geometries
            )

        except Exception:

            cleaned = []

            for geom in geometries:

                try:

                    fixed = geom.buffer(
                        0
                    )

                except Exception:

                    continue

                if (
                    fixed is not None
                    and
                    not fixed.is_empty
                ):

                    cleaned.append(
                        fixed
                    )

            if len(
                cleaned
            ) == 0:

                return None

            merged = unary_union(
                cleaned
            )

        return repair_geometry(
            merged
        )


    print(
        "Defined get_recent_previous_fire_geometry()."
    )

else:

    print(
        "get_recent_previous_fire_geometry() already defined."
    )


# ============================================================
# 14. FINAL HEALTH CHECK
# ============================================================

required_objects = {
    "fire_polygons": (
        fire_polygons,
        [
            "fire_id",
            "detected_datetime",
        ],
    ),

    "candidate_unburned": (
        candidate_unburned,
        [
            "fire_id",
        ],
    ),

    "collection_extents": (
        collection_extents,
        [
            "fire_id",
        ],
    ),

    "all_fires": (
        all_fires,
        [
            "_fire_date",
        ],
    ),
}


for name, (
    frame,
    columns
) in required_objects.items():

    if not healthy_geodataframe(
        frame,
        required_columns=columns,
    ):

        raise RuntimeError(
            f"{name} failed final health check."
        )


required_functions = [
    "repair_geometry",
    "get_event_geometry",
    "get_target_fire_row",
    "get_recent_previous_fire_geometry",
]


for func_name in required_functions:

    if (
        func_name not in globals()
        or
        not callable(
            globals()[
                func_name
            ]
        )
    ):

        raise RuntimeError(
            f"{func_name} is not available."
        )


# ============================================================
# 15. QUICK CHECK
# ============================================================

print(
    "\nReference fires:",
    len(
        fire_polygons
    )
)

print(
    "Candidate unburned regions:",
    len(
        candidate_unburned
    )
)

print(
    "All-fire polygons:",
    len(
        all_fires
    )
)

print(
    "All-fire date field:",
    ALL_FIRES_DATE_FIELD
)

print(
    "\nGeometry helpers ready."
)

In [ ]:
# ============================================================
# RASTER CATALOG FOR MODELING
# IDEMPOTENT / HEALTH-CHECKED VERSION
# ============================================================

from pathlib import Path
import re
from collections import defaultdict


# ============================================================
# 1. PROJECT DIRECTORY
# ============================================================

PROJECT_DIR = PROJECT_DIR

if not PROJECT_DIR.exists():

    raise FileNotFoundError(
        f"Project directory not found:\n"
        f"{PROJECT_DIR}"
    )


# ============================================================
# 2. EXPECTED FIRES / PRODUCTS
# ============================================================

EXPECTED_PRODUCTS = [
    "S2_pre",
    "S2_post",
    "S1_pre",
    "S1_post",
]

expected_fire_ids = [
    f"fire_{i:02d}"
    for i in range(
        1,
        13
    )
]


# ============================================================
# 3. FIRE-ID NORMALIZATION
# DEFINE ONLY IF NOT ALREADY AVAILABLE
# ============================================================

if (
    "normalize_fire_id" not in globals()
    or
    not callable(
        globals()[
            "normalize_fire_id"
        ]
    )
):

    def normalize_fire_id(
        value
    ):

        value = str(
            value
        ).strip().lower()

        match = re.search(
            r"fire[_\- ]*0*(\d+)",
            value,
            flags=re.IGNORECASE
        )

        if match is None:

            match = re.search(
                r"(\d+)",
                value
            )

        if match is None:
            return None

        number = int(
            match.group(1)
        )

        return (
            f"fire_{number:02d}"
        )


    print(
        "Defined normalize_fire_id()."
    )

else:

    print(
        "normalize_fire_id() already defined."
    )


# ============================================================
# 4. PRODUCT DETECTION
# DEFINE ONLY IF NOT ALREADY AVAILABLE
# ============================================================

if (
    "detect_product" not in globals()
    or
    not callable(
        globals()[
            "detect_product"
        ]
    )
):

    def detect_product(
        filename
    ):

        name = (
            str(filename)
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
        )


        # ----------------------------------------------------
        # Sentinel-2
        # ----------------------------------------------------

        if (
            "s2_pre" in name
            or
            "sentinel2_pre" in name
            or
            "sentinel_2_pre" in name
        ):

            return "S2_pre"


        if (
            "s2_post" in name
            or
            "sentinel2_post" in name
            or
            "sentinel_2_post" in name
        ):

            return "S2_post"


        # ----------------------------------------------------
        # Sentinel-1
        # ----------------------------------------------------

        if (
            "s1_pre" in name
            or
            "sentinel1_pre" in name
            or
            "sentinel_1_pre" in name
        ):

            return "S1_pre"


        if (
            "s1_post" in name
            or
            "sentinel1_post" in name
            or
            "sentinel_1_post" in name
        ):

            return "S1_post"


        return None


    print(
        "Defined detect_product()."
    )

else:

    print(
        "detect_product() already defined."
    )


# ============================================================
# 5. CATALOG HEALTH CHECK
# ============================================================

def raster_catalog_is_healthy(
    existing_catalog
):
    """
    Check whether an existing raster catalog is complete
    and all referenced files still exist.
    """

    if not isinstance(
        existing_catalog,
        dict
    ):

        return False


    for fire_id in expected_fire_ids:

        if fire_id not in existing_catalog:
            return False


        products = (
            existing_catalog[
                fire_id
            ]
        )


        if not isinstance(
            products,
            dict
        ):

            return False


        for product in EXPECTED_PRODUCTS:

            if product not in products:
                return False


            paths = (
                products[
                    product
                ]
            )


            if isinstance(
                paths,
                (str, Path)
            ):

                paths = [
                    paths
                ]


            if (
                not isinstance(
                    paths,
                    (list, tuple)
                )
                or
                len(paths) == 0
            ):

                return False


            for path in paths:

                path = Path(
                    path
                )

                if not path.exists():
                    return False


    return True


# ============================================================
# 6. REUSE EXISTING CATALOG IF HEALTHY
# ============================================================

catalog_healthy = (
    "catalog" in globals()
    and
    raster_catalog_is_healthy(
        globals()[
            "catalog"
        ]
    )
)


if catalog_healthy:

    print(
        "Existing raster catalog is complete and healthy."
    )

    print(
        "Skipping recursive GeoTIFF scan."
    )


else:

    print(
        "Raster catalog missing, incomplete, or stale."
    )

    print(
        "Rebuilding raster catalog..."
    )


    # ========================================================
    # 7. FIND GEOTIFF FILES
    # ========================================================

    tif_files = list(
        PROJECT_DIR.rglob(
            "*.tif"
        )
    )

    tif_files += list(
        PROJECT_DIR.rglob(
            "*.tiff"
        )
    )


    # Remove duplicates, just in case
    tif_files = sorted(
        set(
            tif_files
        )
    )


    print(
        f"GeoTIFF files found: "
        f"{len(tif_files)}"
    )


    # ========================================================
    # 8. BUILD TEMPORARY CATALOG
    # ========================================================

    catalog_temp = defaultdict(
        lambda:
            defaultdict(list)
    )


    for path in tif_files:

        filename = (
            path.name
        )


        fire_id = (
            normalize_fire_id(
                filename
            )
        )


        product = (
            detect_product(
                filename
            )
        )


        if (
            fire_id is None
            or
            product is None
        ):

            continue


        catalog_temp[
            fire_id
        ][
            product
        ].append(
            path
        )


    # ========================================================
    # 9. CONVERT TO NORMAL DICT
    # ========================================================

    catalog = {

        fire_id: {

            product:
                sorted(
                    set(
                        Path(path)
                        for path in paths
                    )
                )

            for product, paths
            in products.items()

        }

        for fire_id, products
        in catalog_temp.items()
    }


    print(
        "Raster catalog rebuilt."
    )


# ============================================================
# 10. VALIDATE EXPECTED PRODUCTS
# ============================================================

missing_entries = []


for fire_id in expected_fire_ids:

    if fire_id not in catalog:

        missing_entries.append(
            (
                fire_id,
                "ALL PRODUCTS"
            )
        )

        continue


    for product in EXPECTED_PRODUCTS:

        paths = (
            catalog[
                fire_id
            ].get(
                product,
                []
            )
        )


        if isinstance(
            paths,
            (str, Path)
        ):

            paths = [
                paths
            ]


        if len(
            paths
        ) == 0:

            missing_entries.append(
                (
                    fire_id,
                    product
                )
            )

            continue


        missing_files = [
            Path(path)
            for path in paths
            if not Path(path).exists()
        ]


        if missing_files:

            missing_entries.append(
                (
                    fire_id,
                    f"{product} "
                    f"({len(missing_files)} missing file(s))"
                )
            )


# ============================================================
# 11. PRINT COMPACT CATALOG SUMMARY
# ============================================================

print(
    "\n"
    +
    "=" * 70
)

print(
    "RASTER CATALOG"
)

print(
    "=" * 70
)


for fire_id in expected_fire_ids:

    print(
        f"\n{fire_id}"
    )


    if fire_id not in catalog:

        print(
            "  NOT FOUND"
        )

        continue


    for product in EXPECTED_PRODUCTS:

        files = (
            catalog[
                fire_id
            ].get(
                product,
                []
            )
        )


        if isinstance(
            files,
            (str, Path)
        ):

            files = [
                files
            ]


        print(
            f"  {product:8s}: "
            f"{len(files)} file(s)"
        )


# ============================================================
# 12. FINAL CATALOG CHECK
# ============================================================

print(
    "\n"
    +
    "=" * 70
)


if len(
    missing_entries
) == 0:

    print(
        "SUCCESS:"
    )

    print(
        "All 12 fires have "
        "S2_pre, S2_post, S1_pre and S1_post."
    )

else:

    print(
        "MISSING OR INVALID CATALOG ENTRIES:"
    )

    for item in missing_entries:

        print(
            item
        )


    raise RuntimeError(
        "Raster catalog failed completeness check."
    )


# ============================================================
# 13. DEFINE GET_PRODUCT_PATHS ONLY IF NEEDED
# ============================================================

if (
    "get_product_paths" not in globals()
    or
    not callable(
        globals()[
            "get_product_paths"
        ]
    )
):

    def get_product_paths(
        fire_id,
        product
    ):

        fire_id = (
            normalize_fire_id(
                fire_id
            )
        )


        if fire_id not in catalog:

            raise KeyError(
                f"{fire_id} not found in catalog."
            )


        if product not in (
            catalog[
                fire_id
            ]
        ):

            raise KeyError(
                f"{product} not found "
                f"for {fire_id}."
            )


        paths = (
            catalog[
                fire_id
            ][
                product
            ]
        )


        if isinstance(
            paths,
            (str, Path)
        ):

            paths = [
                Path(
                    paths
                )
            ]

        else:

            paths = [
                Path(path)
                for path
                in paths
            ]


        if len(
            paths
        ) == 0:

            raise FileNotFoundError(
                f"No raster files found "
                f"for {fire_id}, {product}."
            )


        missing = [
            path
            for path in paths
            if not path.exists()
        ]


        if missing:

            raise FileNotFoundError(
                f"{fire_id}, {product}: "
                f"{len(missing)} catalog file(s) "
                f"no longer exist."
            )


        return paths


    print(
        "\nDefined get_product_paths()."
    )

else:

    print(
        "\nget_product_paths() already defined."
    )


# ============================================================
# 14. FINAL HEALTH CHECK
# ============================================================

if not raster_catalog_is_healthy(
    catalog
):

    raise RuntimeError(
        "Raster catalog failed final health check."
    )


# ============================================================
# 15. TEST CATALOG
# ============================================================

print(
    "\nTest: fire_04 / S2_post"
)


test_paths = (
    get_product_paths(
        "fire_04",
        "S2_post"
    )
)


for path in test_paths:

    print(
        path
    )


print(
    "\nRaster catalog ready."
)

In [ ]:
# ============================================================
# REQUIRED RASTER-CATALOG HELPERS FOR MODELING
# ============================================================

from pathlib import Path
import re


def normalize_fire_id(value):

    value = str(value).strip().lower()

    match = re.search(
        r"(\d+)",
        value
    )

    if match is None:
        return value

    number = int(
        match.group(1)
    )

    return f"fire_{number:02d}"


def get_product_paths(
    fire_id,
    product
):
    """
    Return raster path(s) for one fire/product combination.

    Works with catalog entries stored as:
    - list of paths
    - single Path/string
    - dict containing path/paths/file/files/filepath
    """

    fire_id = normalize_fire_id(
        fire_id
    )

    if fire_id not in catalog:

        raise KeyError(
            f"{fire_id} not found in catalog."
        )

    if product not in catalog[fire_id]:

        raise KeyError(
            f"{product} not found for {fire_id}."
        )

    item = catalog[
        fire_id
    ][
        product
    ]

    # --------------------------------------------------------
    # Single path
    # --------------------------------------------------------

    if isinstance(
        item,
        (str, Path)
    ):

        paths = [
            item
        ]


    # --------------------------------------------------------
    # Dictionary form
    # --------------------------------------------------------

    elif isinstance(
        item,
        dict
    ):

        paths = None

        for key in [
            "paths",
            "path",
            "files",
            "file",
            "filepath",
        ]:

            if key in item:

                paths = item[
                    key
                ]

                break


        if paths is None:

            raise ValueError(
                f"Cannot determine raster paths for "
                f"{fire_id}, {product}."
            )


        if isinstance(
            paths,
            (str, Path)
        ):

            paths = [
                paths
            ]


    # --------------------------------------------------------
    # List / tuple / iterable
    # --------------------------------------------------------

    else:

        paths = list(
            item
        )


    paths = [
        Path(path)
        for path in paths
    ]


    if len(paths) == 0:

        raise FileNotFoundError(
            f"No raster files found for "
            f"{fire_id}, {product}."
        )


    for path in paths:

        if not path.exists():

            raise FileNotFoundError(
                f"Raster does not exist:\n{path}"
            )


    return paths


# ============================================================
# QUICK TEST
# ============================================================

print(
    get_product_paths(
        "fire_04",
        "S2_post"
    )
)

## Usage monitoring funcion

In [ ]:
# ============================================================
# RESOURCE MONITORING FOR MODEL TRAINING AND PREDICTION
#
# Records:
# - elapsed time
# - CPU utilization
# - process RAM
# - system RAM
# - GPU utilization
# - GPU memory
# - GPU power
# - trainable parameters
# - total parameters
#
# Designed for:
# - Random Forest
# - U-Net
# - Siamese network
# - Prithvi frozen backbone
# - Prithvi LoRA
# ============================================================

from pathlib import Path
import os
import time
import gc
import csv
import threading
import subprocess
import warnings

import numpy as np
import pandas as pd
import psutil


# ============================================================
# 1. OUTPUT PATH
# ============================================================

PROJECT_DIR = PROJECT_DIR

EFFICIENCY_DIR = (
    PROJECT_DIR / "efficiency"
)

EFFICIENCY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EFFICIENCY_CSV = (
    EFFICIENCY_DIR /
    "efficiency_metrics.csv"
)


# ============================================================
# 2. GPU INFORMATION
# ============================================================

def get_gpu_snapshot():
    """
    Query NVIDIA GPU using nvidia-smi.

    Returns:
        gpu_util_percent
        gpu_memory_used_mb
        gpu_memory_total_mb
        gpu_power_w

    Returns None values when no NVIDIA GPU is available.
    """

    command = [
        "nvidia-smi",
        "--query-gpu="
        "utilization.gpu,"
        "memory.used,"
        "memory.total,"
        "power.draw",
        "--format=csv,noheader,nounits"
    ]

    try:

        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=True,
            timeout=5
        )

        line = (
            result.stdout
            .strip()
            .splitlines()[0]
        )

        values = [
            x.strip()
            for x in line.split(",")
        ]

        return {
            "gpu_util_percent":
                float(values[0]),

            "gpu_memory_used_mb":
                float(values[1]),

            "gpu_memory_total_mb":
                float(values[2]),

            "gpu_power_w":
                float(values[3]),
        }

    except Exception:

        return {
            "gpu_util_percent": None,
            "gpu_memory_used_mb": None,
            "gpu_memory_total_mb": None,
            "gpu_power_w": None,
        }


# ============================================================
# 3. MODEL PARAMETER COUNTS
# ============================================================

def get_model_parameter_counts(
    model
):
    """
    Works with:
    - PyTorch
    - TensorFlow / Keras

    Returns:
        total_parameters
        trainable_parameters
    """

    if model is None:

        return {
            "total_parameters": None,
            "trainable_parameters": None,
        }


    # --------------------------------------------------------
    # PyTorch
    # --------------------------------------------------------

    try:

        import torch

        if isinstance(
            model,
            torch.nn.Module
        ):

            total = sum(
                p.numel()
                for p in model.parameters()
            )

            trainable = sum(
                p.numel()
                for p in model.parameters()
                if p.requires_grad
            )

            return {
                "total_parameters":
                    int(total),

                "trainable_parameters":
                    int(trainable),
            }

    except Exception:
        pass


    # --------------------------------------------------------
    # TensorFlow / Keras
    # --------------------------------------------------------

    try:

        total = int(
            model.count_params()
        )

        trainable = int(
            np.sum(
                [
                    np.prod(
                        v.shape
                    )
                    for v
                    in model.trainable_weights
                ]
            )
        )

        return {
            "total_parameters":
                total,

            "trainable_parameters":
                trainable,
        }

    except Exception:
        pass


    # --------------------------------------------------------
    # RF / sklearn etc.
    # --------------------------------------------------------

    return {
        "total_parameters": None,
        "trainable_parameters": None,
    }


# ============================================================
# 4. RESOURCE MONITOR CLASS
# ============================================================

class ResourceMonitor:

    def __init__(
        self,
        interval_seconds=0.5
    ):

        self.interval_seconds = (
            interval_seconds
        )

        self.running = False

        self.records = []

        self.process = (
            psutil.Process(
                os.getpid()
            )
        )


    # --------------------------------------------------------
    # Internal sampling function
    # --------------------------------------------------------

    def _collect(self):

        while self.running:

            timestamp = time.time()


            # ------------------------------------------------
            # CPU
            # ------------------------------------------------

            cpu_percent = (
                psutil.cpu_percent(
                    interval=None
                )
            )


            process_cpu = (
                self.process
                .cpu_percent(
                    interval=None
                )
            )


            # ------------------------------------------------
            # RAM
            # ------------------------------------------------

            process_ram_gb = (
                self.process
                .memory_info()
                .rss
                /
                1024**3
            )


            virtual_memory = (
                psutil.virtual_memory()
            )


            system_ram_used_gb = (
                virtual_memory.used
                /
                1024**3
            )


            system_ram_percent = (
                virtual_memory.percent
            )


            # ------------------------------------------------
            # GPU
            # ------------------------------------------------

            gpu = (
                get_gpu_snapshot()
            )


            self.records.append({

                "timestamp":
                    timestamp,

                "cpu_system_percent":
                    cpu_percent,

                "cpu_process_percent":
                    process_cpu,

                "process_ram_gb":
                    process_ram_gb,

                "system_ram_used_gb":
                    system_ram_used_gb,

                "system_ram_percent":
                    system_ram_percent,

                "gpu_util_percent":
                    gpu[
                        "gpu_util_percent"
                    ],

                "gpu_memory_used_mb":
                    gpu[
                        "gpu_memory_used_mb"
                    ],

                "gpu_memory_total_mb":
                    gpu[
                        "gpu_memory_total_mb"
                    ],

                "gpu_power_w":
                    gpu[
                        "gpu_power_w"
                    ],

            })


            time.sleep(
                self.interval_seconds
            )


    # --------------------------------------------------------
    # Start monitoring
    # --------------------------------------------------------

    def start(self):

        self.records = []

        self.running = True

        self.start_time = (
            time.perf_counter()
        )


        # initialize psutil CPU measurements
        psutil.cpu_percent(
            interval=None
        )

        self.process.cpu_percent(
            interval=None
        )


        self.thread = (
            threading.Thread(
                target=self._collect,
                daemon=True
            )
        )


        self.thread.start()


    # --------------------------------------------------------
    # Stop monitoring
    # --------------------------------------------------------

    def stop(self):

        self.end_time = (
            time.perf_counter()
        )

        self.running = False


        if hasattr(
            self,
            "thread"
        ):

            self.thread.join(
                timeout=5
            )


        elapsed = (
            self.end_time
            -
            self.start_time
        )


        return (
            self.summarize(
                elapsed
            )
        )


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    def summarize(
        self,
        elapsed_seconds
    ):

        if len(
            self.records
        ) == 0:

            return {
                "elapsed_sec":
                    elapsed_seconds
            }


        df = pd.DataFrame(
            self.records
        )


        result = {

            "elapsed_sec":
                elapsed_seconds,

            # CPU
            "mean_cpu_system_percent":
                df[
                    "cpu_system_percent"
                ].mean(),

            "peak_cpu_system_percent":
                df[
                    "cpu_system_percent"
                ].max(),

            "mean_cpu_process_percent":
                df[
                    "cpu_process_percent"
                ].mean(),

            "peak_cpu_process_percent":
                df[
                    "cpu_process_percent"
                ].max(),

            # RAM
            "mean_process_ram_gb":
                df[
                    "process_ram_gb"
                ].mean(),

            "peak_process_ram_gb":
                df[
                    "process_ram_gb"
                ].max(),

            "mean_system_ram_gb":
                df[
                    "system_ram_used_gb"
                ].mean(),

            "peak_system_ram_gb":
                df[
                    "system_ram_used_gb"
                ].max(),

            "peak_system_ram_percent":
                df[
                    "system_ram_percent"
                ].max(),

        }


        # ----------------------------------------------------
        # GPU metrics
        # ----------------------------------------------------

        gpu_util = pd.to_numeric(
            df[
                "gpu_util_percent"
            ],
            errors="coerce"
        )


        gpu_mem = pd.to_numeric(
            df[
                "gpu_memory_used_mb"
            ],
            errors="coerce"
        )


        gpu_power = pd.to_numeric(
            df[
                "gpu_power_w"
            ],
            errors="coerce"
        )


        result[
            "mean_gpu_util_percent"
        ] = (
            gpu_util.mean()
            if gpu_util.notna().any()
            else None
        )


        result[
            "peak_gpu_util_percent"
        ] = (
            gpu_util.max()
            if gpu_util.notna().any()
            else None
        )


        result[
            "mean_gpu_memory_gb"
        ] = (
            gpu_mem.mean()
            /
            1024
            if gpu_mem.notna().any()
            else None
        )


        result[
            "peak_gpu_memory_gb"
        ] = (
            gpu_mem.max()
            /
            1024
            if gpu_mem.notna().any()
            else None
        )


        result[
            "mean_gpu_power_w"
        ] = (
            gpu_power.mean()
            if gpu_power.notna().any()
            else None
        )


        result[
            "peak_gpu_power_w"
        ] = (
            gpu_power.max()
            if gpu_power.notna().any()
            else None
        )


        # ----------------------------------------------------
        # Approximate GPU energy
        #
        # Wh = W × hours
        # ----------------------------------------------------

        if (
            gpu_power.notna().any()
            and
            elapsed_seconds > 0
        ):

            result[
                "gpu_energy_wh_est"
            ] = (
                gpu_power.mean()
                *
                elapsed_seconds
                /
                3600
            )

        else:

            result[
                "gpu_energy_wh_est"
            ] = None


        return result


# ============================================================
# 5. SAVE ONE EFFICIENCY RECORD
# ============================================================

def save_efficiency_record(
    method,
    input_config,
    stage,
    metrics,
    model=None,
    fire_id=None,
    n_samples=None,
    n_pixels=None,
    notes=None,
):
    """Schema-safe upsert for training and prediction efficiency."""

    parameter_counts = {}
    if model is not None:
        try:
            parameter_counts = get_model_parameter_counts(model) or {}
        except Exception:
            parameter_counts = {}

    record = {
        "method": method,
        "input_config": input_config,
        "stage": stage,
        "fire_id": fire_id,
        "n_samples": n_samples,
        "n_pixels": n_pixels,
        **parameter_counts,
        **(metrics or {}),
        "notes": notes,
    }

    new_df = pd.DataFrame([record])

    if EFFICIENCY_CSV.exists():
        try:
            old_df = pd.read_csv(EFFICIENCY_CSV)
        except Exception:
            old_df = pd.DataFrame()

        if not old_df.empty:
            matching = pd.Series(True, index=old_df.index)
            for key in ["method", "input_config", "stage", "fire_id"]:
                if key not in old_df.columns:
                    matching &= False
                    continue
                value = record.get(key)
                if value is None:
                    matching &= old_df[key].isna()
                else:
                    matching &= old_df[key].astype(str).eq(str(value))
            old_df = old_df.loc[~matching].copy()

        final_df = pd.concat([old_df, new_df], ignore_index=True, sort=False)
    else:
        final_df = new_df

    final_df.to_csv(EFFICIENCY_CSV, index=False)
    print(f"Saved efficiency metrics: {method} | {input_config} | {stage}")
    return record


# ============================================================
# 6. TRAINING WRAPPER
# ============================================================

def monitor_training(
    train_function,
    method,
    input_config,
    model=None,
    n_samples=None,
    notes=None
):
    """
    train_function must be a zero-argument callable.

    Example:
        lambda: model.fit(...)
    """

    gc.collect()


    try:

        import torch

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

            torch.cuda.reset_peak_memory_stats()

    except Exception:
        pass


    monitor = ResourceMonitor(
        interval_seconds=0.5
    )


    monitor.start()


    try:

        result = train_function()

    finally:

        metrics = monitor.stop()


    record = save_efficiency_record(

        method=method,

        input_config=input_config,

        stage="training",

        metrics=metrics,

        model=model,

        n_samples=n_samples,

        notes=notes,

    )


    return (
        result,
        record
    )


# ============================================================
# 7. PREDICTION WRAPPER
# ============================================================

def monitor_prediction(
    predict_function,
    method,
    input_config,
    model=None,
    fire_id=None,
    n_pixels=None,
    notes=None
):
    """
    predict_function must be a zero-argument callable.

    Prediction should normally be performed over one complete
    held-out fire extent.
    """

    gc.collect()


    try:

        import torch

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

            torch.cuda.reset_peak_memory_stats()

    except Exception:
        pass


    monitor = ResourceMonitor(
        interval_seconds=0.5
    )


    monitor.start()


    try:

        prediction = (
            predict_function()
        )

    finally:

        metrics = (
            monitor.stop()
        )


    # --------------------------------------------------------
    # Useful normalized inference metric
    # --------------------------------------------------------

    if (
        n_pixels is not None
        and
        n_pixels > 0
    ):

        metrics[
            "inference_sec_per_million_pixels"
        ] = (
            metrics[
                "elapsed_sec"
            ]
            /
            (
                n_pixels
                /
                1_000_000
            )
        )

    else:

        metrics[
            "inference_sec_per_million_pixels"
        ] = None


    record = save_efficiency_record(

        method=method,

        input_config=input_config,

        stage="prediction",

        metrics=metrics,

        model=model,

        fire_id=fire_id,

        n_pixels=n_pixels,

        notes=notes,

    )


    return (
        prediction,
        record
    )


# ============================================================
# 8. EXAMPLE — RANDOM FOREST TRAINING
# ============================================================

# Example only:
#
# rf = RandomForestClassifier(...)
#
# _, rf_train_efficiency = monitor_training(
#
#     train_function=lambda:
#         rf.fit(
#             X_train,
#             y_train
#         ),
#
#     method="Random Forest",
#
#     input_config="Post-fire S2",
#
#     model=rf,
#
#     n_samples=len(y_train),
#
# )


# ============================================================
# 9. EXAMPLE — RANDOM FOREST PREDICTION
# ============================================================

# prediction, rf_prediction_efficiency = (
#     monitor_prediction(
#
#         predict_function=lambda:
#             rf.predict(
#                 X_test
#             ),
#
#         method="Random Forest",
#
#         input_config="Post-fire S2",
#
#         model=rf,
#
#         fire_id="fire_01",
#
#         n_pixels=len(X_test),
#
#     )
# )


# ============================================================
# 10. EXAMPLE — KERAS / U-NET TRAINING
# ============================================================

# history, unet_train_efficiency = (
#     monitor_training(
#
#         train_function=lambda:
#             model.fit(
#                 train_dataset,
#                 validation_data=val_dataset,
#                 epochs=50,
#                 callbacks=callbacks
#             ),
#
#         method="U-Net",
#
#         input_config="Pre + Post S2",
#
#         model=model,
#
#         n_samples=len(train_samples),
#
#     )
# )


# ============================================================
# 11. EXAMPLE — KERAS / U-NET PREDICTION
# ============================================================

# prediction, unet_pred_efficiency = (
#     monitor_prediction(
#
#         predict_function=lambda:
#             model.predict(
#                 test_dataset
#             ),
#
#         method="U-Net",
#
#         input_config="Pre + Post S2",
#
#         model=model,
#
#         fire_id="fire_01",
#
#         n_pixels=test_valid_pixels,
#
#     )
# )


# ============================================================
# 12. EXAMPLE — PYTORCH / PRITHVI TRAINING
# ============================================================

# trained_model, prithvi_efficiency = (
#     monitor_training(
#
#         train_function=lambda:
#             train_prithvi(
#                 model,
#                 train_loader,
#                 val_loader
#             ),
#
#         method="Prithvi LoRA",
#
#         input_config="Pre + Post S2",
#
#         model=model,
#
#         n_samples=len(train_samples),
#
#     )
# )


# ============================================================
# 13. CHECK SAVED RESULTS
# ============================================================

def show_efficiency_results():

    if not EFFICIENCY_CSV.exists():

        print(
            "No efficiency records yet."
        )

        return


    df = pd.read_csv(
        EFFICIENCY_CSV
    )


    display(
        df
    )


    return df


print(
    "Resource-monitoring system ready."
)

print(
    "Efficiency results will be saved to:"
)

print(
    EFFICIENCY_CSV
)

## 1. Common memory-safe data loader

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = PROJECT_DIR

SAMPLING_DIR = (
    PROJECT_DIR / "sampling"
)

PER_FIRE_DIR = (
    SAMPLING_DIR / "per_fire"
)

SAMPLE_CSV = (
    SAMPLING_DIR /
    "common_training_samples.csv"
)


# ------------------------------------------------------------
# Find all saved per-fire sample CSVs
# ------------------------------------------------------------

sample_files = sorted(
    PER_FIRE_DIR.glob(
        "*_samples.csv"
    )
)

print(
    f"Found {len(sample_files)} "
    "per-fire sample files:"
)

for path in sample_files:
    print(
        " -",
        path.name
    )


if len(sample_files) == 0:

    raise FileNotFoundError(
        f"No per-fire sample CSV files found in:\n"
        f"{PER_FIRE_DIR}"
    )


# ------------------------------------------------------------
# Combine them
# ------------------------------------------------------------

master_samples = pd.concat(
    [
        pd.read_csv(path)
        for path in sample_files
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# Save master CSV
# ------------------------------------------------------------

master_samples.to_csv(
    SAMPLE_CSV,
    index=False
)


print(
    "\nMaster sampling CSV created:"
)

print(
    SAMPLE_CSV
)

print(
    "\nTotal samples:",
    len(master_samples)
)


# ------------------------------------------------------------
# Check contents
# ------------------------------------------------------------

print(
    "\nSamples by split:"
)

print(
    master_samples[
        "split"
    ].value_counts()
)


print(
    "\nSamples by fire:"
)

print(
    master_samples
    .groupby(
        [
            "split",
            "fire_id"
        ]
    )
    .size()
)


print(
    "\nSamples by stratum:"
)

print(
    master_samples[
        "stratum"
    ].value_counts()
)

In [ ]:
# ============================================================
# MODELING BLOCK A
# COMMON MEMORY-SAFE PATCH READER
#
# UPDATED:
# Sentinel-2 is already exported as reflectance during EO
# preprocessing, so NO additional division by 10,000 is used.
# ============================================================


from pathlib import Path
from contextlib import ExitStack

import gc
import math
import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    jaccard_score,
)


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = PROJECT_DIR

SAMPLE_CSV = (
    PROJECT_DIR /
    "sampling/common_training_samples.csv"
)

MODEL_DIR = (
    PROJECT_DIR /
    "models"
)

RESULT_DIR = (
    PROJECT_DIR /
    "model_results"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_CSV = (
    RESULT_DIR /
    "accuracy_results.csv"
)


# ============================================================
# LOAD COMMON SAMPLES
# ============================================================

master_samples = pd.read_csv(
    SAMPLE_CSV
)

master_samples[
    "fire_id"
] = (
    master_samples[
        "fire_id"
    ]
    .astype(str)
)

train_samples = (
    master_samples[
        master_samples[
            "split"
        ]
        ==
        "train"
    ]
    .reset_index(
        drop=True
    )
)

val_samples = (
    master_samples[
        master_samples[
            "split"
        ]
        ==
        "validation"
    ]
    .reset_index(
        drop=True
    )
)

print(
    "Training patches:",
    len(
        train_samples
    )
)

print(
    "Validation patches:",
    len(
        val_samples
    )
)


# ============================================================
# INPUT CONFIGURATIONS
# ============================================================

S2_BANDS = [
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12",
]

S1_BANDS = [
    "VV",
    "VH",
]

PRITHVI_BANDS = [
    "B2",
    "B3",
    "B4",
    "B8A",
    "B11",
    "B12",
]


BAND_INDEX = {

    "S2_pre": {
        name: i + 1
        for i, name
        in enumerate(
            S2_BANDS
        )
    },

    "S2_post": {
        name: i + 1
        for i, name
        in enumerate(
            S2_BANDS
        )
    },

    "S1_pre": {
        "VV": 1,
        "VH": 2,
    },

    "S1_post": {
        "VV": 1,
        "VH": 2,
    },
}


INPUT_CONFIGS = {

    "post_s2": [
        (
            "S2_post",
            S2_BANDS
        ),
    ],

    "prepost_s2": [
        (
            "S2_pre",
            S2_BANDS
        ),
        (
            "S2_post",
            S2_BANDS
        ),
    ],

    "prepost_s2s1": [
        (
            "S2_pre",
            S2_BANDS
        ),
        (
            "S2_post",
            S2_BANDS
        ),
        (
            "S1_pre",
            S1_BANDS
        ),
        (
            "S1_post",
            S1_BANDS
        ),
    ],
}


# ============================================================
# PATCH GRID FROM CSV ROW
# ============================================================

def get_patch_grid(
    sample_row
):

    size = int(
        sample_row[
            "patch_size_pixels"
        ]
    )

    pixel_size = float(
        sample_row[
            "pixel_size_m"
        ]
    )

    x = float(
        sample_row[
            "x"
        ]
    )

    y = float(
        sample_row[
            "y"
        ]
    )

    width_m = (
        size
        *
        pixel_size
    )

    xmin = (
        x
        -
        width_m / 2
    )

    ymax = (
        y
        +
        width_m / 2
    )

    transform = from_origin(
        xmin,
        ymax,
        pixel_size,
        pixel_size,
    )

    return (
        transform,
        size,
        size,
    )


# ============================================================
# READ ONLY ONE SMALL WINDOW FROM A PRODUCT
# ============================================================

def read_product_to_grid(
    fire_id,
    product,
    band_names,
    dst_transform,
    height,
    width,
):
    """
    Memory-safe reading.

    Only the requested patch is reprojected from disk.
    The complete raster is never loaded.
    """

    paths = (
        get_product_paths(
            fire_id,
            product
        )
    )

    indexes = [

        BAND_INDEX[
            product
        ][
            band
        ]

        for band
        in band_names
    ]

    output = np.full(
        (
            len(
                indexes
            ),
            height,
            width,
        ),
        np.nan,
        dtype=np.float32,
    )

    dst_crs = None


    # --------------------------------------------------------
    # Determine target CRS
    # --------------------------------------------------------

    with rasterio.open(
        paths[
            0
        ]
    ) as src:

        dst_crs = (
            src.crs
        )


    # --------------------------------------------------------
    # Read each source only into requested grid
    # --------------------------------------------------------

    for path in paths:

        with rasterio.open(
            path
        ) as src:

            for j, band_index in enumerate(
                indexes
            ):

                temp = np.full(
                    (
                        height,
                        width,
                    ),
                    np.nan,
                    dtype=np.float32,
                )

                reproject(

                    source=
                        rasterio.band(
                            src,
                            band_index
                        ),

                    destination=
                        temp,

                    src_transform=
                        src.transform,

                    src_crs=
                        src.crs,

                    dst_transform=
                        dst_transform,

                    dst_crs=
                        dst_crs,

                    src_nodata=
                        src.nodata,

                    dst_nodata=
                        np.nan,

                    resampling=
                        Resampling.bilinear,
                )

                # ------------------------------------------------
                # Fill only pixels not already populated
                # ------------------------------------------------

                use = (
                    np.isnan(
                        output[
                            j
                        ]
                    )
                    &
                    np.isfinite(
                        temp
                    )
                )

                output[
                    j
                ][
                    use
                ] = (
                    temp[
                        use
                    ]
                )

                del temp


    return (
        output,
        dst_crs
    )


# ============================================================
# NORMALIZATION
#
# IMPORTANT:
# Sentinel-2 was already scaled during Earth Engine
# preprocessing using:
#
#     spectral.multiply(0.0001)
#
# Therefore Sentinel-2 MUST NOT be divided by 10000 here.
# ============================================================

def normalize_product(
    data,
    product
):
    """
    Sentinel-2:
        Already exported as reflectance.
        Keep on reflectance scale and clip only.

    Sentinel-1:
        Clip typical dB interval and scale approximately 0-1.
    """

    data = (
        data.astype(
            np.float32,
            copy=False
        )
    )


    # ========================================================
    # SENTINEL-2
    # Already in reflectance units
    # ========================================================

    if product.startswith(
        "S2"
    ):

        data = np.clip(
            data,
            -0.2,
            1.5
        )


    # ========================================================
    # SENTINEL-1
    # dB -> approximately 0-1
    # ========================================================

    elif product.startswith(
        "S1"
    ):

        data = np.clip(
            data,
            -30.0,
            5.0
        )

        data = (
            data
            +
            30.0
        ) / 35.0


    else:

        raise ValueError(
            f"Unknown product: "
            f"{product}"
        )


    return data


# ============================================================
# READ ONE INPUT CONFIGURATION
# ============================================================

def read_input_patch(
    sample_row,
    input_config
):

    (
        transform,
        height,
        width
    ) = (
        get_patch_grid(
            sample_row
        )
    )

    fire_id = (
        sample_row[
            "fire_id"
        ]
    )

    arrays = []

    common_valid = np.ones(
        (
            height,
            width
        ),
        dtype=bool
    )

    target_crs = None


    for (
        product,
        bands
    ) in (
        INPUT_CONFIGS[
            input_config
        ]
    ):

        (
            data,
            crs
        ) = (
            read_product_to_grid(
                fire_id,
                product,
                bands,
                transform,
                height,
                width,
            )
        )


        # ----------------------------------------------------
        # Apply product normalization
        # ----------------------------------------------------

        data = normalize_product(
            data,
            product
        )


        # ----------------------------------------------------
        # Common validity mask
        # ----------------------------------------------------

        valid = (
            np.isfinite(
                data
            )
            .all(
                axis=0
            )
        )

        common_valid &= (
            valid
        )


        # ----------------------------------------------------
        # Replace NaN only after valid mask is created
        # ----------------------------------------------------

        data = np.nan_to_num(
            data,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )


        arrays.append(
            data
        )

        target_crs = (
            crs
        )


    # ========================================================
    # STACK PRODUCTS
    # ========================================================

    x = np.concatenate(
        arrays,
        axis=0
    )


    # ========================================================
    # CHANNELS FIRST -> CHANNELS LAST
    # ========================================================

    x = np.moveaxis(
        x,
        0,
        -1
    )


    return (
        x.astype(
            np.float32
        ),
        common_valid,
        transform,
        target_crs,
    )


# ============================================================
# REFERENCE MASK
# ============================================================

_reference_cache = {}


def get_reference_geometries(
    fire_id,
    target_crs
):
    """
    Returns:
        current-fire polygon
        eligible evaluation region

    Previous 2-year scars are removed only from the
    unburned reference region.
    """

    key = (
        fire_id,
        str(
            target_crs
        )
    )


    if key in (
        _reference_cache
    ):

        return (
            _reference_cache[
                key
            ]
        )


    fire_geom = (
        get_event_geometry(
            fire_polygons,
            fire_id,
            target_crs
        )
    )


    unburned_geom = (
        get_event_geometry(
            candidate_unburned,
            fire_id,
            target_crs
        )
    )


    recent_scars = (
        get_recent_previous_fire_geometry(
            fire_id,
            target_crs
        )
    )


    if (
        recent_scars
        is not None
    ):

        unburned_geom = (
            unburned_geom
            .difference(
                recent_scars
            )
        )


    eligible_geom = (
        fire_geom.union(
            unburned_geom
        )
    )


    _reference_cache[
        key
    ] = (
        fire_geom,
        eligible_geom,
    )


    return (
        fire_geom,
        eligible_geom,
    )


# ============================================================
# CREATE REFERENCE PATCH
# ============================================================

def create_reference_patch(
    fire_id,
    target_crs,
    transform,
    height,
    width,
    valid_mask,
):

    (
        fire_geom,
        eligible_geom
    ) = (
        get_reference_geometries(
            fire_id,
            target_crs
        )
    )


    # ========================================================
    # CURRENT FIRE = BURNED CLASS
    # ========================================================

    burned = rasterize(
        [
            (
                fire_geom,
                1
            )
        ],

        out_shape=(
            height,
            width
        ),

        transform=
            transform,

        fill=
            0,

        dtype=
            "uint8",
    )


    # ========================================================
    # ELIGIBLE EVALUATION REGION
    # ========================================================

    eligible = rasterize(
        [
            (
                eligible_geom,
                1
            )
        ],

        out_shape=(
            height,
            width
        ),

        transform=
            transform,

        fill=
            0,

        dtype=
            "uint8",

    ).astype(
        bool
    )


    # ========================================================
    # VALID PIXELS ONLY
    # ========================================================

    weight = (
        eligible
        &
        valid_mask
    )


    return (
        burned.astype(
            np.float32
        ),

        weight.astype(
            np.float32
        ),
    )


# ============================================================
# ONE COMPLETE SAMPLE
# ============================================================

def load_sample(
    sample_row,
    input_config
):

    (
        x,
        valid,
        transform,
        crs
    ) = (
        read_input_patch(
            sample_row,
            input_config
        )
    )


    (
        y,
        weight
    ) = (
        create_reference_patch(

            sample_row[
                "fire_id"
            ],

            crs,

            transform,

            x.shape[
                0
            ],

            x.shape[
                1
            ],

            valid,
        )
    )


    return (
        x,
        y,
        weight,
    )


# ============================================================
# QUICK NORMALIZATION CHECK
#
# This verifies the new Sentinel-2 scale before U-Net.
# ============================================================

print(
    "\n"
    +
    "=" * 75
)

print(
    "MODELING BLOCK A — NORMALIZATION CHECK"
)

print(
    "=" * 75
)


test_row = (
    train_samples
    .iloc[
        0
    ]
)


test_x, test_y, test_weight = (
    load_sample(
        test_row,
        "post_s2"
    )
)


print(
    "\nTest patch shape:",
    test_x.shape
)


print(
    "Sentinel-2 minimum:",
    float(
        np.nanmin(
            test_x
        )
    )
)


print(
    "Sentinel-2 maximum:",
    float(
        np.nanmax(
            test_x
        )
    )
)


print(
    "Sentinel-2 mean:",
    float(
        np.nanmean(
            test_x
        )
    )
)


print(
    "\nMask values:",
    np.unique(
        test_y
    )
)


print(
    "Eligible pixels:",
    int(
        np.sum(
            test_weight > 0
        )
    )
)


print(
    "\nMODELING BLOCK A READY."
)

## 2. Random Forest

The RF models are **retrained under the corrected Sentinel-2 reflectance convention**. Sentinel-2 GeoTIFFs were already converted to reflectance during Earth Engine export, so no second division by 10,000 is applied. Each RF model is saved with a metadata sidecar; full-fire inference refuses to load checkpoints created under an older preprocessing convention.


In [ ]:
# ============================================================
# MODELING BLOCK B
# RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier
import joblib
import json


RF_PIXELS_PER_PATCH = 256


# ============================================================
# RF PIPELINE VERSION
#
# This tag prevents RF models trained before the Sentinel-2
# reflectance correction from being reused accidentally.
# ============================================================

RF_PIPELINE_VERSION = "rf_reflectance_v2_no_s2_div10000"


def rf_metadata_path(input_config):
    return MODEL_DIR / f"RF_{input_config}.metadata.json"



def build_rf_dataset(
    samples,
    input_config,
    pixels_per_patch=256,
    seed=42
):

    rng = np.random.default_rng(
        seed
    )

    X_blocks = []
    y_blocks = []

    for i, row in (
        samples.iterrows()
    ):

        x, y, weight = (
            load_sample(
                row,
                input_config
            )
        )

        valid_index = np.flatnonzero(
            weight.ravel()
            >
            0
        )

        if len(valid_index) == 0:
            continue

        n = min(
            pixels_per_patch,
            len(valid_index)
        )

        selected = rng.choice(
            valid_index,
            size=n,
            replace=False
        )

        x_flat = x.reshape(
            -1,
            x.shape[-1]
        )

        y_flat = y.ravel()

        X_blocks.append(
            x_flat[
                selected
            ].astype(
                np.float32
            )
        )

        y_blocks.append(
            y_flat[
                selected
            ].astype(
                np.uint8
            )
        )

        del (
            x,
            y,
            weight,
            x_flat,
            y_flat
        )

        if i % 100 == 0:
            gc.collect()

    X = np.concatenate(
        X_blocks,
        axis=0
    )

    Y = np.concatenate(
        y_blocks,
        axis=0
    )

    del (
        X_blocks,
        y_blocks
    )

    gc.collect()

    return (
        X,
        Y
    )


def train_random_forest(
    input_config
):

    print(
        "\nBuilding RF training data:",
        input_config
    )

    X_train, y_train = (
        build_rf_dataset(
            train_samples,
            input_config,
            RF_PIXELS_PER_PATCH,
            seed=42
        )
    )

    print(
        "RF matrix:",
        X_train.shape
    )

    rf = RandomForestClassifier(
        n_estimators=300,

        max_features="sqrt",

        class_weight="balanced_subsample",

        random_state=42,

        n_jobs=2,

        max_depth=None,

        min_samples_leaf=1,
    )

    _, efficiency = (
        monitor_training(

            train_function=lambda:
                rf.fit(
                    X_train,
                    y_train
                ),

            method="Random Forest",

            input_config=input_config,

            model=rf,

            n_samples=len(
                y_train
            ),
        )
    )

    output = (
        MODEL_DIR /
        f"RF_{input_config}.joblib"
    )

    joblib.dump(
        rf,
        output
    )

    # --------------------------------------------------------
    # Save preprocessing/version metadata beside the model.
    # Full-fire inference checks this file before loading RF.
    # --------------------------------------------------------
    metadata = {
        "pipeline_version": RF_PIPELINE_VERSION,
        "input_config": input_config,
        "sentinel2_scale": "reflectance_already_scaled_in_GEE",
        "sentinel2_second_divide_by_10000": False,
        "sentinel1_normalization": "clip_-30_5_then_scale_0_1",
        "rf_pixels_per_patch": int(RF_PIXELS_PER_PATCH),
        "n_training_pixels": int(len(y_train)),
        "n_features": int(X_train.shape[1]),
    }

    with open(
        rf_metadata_path(input_config),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(metadata, f, indent=2)

    print("Saved RF model:", output)
    print("Saved RF metadata:", rf_metadata_path(input_config))

    del (
        X_train,
        y_train
    )

    gc.collect()

    return rf



In [ ]:
# ============================================================
# RANDOM FOREST — SAFE REUSE / CONDITIONAL RETRAINING
#
# Correct Sentinel-2 reflectance pipeline:
#   RF_PIPELINE_VERSION = "rf_reflectance_v2_no_s2_div10000"
#
# Behavior
# --------
# RF_FORCE_RETRAIN = False
#   -> reuse healthy compatible RF models
#   -> retrain only missing/incompatible configurations
#
# RF_FORCE_RETRAIN = True
#   -> remove RF artifacts/results
#   -> retrain all three configurations
# ============================================================

import json
import gc
import joblib
import pandas as pd


RF_FORCE_RETRAIN = False


RF_CONFIGS = [
    "post_s2",
    "prepost_s2",
    "prepost_s2s1",
]


# ============================================================
# 1. HEALTH CHECK FOR ONE RF CONFIGURATION
# ============================================================

def rf_model_is_healthy(
    config
):

    model_path = (
        MODEL_DIR /
        f"RF_{config}.joblib"
    )

    metadata_path = (
        MODEL_DIR /
        f"RF_{config}.metadata.json"
    )


    # --------------------------------------------------------
    # Required files must exist
    # --------------------------------------------------------

    if not model_path.exists():

        print(
            f"{config}: model missing."
        )

        return False


    if not metadata_path.exists():

        print(
            f"{config}: metadata missing."
        )

        return False


    # --------------------------------------------------------
    # Metadata must be readable and compatible
    # --------------------------------------------------------

    try:

        with open(
            metadata_path,
            "r"
        ) as f:

            metadata = json.load(
                f
            )

    except Exception as exc:

        print(
            f"{config}: metadata unreadable:",
            exc
        )

        return False


    pipeline_version = (
        metadata.get(
            "pipeline_version"
        )
    )


    if (
        pipeline_version
        !=
        RF_PIPELINE_VERSION
    ):

        print(
            f"{config}: incompatible pipeline version "
            f"({pipeline_version!r} != "
            f"{RF_PIPELINE_VERSION!r})."
        )

        return False


    # --------------------------------------------------------
    # Optional model-load validation
    # --------------------------------------------------------

    try:

        model = joblib.load(
            model_path
        )

    except Exception as exc:

        print(
            f"{config}: model could not be loaded:",
            exc
        )

        return False


    # --------------------------------------------------------
    # Basic fitted-model validation
    # --------------------------------------------------------

    if not hasattr(
        model,
        "predict"
    ):

        print(
            f"{config}: loaded object has no predict()."
        )

        del model
        gc.collect()

        return False


    if not hasattr(
        model,
        "estimators_"
    ):

        print(
            f"{config}: RF appears unfitted."
        )

        del model
        gc.collect()

        return False


    del model
    gc.collect()


    print(
        f"{config}: healthy compatible RF model found."
    )

    return True


# ============================================================
# 2. REMOVE OLD RF ARTIFACTS IF FORCE RETRAINING
# ============================================================

if RF_FORCE_RETRAIN:

    print(
        "\nRF_FORCE_RETRAIN=True"
    )

    print(
        "Removing RF checkpoints/results before retraining..."
    )


    # --------------------------------------------------------
    # RF models and metadata
    # --------------------------------------------------------

    for config in RF_CONFIGS:

        for path in [

            MODEL_DIR /
            f"RF_{config}.joblib",

            MODEL_DIR /
            f"RF_{config}.metadata.json",

        ]:

            if path.exists():

                path.unlink()

                print(
                    "Deleted:",
                    path
                )


    # --------------------------------------------------------
    # RF prediction rasters
    # --------------------------------------------------------

    rf_pred_dir = (
        RESULT_DIR /
        "RF_predictions"
    )


    if rf_pred_dir.exists():

        for path in rf_pred_dir.glob(
            "RF_*.tif"
        ):

            path.unlink()

            print(
                "Deleted:",
                path
            )


    # --------------------------------------------------------
    # RF accuracy / summary outputs
    # --------------------------------------------------------

    for path in [

        RESULT_DIR /
        "RF_test_accuracy.csv",

        RESULT_DIR /
        "RF_test_summary.csv",

        RESULT_DIR /
        "RF_accuracy_efficiency_combined.csv",

    ]:

        if path.exists():

            path.unlink()

            print(
                "Deleted:",
                path
            )


    # --------------------------------------------------------
    # Remove only RF rows from shared efficiency table
    # --------------------------------------------------------

    if EFFICIENCY_CSV.exists():

        eff = pd.read_csv(
            EFFICIENCY_CSV
        )


        if (
            "method"
            in eff.columns
        ):

            rf_mask = (
                eff[
                    "method"
                ]
                ==
                "Random Forest"
            )


            n_removed = int(
                rf_mask.sum()
            )


            if n_removed > 0:

                eff = (
                    eff[
                        ~rf_mask
                    ]
                    .copy()
                )


                eff.to_csv(
                    EFFICIENCY_CSV,
                    index=False
                )


                print(
                    f"Removed {n_removed} old Random Forest "
                    f"row(s) from:",
                    EFFICIENCY_CSV
                )


# ============================================================
# 3. DETERMINE WHICH RF MODELS NEED TRAINING
# ============================================================

configs_to_train = []


for config in RF_CONFIGS:

    if RF_FORCE_RETRAIN:

        configs_to_train.append(
            config
        )

        continue


    if not rf_model_is_healthy(
        config
    ):

        configs_to_train.append(
            config
        )


# ============================================================
# 4. TRAIN ONLY MISSING / INCOMPATIBLE CONFIGURATIONS
# ============================================================

if len(
    configs_to_train
) == 0:

    print(
        "\nAll Random Forest models are already "
        "healthy and compatible."
    )

    print(
        "Skipping RF retraining."
    )


else:

    print(
        "\nRF configurations requiring training:"
    )

    for config in configs_to_train:

        print(
            "  -",
            config
        )


    for config in configs_to_train:

        print(
            "\n"
            +
            "=" * 70
        )

        print(
            "Training Random Forest:",
            config
        )

        print(
            "=" * 70
        )


        model = train_random_forest(
            config
        )


        del model

        gc.collect()


# ============================================================
# 5. FINAL VALIDATION
# ============================================================

print(
    "\n"
    +
    "=" * 70
)

print(
    "FINAL RANDOM FOREST CHECK"
)

print(
    "=" * 70
)


rf_health = {}


for config in RF_CONFIGS:

    rf_health[
        config
    ] = (
        rf_model_is_healthy(
            config
        )
    )


if not all(
    rf_health.values()
):

    failed = [

        config

        for config, healthy
        in rf_health.items()

        if not healthy
    ]


    raise RuntimeError(
        "RF model validation failed for: "
        +
        ", ".join(
            failed
        )
    )


print(
    "\nAll RF models ready under pipeline version:"
)

print(
    RF_PIPELINE_VERSION
)

## 3. U-Net

In [ ]:
# ============================================================
# REQUIRED IMPORTS FOR U-NET EVALUATION
# ============================================================

import gc
import numpy as np
import pandas as pd
import rasterio
import tensorflow as tf

from rasterio.windows import Window

In [ ]:
# ============================================================
# MODELING BLOCK C
# U-NET — PRE + POST S2 TRAINING-TIME RECOVERY
#
# PURPOSE
# -------
# Re-run ONLY the U-Net Pre + Post Sentinel-2 configuration
# to recover its missing measured training-efficiency record.
#
# IMPORTANT
# ---------
# - Same training samples
# - Same validation samples
# - Same U-Net architecture
# - Same hyperparameters
# - Same random seed
# - Fresh model initialized from scratch
#
# This run DOES NOT replace the final model used for accuracy.
# A separate "_timing_only" checkpoint/history is saved.
#
# Required from previous common setup blocks:
#     load_sample()
#     ResourceMonitor
#     save_efficiency_record()
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path

import gc
import math
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import callbacks
from tensorflow.keras import optimizers
from tensorflow.keras.utils import Sequence


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

RANDOM_SEED = 42

np.random.seed(
    RANDOM_SEED
)

tf.random.set_seed(
    RANDOM_SEED
)


# ============================================================
# 3. PATHS
# ============================================================

PROJECT_DIR = PROJECT_DIR

SAMPLING_DIR = (
    PROJECT_DIR
    /
    "sampling"
)

MODEL_DIR = (
    PROJECT_DIR
    /
    "models"
)

RESULT_DIR = (
    PROJECT_DIR
    /
    "model_results"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SAMPLE_CSV = (
    SAMPLING_DIR
    /
    "common_training_samples.csv"
)


# ============================================================
# 4. U-NET SETTINGS
#
# SAME SETTINGS AS ORIGINAL U-NET TRAINING
# ============================================================

UNET_BATCH_SIZE = 4

UNET_EPOCHS = 50

UNET_LEARNING_RATE = 1e-3

UNET_PATIENCE = 8


# ============================================================
# 5. CONFIGURATION TO RE-RUN
#
# ONLY the missing training-efficiency configuration.
# ============================================================

INPUT_CONFIG = "prepost_s2"


# ============================================================
# 6. TIMING-ONLY OUTPUT PATHS
#
# These are intentionally different from the original final
# U-Net files so that no publication model is overwritten.
# ============================================================

TIMING_MODEL_PATH = (
    MODEL_DIR
    /
    "UNet_prepost_s2_timing_only.keras"
)

TIMING_HISTORY_PATH = (
    RESULT_DIR
    /
    "UNet_prepost_s2_timing_only_history.csv"
)


# ============================================================
# 7. READ COMMON SAMPLE TABLE
# ============================================================

unet_samples = pd.read_csv(
    SAMPLE_CSV
)


print(
    "=" * 75
)

print(
    "U-NET PRE + POST S2 — TRAINING-TIME RECOVERY"
)

print(
    "=" * 75
)


print(
    "\nTotal sampled patch centres:",
    len(
        unet_samples
    )
)


print(
    "\nSamples by split:"
)

print(
    unet_samples[
        "split"
    ]
    .value_counts()
)


# ============================================================
# 8. TRAIN / VALIDATION TABLES
# ============================================================

unet_train_samples = (
    unet_samples[
        unet_samples[
            "split"
        ]
        ==
        "train"
    ]
    .reset_index(
        drop=True
    )
)


unet_val_samples = (
    unet_samples[
        unet_samples[
            "split"
        ]
        ==
        "validation"
    ]
    .reset_index(
        drop=True
    )
)


print(
    "\nTraining samples:",
    len(
        unet_train_samples
    )
)


print(
    "Validation samples:",
    len(
        unet_val_samples
    )
)


# ============================================================
# 9. MEMORY-SAFE U-NET SEQUENCE
#
# Uses the existing common load_sample() function.
#
# Expected:
#
#     X, y, weight = load_sample(sample_row, input_config)
#
# X:
#     [H, W, C] or [C, H, W]
#
# y:
#     [H, W] or [H, W, 1]
#
# weight:
#     valid/evaluation pixel mask
# ============================================================

class UNetSequence(Sequence):

    def __init__(
        self,
        samples,
        input_config,
        batch_size=4,
        shuffle=True,
        seed=42,
    ):

        super().__init__()

        self.samples = (
            samples
            .reset_index(
                drop=True
            )
        )

        self.input_config = (
            input_config
        )

        self.batch_size = int(
            batch_size
        )

        self.shuffle = bool(
            shuffle
        )

        self.rng = (
            np.random.default_rng(
                seed
            )
        )

        self.indices = np.arange(
            len(
                self.samples
            )
        )

        self.on_epoch_end()


    def __len__(
        self
    ):

        return int(
            math.ceil(
                len(
                    self.samples
                )
                /
                self.batch_size
            )
        )


    def on_epoch_end(
        self
    ):

        if self.shuffle:

            self.rng.shuffle(
                self.indices
            )


    def __getitem__(
        self,
        batch_index
    ):

        batch_indices = (
            self.indices[
                batch_index
                *
                self.batch_size
                :
                (
                    batch_index
                    +
                    1
                )
                *
                self.batch_size
            ]
        )


        X_list = []
        y_list = []
        w_list = []


        for idx in batch_indices:

            sample_row = (
                self.samples.iloc[
                    int(
                        idx
                    )
                ]
            )


            loaded = load_sample(
                sample_row,
                self.input_config
            )


            if (
                not isinstance(
                    loaded,
                    (
                        tuple,
                        list
                    )
                )
                or
                len(
                    loaded
                )
                <
                3
            ):

                raise ValueError(
                    "load_sample() must return "
                    "(X, y, weight)."
                )


            X, y, weight = (
                loaded[
                    :3
                ]
            )


            X = np.asarray(
                X,
                dtype=np.float32
            )

            y = np.asarray(
                y,
                dtype=np.float32
            )

            weight = np.asarray(
                weight,
                dtype=np.float32
            )


            # ------------------------------------------------
            # Ensure channels-last input
            # ------------------------------------------------

            if X.ndim != 3:

                raise ValueError(
                    f"Unexpected X shape: "
                    f"{X.shape}"
                )


            if (
                X.shape[
                    0
                ]
                in
                (
                    10,
                    20,
                    24
                )
                and
                X.shape[
                    -1
                ]
                not in
                (
                    10,
                    20,
                    24
                )
            ):

                X = np.moveaxis(
                    X,
                    0,
                    -1
                )


            # ------------------------------------------------
            # Ensure mask shape H,W,1
            # ------------------------------------------------

            if y.ndim == 2:

                y = y[
                    ...,
                    None
                ]


            elif (
                y.ndim == 3
                and
                y.shape[
                    0
                ]
                ==
                1
                and
                y.shape[
                    -1
                ]
                !=
                1
            ):

                y = np.moveaxis(
                    y,
                    0,
                    -1
                )


            # ------------------------------------------------
            # Binary labels / weights
            # ------------------------------------------------

            y = (
                y
                >
                0.5
            ).astype(
                np.float32
            )


            weight = (
                weight
                >
                0
            ).astype(
                np.float32
            )


            # ------------------------------------------------
            # Remove non-finite inputs
            # ------------------------------------------------

            X = np.nan_to_num(
                X,
                nan=0.0,
                posinf=0.0,
                neginf=0.0
            )


            X_list.append(
                X
            )

            y_list.append(
                y
            )

            w_list.append(
                weight
            )


        return (
            np.stack(
                X_list,
                axis=0
            ),

            np.stack(
                y_list,
                axis=0
            ),

            np.stack(
                w_list,
                axis=0
            ),
        )


# ============================================================
# 10. U-NET ARCHITECTURE
#
# SAME ARCHITECTURE AS ORIGINAL MODEL
# ============================================================

def conv_block(
    x,
    filters
):

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(
        x
    )


    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(
        x
    )


    return x


def build_unet(
    input_shape
):

    inputs = layers.Input(
        shape=input_shape
    )


    # --------------------------------------------------------
    # Encoder
    # --------------------------------------------------------

    c1 = conv_block(
        inputs,
        32
    )

    p1 = layers.MaxPooling2D(
        2
    )(
        c1
    )


    c2 = conv_block(
        p1,
        64
    )

    p2 = layers.MaxPooling2D(
        2
    )(
        c2
    )


    c3 = conv_block(
        p2,
        128
    )

    p3 = layers.MaxPooling2D(
        2
    )(
        c3
    )


    # --------------------------------------------------------
    # Bottleneck
    # --------------------------------------------------------

    b = conv_block(
        p3,
        256
    )


    # --------------------------------------------------------
    # Decoder
    # --------------------------------------------------------

    u3 = layers.UpSampling2D(
        2
    )(
        b
    )

    u3 = layers.Concatenate()(
        [
            u3,
            c3
        ]
    )

    c4 = conv_block(
        u3,
        128
    )


    u2 = layers.UpSampling2D(
        2
    )(
        c4
    )

    u2 = layers.Concatenate()(
        [
            u2,
            c2
        ]
    )

    c5 = conv_block(
        u2,
        64
    )


    u1 = layers.UpSampling2D(
        2
    )(
        c5
    )

    u1 = layers.Concatenate()(
        [
            u1,
            c1
        ]
    )

    c6 = conv_block(
        u1,
        32
    )


    outputs = layers.Conv2D(
        1,
        1,
        activation="sigmoid"
    )(
        c6
    )


    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="unet"
    )


    model.compile(

        optimizer=
            optimizers.Adam(
                learning_rate=
                    UNET_LEARNING_RATE
            ),

        loss=
            "binary_crossentropy",

        metrics=[
            "accuracy"
        ],

    )


    return model


# ============================================================
# 11. CREATE TRAIN / VALIDATION SEQUENCES
# ============================================================

train_seq = UNetSequence(

    samples=
        unet_train_samples,

    input_config=
        INPUT_CONFIG,

    batch_size=
        UNET_BATCH_SIZE,

    shuffle=
        True,

    seed=
        RANDOM_SEED,
)


val_seq = UNetSequence(

    samples=
        unet_val_samples,

    input_config=
        INPUT_CONFIG,

    batch_size=
        UNET_BATCH_SIZE,

    shuffle=
        False,

    seed=
        RANDOM_SEED,
)


# ============================================================
# 12. GET INPUT SHAPE
# ============================================================

example_batch = (
    train_seq[
        0
    ]
)

X_example, y_example = (
    example_batch[
        :2
    ]
)


input_shape = (
    X_example.shape[
        1:
    ]
)


print(
    "\nInput configuration:",
    INPUT_CONFIG
)

print(
    "Input shape:",
    input_shape
)

print(
    "Mask shape:",
    y_example.shape[
        1:
    ]
)


# ============================================================
# 13. BUILD A FRESH MODEL
#
# IMPORTANT:
# Do not load the existing checkpoint.
# Training time must be measured from a fresh model.
# ============================================================

tf.keras.backend.clear_session()

np.random.seed(
    RANDOM_SEED
)

tf.random.set_seed(
    RANDOM_SEED
)


model = build_unet(
    input_shape=
        input_shape
)


print(
    "\nU-Net parameters:"
)

print(
    "Total:",
    f"{model.count_params():,}"
)


# ============================================================
# 14. CALLBACKS
# ============================================================

checkpoint = callbacks.ModelCheckpoint(

    filepath=
        TIMING_MODEL_PATH,

    monitor=
        "val_loss",

    save_best_only=
        True,

    mode=
        "min",

    verbose=
        1,
)


early_stop = callbacks.EarlyStopping(

    monitor=
        "val_loss",

    patience=
        UNET_PATIENCE,

    restore_best_weights=
        True,

    verbose=
        1,
)


reduce_lr = callbacks.ReduceLROnPlateau(

    monitor=
        "val_loss",

    factor=
        0.5,

    patience=
        3,

    min_lr=
        1e-6,

    verbose=
        1,
)


# ============================================================
# 15. TRAINING RESOURCE MONITOR
#
# The timer starts immediately before model.fit()
# and stops immediately after training finishes.
#
# This is the training time that will enter the
# computational-performance table.
# ============================================================

monitor = ResourceMonitor(
    interval_seconds=
        0.5
)


print(
    "\n"
    +
    "=" * 75
)

print(
    "STARTING U-NET PRE + POST S2 TIMING RUN"
)

print(
    "=" * 75
)


monitor.start()


try:

    history = model.fit(

        train_seq,

        validation_data=
            val_seq,

        epochs=
            UNET_EPOCHS,

        callbacks=[
            checkpoint,
            early_stop,
            reduce_lr,
        ],

        verbose=
            1,
    )


finally:

    efficiency_metrics = (
        monitor.stop()
    )


# ============================================================
# 16. PRINT MEASURED TRAINING TIME
# ============================================================

elapsed_sec = (
    efficiency_metrics[
        "elapsed_sec"
    ]
)


elapsed_min = (
    elapsed_sec
    /
    60.0
)


print(
    "\n"
    +
    "=" * 75
)

print(
    "MEASURED U-NET PRE + POST S2 TRAINING TIME"
)

print(
    "=" * 75
)


print(
    f"\nElapsed time: "
    f"{elapsed_sec:.2f} s"
)

print(
    f"Elapsed time: "
    f"{elapsed_min:.2f} min"
)


# ============================================================
# 17. SAVE TRAINING-EFFICIENCY RECORD
#
# This is the missing row required for the publication
# computational-performance table.
# ============================================================

save_efficiency_record(

    method=
        "U-Net",

    input_config=
        INPUT_CONFIG,

    stage=
        "training",

    metrics=
        efficiency_metrics,

    model=
        model,

    fire_id=
        None,

    n_pixels=
        None,

    notes=
        (
            "Timing-only rerun to recover missing "
            "U-Net Pre + Post S2 training efficiency; "
            f"batch_size={UNET_BATCH_SIZE}; "
            f"seed={RANDOM_SEED}"
        ),
)


# ============================================================
# 18. SAVE TIMING-ONLY TRAINING HISTORY
#
# Do NOT overwrite the original final-model history.
# ============================================================

history_df = pd.DataFrame(
    history.history
)


history_df.to_csv(
    TIMING_HISTORY_PATH,
    index=False
)


print(
    "\nTiming-only model saved:"
)

print(
    TIMING_MODEL_PATH
)


print(
    "\nTiming-only history saved:"
)

print(
    TIMING_HISTORY_PATH
)


# ============================================================
# 19. SUMMARY
# ============================================================

print(
    "\n"
    +
    "=" * 75
)

print(
    "U-NET PRE + POST S2 TIMING RUN COMPLETE"
)

print(
    "=" * 75
)


print(
    "\nConfiguration:",
    INPUT_CONFIG
)

print(
    "Training time (s):",
    round(
        elapsed_sec,
        2
    )
)

print(
    "Training time (min):",
    round(
        elapsed_min,
        2
    )
)


if (
    "peak_process_ram_gb"
    in
    efficiency_metrics
):

    print(
        "Peak RAM (GB):",
        round(
            efficiency_metrics[
                "peak_process_ram_gb"
            ],
            3
        )
    )


if (
    "mean_cpu_system_percent"
    in
    efficiency_metrics
):

    print(
        "Mean CPU utilization (%):",
        round(
            efficiency_metrics[
                "mean_cpu_system_percent"
            ],
            2
        )
    )


if (
    "peak_cpu_system_percent"
    in
    efficiency_metrics
):

    print(
        "Peak CPU utilization (%):",
        round(
            efficiency_metrics[
                "peak_cpu_system_percent"
            ],
            2
        )
    )


if (
    "mean_gpu_util_percent"
    in
    efficiency_metrics
):

    print(
        "Mean GPU utilization (%):",
        round(
            efficiency_metrics[
                "mean_gpu_util_percent"
            ],
            2
        )
    )


if (
    "peak_gpu_memory_gb"
    in
    efficiency_metrics
):

    print(
        "Peak GPU memory (GB):",
        round(
            efficiency_metrics[
                "peak_gpu_memory_gb"
            ],
            3
        )
    )


# ============================================================
# 20. CLEANUP
# ============================================================

del train_seq
del val_seq

gc.collect()


print(
    "\nDone."
)

print(
    "The original final U-Net checkpoint was NOT overwritten."
)

## 4. Siamese network

## Block D1 — Complete Siamese model and training definitions

In [ ]:
# ============================================================
# MODELING BLOCK D1
# SIAMESE CHANGE-DETECTION NETWORK
#
# UPDATED:
# - No Lambda layers
# - Uses serializable AbsoluteDifference layer
# - Compatible with model saving/loading
# - Uses current normalization pipeline
# - Supports:
#       prepost_s2
#       prepost_s2s1
# ============================================================


import gc
import math
import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# SERIALIZABLE ABSOLUTE-DIFFERENCE LAYER
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="FireDetection"
)
class AbsoluteDifference(
    tf.keras.layers.Layer
):

    def call(
        self,
        inputs
    ):

        a, b = inputs

        return tf.abs(
            a - b
        )


    def compute_output_shape(
        self,
        input_shape
    ):

        return input_shape[0]


    def get_config(
        self
    ):

        config = super().get_config()

        return config


# ============================================================
# READ ONE SIAMESE PATCH
# ============================================================

def read_siamese_patch(
    sample_row,
    input_config
):

    if input_config not in [
        "prepost_s2",
        "prepost_s2s1",
    ]:

        raise ValueError(
            "Siamese requires pre/post inputs."
        )


    transform, height, width = (
        get_patch_grid(
            sample_row
        )
    )


    fire_id = (
        sample_row[
            "fire_id"
        ]
    )


    # --------------------------------------------------------
    # SENTINEL-2
    # --------------------------------------------------------

    pre_s2, crs = (
        read_product_to_grid(
            fire_id,
            "S2_pre",
            S2_BANDS,
            transform,
            height,
            width,
        )
    )


    post_s2, _ = (
        read_product_to_grid(
            fire_id,
            "S2_post",
            S2_BANDS,
            transform,
            height,
            width,
        )
    )


    # Sentinel-2 is ALREADY reflectance.
    # Do NOT divide by 10,000 again.

    pre_s2 = (
        normalize_product(
            pre_s2,
            "S2_pre"
        )
    )


    post_s2 = (
        normalize_product(
            post_s2,
            "S2_post"
        )
    )


    pre_arrays = [
        pre_s2
    ]


    post_arrays = [
        post_s2
    ]


    # --------------------------------------------------------
    # OPTIONAL SENTINEL-1
    # --------------------------------------------------------

    if (
        input_config
        ==
        "prepost_s2s1"
    ):

        pre_s1, _ = (
            read_product_to_grid(
                fire_id,
                "S1_pre",
                S1_BANDS,
                transform,
                height,
                width,
            )
        )


        post_s1, _ = (
            read_product_to_grid(
                fire_id,
                "S1_post",
                S1_BANDS,
                transform,
                height,
                width,
            )
        )


        pre_s1 = (
            normalize_product(
                pre_s1,
                "S1_pre"
            )
        )


        post_s1 = (
            normalize_product(
                post_s1,
                "S1_post"
            )
        )


        pre_arrays.append(
            pre_s1
        )


        post_arrays.append(
            post_s1
        )


    # --------------------------------------------------------
    # STACK CHANNELS
    # --------------------------------------------------------

    pre = np.concatenate(
        pre_arrays,
        axis=0
    )


    post = np.concatenate(
        post_arrays,
        axis=0
    )


    # --------------------------------------------------------
    # VALIDITY MASK
    # --------------------------------------------------------

    valid = (
        np.isfinite(
            pre
        )
        .all(
            axis=0
        )
        &
        np.isfinite(
            post
        )
        .all(
            axis=0
        )
    )


    # --------------------------------------------------------
    # REPLACE INVALID VALUES AFTER VALID MASK CREATED
    # --------------------------------------------------------

    pre = np.nan_to_num(
        pre,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )


    post = np.nan_to_num(
        post,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )


    # --------------------------------------------------------
    # CHANNELS-FIRST -> CHANNELS-LAST
    # --------------------------------------------------------

    pre = np.moveaxis(
        pre,
        0,
        -1
    )


    post = np.moveaxis(
        post,
        0,
        -1
    )


    # --------------------------------------------------------
    # REFERENCE MASK
    # --------------------------------------------------------

    y, weight = (
        create_reference_patch(
            fire_id,
            crs,
            transform,
            height,
            width,
            valid,
        )
    )


    return (
        pre.astype(
            np.float32
        ),

        post.astype(
            np.float32
        ),

        y.astype(
            np.float32
        ),

        weight.astype(
            np.float32
        ),
    )


# ============================================================
# SIAMESE DATA GENERATOR
# ============================================================

class SiameseSequence(
    tf.keras.utils.Sequence
):

    def __init__(
        self,
        samples,
        input_config,
        batch_size=4,
        shuffle=True,
        augment=False,
    ):

        super().__init__()

        self.samples = (
            samples
            .reset_index(
                drop=True
            )
        )


        self.input_config = (
            input_config
        )


        self.batch_size = (
            batch_size
        )


        self.shuffle = (
            shuffle
        )


        self.augment = (
            augment
        )


        self.indices = np.arange(
            len(
                self.samples
            )
        )


        self.on_epoch_end()


    def __len__(
        self
    ):

        return math.ceil(
            len(
                self.samples
            )
            /
            self.batch_size
        )


    def on_epoch_end(
        self
    ):

        if self.shuffle:

            np.random.shuffle(
                self.indices
            )


    def __getitem__(
        self,
        batch_index
    ):

        indexes = (
            self.indices[
                batch_index
                *
                self.batch_size
                :
                (
                    batch_index
                    +
                    1
                )
                *
                self.batch_size
            ]
        )


        PRE = []
        POST = []
        Y = []
        W = []


        for idx in indexes:

            row = (
                self.samples
                .iloc[
                    idx
                ]
            )


            (
                pre,
                post,
                y,
                weight
            ) = (
                read_siamese_patch(
                    row,
                    self.input_config
                )
            )


            # ------------------------------------------------
            # HORIZONTAL FLIP AUGMENTATION
            # ------------------------------------------------

            if (
                self.augment
                and
                np.random.rand() < 0.5
            ):

                pre = np.flip(
                    pre,
                    axis=1
                )

                post = np.flip(
                    post,
                    axis=1
                )

                y = np.flip(
                    y,
                    axis=1
                )

                weight = np.flip(
                    weight,
                    axis=1
                )


            PRE.append(
                pre
            )


            POST.append(
                post
            )


            Y.append(
                y[
                    ...,
                    None
                ]
            )


            W.append(
                weight
            )


        # IMPORTANT:
        # Multiple Keras inputs are returned as a tuple.

        return (

            (
                np.asarray(
                    PRE,
                    dtype=np.float32
                ),

                np.asarray(
                    POST,
                    dtype=np.float32
                ),
            ),

            np.asarray(
                Y,
                dtype=np.float32
            ),

            np.asarray(
                W,
                dtype=np.float32
            ),
        )


# ============================================================
# SHARED CONVOLUTIONAL BLOCK
# ============================================================

# Only define this here if conv_block has not already been
# defined by the U-Net block.

if "conv_block" not in globals():

    def conv_block(
        x,
        filters
    ):

        x = (
            tf.keras.layers.Conv2D(
                filters,
                3,
                padding="same",
                activation="relu"
            )(
                x
            )
        )


        x = (
            tf.keras.layers.Conv2D(
                filters,
                3,
                padding="same",
                activation="relu"
            )(
                x
            )
        )


        return x


# ============================================================
# SHARED SIAMESE ENCODER
# ============================================================

def build_shared_encoder(
    input_shape
):

    inp = (
        tf.keras.layers.Input(
            shape=input_shape,
            name="encoder_input"
        )
    )


    # --------------------------------------------------------
    # LEVEL 1
    # --------------------------------------------------------

    c1 = conv_block(
        inp,
        32
    )


    p1 = (
        tf.keras.layers.MaxPooling2D(
            pool_size=2
        )(
            c1
        )
    )


    # --------------------------------------------------------
    # LEVEL 2
    # --------------------------------------------------------

    c2 = conv_block(
        p1,
        64
    )


    p2 = (
        tf.keras.layers.MaxPooling2D(
            pool_size=2
        )(
            c2
        )
    )


    # --------------------------------------------------------
    # LEVEL 3
    # --------------------------------------------------------

    c3 = conv_block(
        p2,
        128
    )


    p3 = (
        tf.keras.layers.MaxPooling2D(
            pool_size=2
        )(
            c3
        )
    )


    # --------------------------------------------------------
    # BOTTLENECK
    # --------------------------------------------------------

    b = conv_block(
        p3,
        256
    )


    return tf.keras.Model(

        inputs=
            inp,

        outputs=[
            c1,
            c2,
            c3,
            b,
        ],

        name=
            "shared_encoder"
    )


# ============================================================
# BUILD SIAMESE CHANGE-DETECTION MODEL
# ============================================================

def build_siamese(
    input_shape
):

    # --------------------------------------------------------
    # TWO INPUT BRANCHES
    # --------------------------------------------------------

    pre_input = (
        tf.keras.layers.Input(
            shape=input_shape,
            name="pre"
        )
    )


    post_input = (
        tf.keras.layers.Input(
            shape=input_shape,
            name="post"
        )
    )


    # --------------------------------------------------------
    # SHARED-WEIGHT ENCODER
    # --------------------------------------------------------

    encoder = (
        build_shared_encoder(
            input_shape
        )
    )


    pre_features = (
        encoder(
            pre_input
        )
    )


    post_features = (
        encoder(
            post_input
        )
    )


    # --------------------------------------------------------
    # FEATURE DIFFERENCES
    #
    # Uses serializable layer instead of Lambda.
    # --------------------------------------------------------

    differences = [

        AbsoluteDifference(
            name=
                f"absdiff_{i}"
        )(
            (
                pre_feature,
                post_feature
            )
        )

        for i, (
            pre_feature,
            post_feature
        )
        in enumerate(
            zip(
                pre_features,
                post_features
            )
        )
    ]


    (
        d1,
        d2,
        d3,
        db
    ) = (
        differences
    )


    # --------------------------------------------------------
    # DECODER LEVEL 3
    # --------------------------------------------------------

    u3 = (
        tf.keras.layers.Conv2DTranspose(
            128,
            2,
            strides=2,
            padding="same"
        )(
            db
        )
    )


    u3 = (
        tf.keras.layers.Concatenate(
            name="decoder_concat_3"
        )(
            (
                u3,
                d3
            )
        )
    )


    u3 = conv_block(
        u3,
        128
    )


    # --------------------------------------------------------
    # DECODER LEVEL 2
    # --------------------------------------------------------

    u2 = (
        tf.keras.layers.Conv2DTranspose(
            64,
            2,
            strides=2,
            padding="same"
        )(
            u3
        )
    )


    u2 = (
        tf.keras.layers.Concatenate(
            name="decoder_concat_2"
        )(
            (
                u2,
                d2
            )
        )
    )


    u2 = conv_block(
        u2,
        64
    )


    # --------------------------------------------------------
    # DECODER LEVEL 1
    # --------------------------------------------------------

    u1 = (
        tf.keras.layers.Conv2DTranspose(
            32,
            2,
            strides=2,
            padding="same"
        )(
            u2
        )
    )


    u1 = (
        tf.keras.layers.Concatenate(
            name="decoder_concat_1"
        )(
            (
                u1,
                d1
            )
        )
    )


    u1 = conv_block(
        u1,
        32
    )


    # --------------------------------------------------------
    # BINARY BURNED / UNBURNED OUTPUT
    # --------------------------------------------------------

    output = (
        tf.keras.layers.Conv2D(
            1,
            1,
            activation="sigmoid",
            name="burn_probability"
        )(
            u1
        )
    )


    return tf.keras.Model(

        inputs=(
            pre_input,
            post_input
        ),

        outputs=
            output,

        name=
            "Siamese"
    )


# ============================================================
# TRAIN ONE SIAMESE CONFIGURATION
# ============================================================

def train_siamese(
    input_config
):

    tf.keras.backend.clear_session()

    gc.collect()


    # --------------------------------------------------------
    # DETERMINE INPUT SHAPE
    # --------------------------------------------------------

    example = (
        train_samples
        .iloc[
            0
        ]
    )


    (
        pre,
        post,
        _,
        _
    ) = (
        read_siamese_patch(
            example,
            input_config
        )
    )


    input_shape = (
        pre.shape
    )


    del pre
    del post

    gc.collect()


    # --------------------------------------------------------
    # BUILD MODEL
    # --------------------------------------------------------

    model = (
        build_siamese(
            input_shape
        )
    )


    print(
        "\n"
        +
        "=" * 75
    )

    print(
        "SIAMESE CONFIGURATION"
    )

    print(
        "=" * 75
    )


    print(
        "\nInput:",
        input_config
    )


    print(
        "Input shape:",
        input_shape
    )


    print(
        "\nSiamese parameters:"
    )


    print(
        "Total:",
        f"{model.count_params():,}"
    )


    # --------------------------------------------------------
    # COMPILE
    # --------------------------------------------------------

    model.compile(

        optimizer=
            tf.keras.optimizers.Adam(
                learning_rate=
                    1e-3
            ),

        loss=
            "binary_crossentropy",

        metrics=[
            "accuracy"
        ],
    )


    # --------------------------------------------------------
    # GENERATORS
    # --------------------------------------------------------

    train_gen = (
        SiameseSequence(
            train_samples,
            input_config,
            batch_size=4,
            shuffle=True,
            augment=True,
        )
    )


    val_gen = (
        SiameseSequence(
            val_samples,
            input_config,
            batch_size=4,
            shuffle=False,
            augment=False,
        )
    )


    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    checkpoint = (
        MODEL_DIR
        /
        f"Siamese_{input_config}.keras"
    )


    # --------------------------------------------------------
    # CALLBACKS
    # --------------------------------------------------------

    callbacks = [

        tf.keras.callbacks.EarlyStopping(

            monitor=
                "val_loss",

            patience=
                8,

            restore_best_weights=
                True,

            verbose=
                1,
        ),


        tf.keras.callbacks.ModelCheckpoint(

            filepath=
                checkpoint,

            monitor=
                "val_loss",

            save_best_only=
                True,

            verbose=
                1,
        ),


        tf.keras.callbacks.ReduceLROnPlateau(

            monitor=
                "val_loss",

            factor=
                0.5,

            patience=
                3,

            min_lr=
                1e-6,

            verbose=
                1,
        ),
    ]


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    history, efficiency = (
        monitor_training(

            train_function=lambda:
                model.fit(

                    train_gen,

                    validation_data=
                        val_gen,

                    epochs=
                        50,

                    callbacks=
                        callbacks,

                    verbose=
                        1,
                ),

            method=
                "Siamese",

            input_config=
                input_config,

            model=
                model,

            n_samples=
                len(
                    train_samples
                ),

            notes=
                (
                    "Shared-weight Siamese "
                    "change-detection network"
                ),
        )
    )


    # --------------------------------------------------------
    # SAVE BEST RESTORED MODEL
    # --------------------------------------------------------

    model.save(
        checkpoint
    )


    # --------------------------------------------------------
    # SAVE HISTORY
    # --------------------------------------------------------

    history_path = (
        RESULT_DIR
        /
        f"Siamese_{input_config}_history.csv"
    )


    pd.DataFrame(
        history.history
    ).to_csv(
        history_path,
        index=False
    )


    print(
        "\nSaved model:"
    )

    print(
        checkpoint
    )


    print(
        "\nSaved history:"
    )

    print(
        history_path
    )


    return model


print(
    "\n"
    +
    "=" * 75
)

print(
    "SIAMESE MODEL DEFINITIONS READY"
)

print(
    "=" * 75
)

## Block D2 — Remove the old Lambda-based checkpoints

## Block D3 — Train the new Siamese models

In [ ]:
# ============================================================
# MODELING BLOCK D3
# SIAMESE TRAINING CONTROLLER
#
# - Reuse compatible checkpoints.
# - Back up legacy Lambda-based checkpoints automatically.
# - Train only missing/incompatible configurations.
# ============================================================

SIAMESE_RESULTS = {}

for config in [
    "prepost_s2",
    "prepost_s2s1",
]:

    checkpoint = MODEL_DIR / f"Siamese_{config}.keras"

    print("\n" + "=" * 75)
    print(f"SIAMESE: {config}")
    print("=" * 75)

    model = None

    if checkpoint.exists():
        print("Found saved Siamese checkpoint:")
        print(checkpoint)

        try:
            model = tf.keras.models.load_model(
                checkpoint,
                custom_objects={
                    "AbsoluteDifference": AbsoluteDifference,
                },
                compile=False,
            )
            print("Compatible checkpoint loaded successfully.")

        except Exception as exc:
            print("Existing checkpoint is incompatible with the current serializable architecture.")
            print(f"Reason: {type(exc).__name__}: {exc}")

            backup = checkpoint.with_name(
                checkpoint.stem + "_legacy_incompatible.keras"
            )

            if backup.exists():
                backup.unlink()

            checkpoint.replace(backup)
            print("Legacy checkpoint backed up to:")
            print(backup)
            model = None

    if model is None:
        print("Starting Siamese training...")
        model = train_siamese(config)

    # Compile after loading/training for a consistent evaluation interface.
    if not getattr(model, "optimizer", None):
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )

    SIAMESE_RESULTS[config] = model
    gc.collect()

print("\n" + "=" * 75)
print("SIAMESE TRAINING COMPLETE")
print("=" * 75)


## Block D4 — Verify that the new checkpoints actually reload

In [ ]:
# ============================================================
# MODELING BLOCK D4
# VERIFY SIAMESE CHECKPOINT RELOADING
# ============================================================


print(
    "\n"
    +
    "=" * 75
)

print(
    "VERIFYING SIAMESE CHECKPOINTS"
)

print(
    "=" * 75
)


for config in [

    "prepost_s2",
    "prepost_s2s1",

]:

    checkpoint = (
        MODEL_DIR
        /
        f"Siamese_{config}.keras"
    )


    if not checkpoint.exists():

        raise FileNotFoundError(
            f"Missing checkpoint: {checkpoint}"
        )


    print(
        f"\nLoading: {checkpoint}"
    )


    test_model = (
        tf.keras.models.load_model(

            checkpoint,

            custom_objects={
                "AbsoluteDifference":
                    AbsoluteDifference
            },

            compile=False,
        )
    )


    print(
        "Loaded successfully."
    )


    print(
        "Input shapes:",
        [
            tuple(
                inp.shape
            )
            for inp
            in test_model.inputs
        ]
    )


    print(
        "Output shape:",
        tuple(
            test_model.output.shape
        )
    )


    print(
        "Parameters:",
        f"{test_model.count_params():,}"
    )


    del test_model

    tf.keras.backend.clear_session()

    gc.collect()


print(
    "\n"
    +
    "=" * 75
)

print(
    "ALL SIAMESE CHECKPOINTS VERIFIED"
)

print(
    "=" * 75
)

## 5. Prithvi dataset using the same CSV patches

In [ ]:
# ============================================================
# MODELING BLOCK E
# PRITHVI DATASET
# ============================================================

import torch

from torch.utils.data import (
    Dataset,
    DataLoader
)

import torch.nn.functional as F


class PrithviFireDataset(
    Dataset
):

    def __init__(
        self,
        samples,
        temporal=False,
        output_size=224,
    ):

        self.samples = (
            samples.reset_index(
                drop=True
            )
        )

        self.temporal = (
            temporal
        )

        self.output_size = (
            output_size
        )


    def __len__(self):

        return len(
            self.samples
        )


    def __getitem__(
        self,
        idx
    ):

        row = (
            self.samples
            .iloc[idx]
        )

        transform, h, w = (
            get_patch_grid(
                row
            )
        )

        fire_id = (
            row[
                "fire_id"
            ]
        )

        # -----------------------------------------------
        # Post
        # -----------------------------------------------

        post, crs = (
            read_product_to_grid(
                fire_id,
                "S2_post",
                PRITHVI_BANDS,
                transform,
                h,
                w,
            )
        )

        post = (
            normalize_product(
                post,
                "S2_post"
            )
        )

        valid = (
            np.isfinite(
                post
            )
            .all(axis=0)
        )

        post = np.nan_to_num(
            post,
            nan=0.0
        )

        if self.temporal:

            pre, _ = (
                read_product_to_grid(
                    fire_id,
                    "S2_pre",
                    PRITHVI_BANDS,
                    transform,
                    h,
                    w,
                )
            )

            pre = (
                normalize_product(
                    pre,
                    "S2_pre"
                )
            )

            valid &= (
                np.isfinite(
                    pre
                )
                .all(axis=0)
            )

            pre = np.nan_to_num(
                pre,
                nan=0.0
            )

        y, weight = (
            create_reference_patch(
                fire_id,
                crs,
                transform,
                h,
                w,
                valid,
            )
        )

        # TerraTorch uses ignore_index = -1
        mask = y.astype(
            np.int64
        )

        mask[
            weight == 0
        ] = -1


        # -----------------------------------------------
        # PyTorch resize to 224
        # -----------------------------------------------

        post_tensor = torch.from_numpy(
            post
        ).float()

        post_tensor = F.interpolate(
            post_tensor.unsqueeze(0),
            size=(
                self.output_size,
                self.output_size
            ),
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)


        if self.temporal:

            pre_tensor = (
                torch.from_numpy(
                    pre
                )
                .float()
            )

            pre_tensor = F.interpolate(
                pre_tensor.unsqueeze(0),
                size=(
                    self.output_size,
                    self.output_size
                ),
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)

            # [C, T, H, W]
            image = torch.stack(
                [
                    pre_tensor,
                    post_tensor
                ],
                dim=1
            )

        else:

            # [C, 1, H, W]
            image = (
                post_tensor
                .unsqueeze(1)
            )


        mask_tensor = (
            torch.from_numpy(
                mask
            )
            .float()
            .unsqueeze(0)
            .unsqueeze(0)
        )

        mask_tensor = F.interpolate(
            mask_tensor,
            size=(
                self.output_size,
                self.output_size
            ),
            mode="nearest",
        )

        mask_tensor = (
            mask_tensor
            .squeeze()
            .long()
        )


        return {
            "image":
                image,

            "mask":
                mask_tensor,
        }

In [ ]:
def create_prithvi_loaders(
    temporal
):

    train_dataset = (
        PrithviFireDataset(
            train_samples,
            temporal=temporal,
            output_size=224,
        )
    )

    val_dataset = (
        PrithviFireDataset(
            val_samples,
            temporal=temporal,
            output_size=224,
        )
    )

    train_loader = DataLoader(
        train_dataset,

        batch_size=2,

        shuffle=True,

        num_workers=0,

        pin_memory=True,

        # Prevent a final one-sample batch if dataset size
        # is ever not divisible by 2.
        drop_last=True,
    )

    val_loader = DataLoader(
        val_dataset,

        batch_size=2,

        shuffle=False,

        num_workers=0,

        pin_memory=True,

        drop_last=False,
    )

    return (
        train_loader,
        val_loader
    )

## 6. Prithvi frozen-backbone and LoRA adaptation

Full fine-tuning is intentionally excluded to keep the comparison focused on resource-efficient adaptation.


In [ ]:
# ============================================================
# MODELING BLOCK F
# PRITHVI MODEL CREATION
# ============================================================

import lightning as L

from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
)

from terratorch.tasks import (
    SemanticSegmentationTask
)


def build_prithvi_task(
    adaptation,
    n_frames
):
    """
    adaptation:
        frozen
        lora
    """

    if adaptation not in {
        "frozen",
        "lora"
    }:

        raise ValueError(
            "Prithvi adaptation must be "
            "'frozen' or 'lora'. "
            "Full fine-tuning is intentionally "
            "excluded from this study."
        )


    model_args = {

        # ----------------------------------------------------
        # PRITHVI-EO-2.0 300M BACKBONE
        # ----------------------------------------------------

        "backbone":
            "prithvi_eo_v2_300",

        "backbone_pretrained":
            True,

        "backbone_bands": [
            "BLUE",
            "GREEN",
            "RED",
            "NIR_NARROW",
            "SWIR_1",
            "SWIR_2",
        ],

        "backbone_num_frames":
            n_frames,


        # ----------------------------------------------------
        # SEGMENTATION DECODER
        # ----------------------------------------------------

        "decoder":
            "UperNetDecoder",

        "decoder_channels":
            256,

        "num_classes":
            2,

        "rescale":
            True,

        "head_dropout":
            0.1,


        # ----------------------------------------------------
        # PRITHVI FEATURE NECKS
        # ----------------------------------------------------

        "necks": [

            {
                "name":
                    "SelectIndices",

                "indices": [
                    5,
                    11,
                    17,
                    23,
                ],
            },

            {
                "name":
                    "ReshapeTokensToImage",

                "effective_time_dim":
                    n_frames,
            },
        ],
    }


    # ========================================================
    # LoRA
    # ========================================================

    if adaptation == "lora":

        model_args[
            "peft_config"
        ] = {

            "method":
                "LORA",

            "replace_qkv":
                "qkv",

            "peft_config_kwargs": {

                "target_modules": [
                    "qkv.q_linear",
                    "qkv.v_linear",
                    "mlp.fc1",
                    "mlp.fc2",
                ],

                "lora_alpha":
                    16,

                "r":
                    16,

                "lora_dropout":
                    0.05,
            },
        }


    # ========================================================
    # FREEZE BACKBONE ONLY FOR FROZEN CONFIGURATION
    # ========================================================

    freeze_backbone = (
        adaptation
        ==
        "frozen"
    )


    task = (
        SemanticSegmentationTask(

            model_factory=
                "EncoderDecoderFactory",

            model_args=
                model_args,

            loss=
                "ce",

            ignore_index=
                -1,

            freeze_backbone=
                freeze_backbone,

            freeze_decoder=
                False,

            # ------------------------------------------------
            # IMPORTANT:
            # We pass DataLoaders directly to Trainer.fit()
            # rather than using a TerraTorch DataModule.
            # Disable automatic validation plotting because
            # plot_sample() expects trainer.datamodule.
            # ------------------------------------------------

            plot_on_val=
                False,

            optimizer=
                "AdamW",

            lr=
                5e-5,

            optimizer_hparams={
                "weight_decay":
                    0.05
            },
        )
    )


    return task


print(
    "Prithvi model creation block ready."
)

In [ ]:
def train_prithvi(
    adaptation,
    temporal
):

    # ========================================================
    # MEMORY CLEANUP BEFORE MODEL CREATION
    # ========================================================

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    # ========================================================
    # INPUT CONFIGURATION
    # ========================================================

    n_frames = (
        2
        if temporal
        else 1
    )

    input_name = (
        "prepost_s2"
        if temporal
        else "post_s2"
    )


    # ========================================================
    # DATA LOADERS
    # batch_size = 1 is defined inside create_prithvi_loaders()
    # ========================================================

    train_loader, val_loader = (
        create_prithvi_loaders(
            temporal
        )
    )


    # ========================================================
    # MODEL
    # ========================================================

    task = build_prithvi_task(
        adaptation,
        n_frames
    )


    # ========================================================
    # OUTPUT DIRECTORY
    # ========================================================

    output_dir = (
        MODEL_DIR /
        f"Prithvi_{adaptation}_{input_name}"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # ========================================================
    # CHECKPOINTING
    # ========================================================

    checkpoint_callback = (
        ModelCheckpoint(

            dirpath=
                output_dir,

            monitor=
                "val/loss",

            mode=
                "min",

            save_top_k=
                1,

            filename=
                "best",
        )
    )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    early_stop = (
        EarlyStopping(

            monitor=
                "val/loss",

            patience=
                8,

            mode=
                "min",
        )
    )


    # ========================================================
    # TRAINER
    # ========================================================

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA GPU is not available. "
            "Prithvi training should not be started on CPU."
        )


    print(
        "Training on GPU:",
        torch.cuda.get_device_name(0)
    )


    trainer = L.Trainer(

        accelerator=
            "gpu",

        devices=
            1,

        precision=
            "16-mixed",

        max_epochs=
            50,

        callbacks=[
            checkpoint_callback,
            early_stop,
        ],

        default_root_dir=
            output_dir,

        log_every_n_steps=
            5,
    )


    # ========================================================
    # TRAINING + EFFICIENCY MONITORING
    # ========================================================

    _, efficiency = (
        monitor_training(

            train_function=lambda:
                trainer.fit(
                    task,

                    train_dataloaders=
                        train_loader,

                    val_dataloaders=
                        val_loader,
                ),

            method=
                f"Prithvi-{adaptation}",

            input_config=
                input_name,

            model=
                task,

            n_samples=
                len(train_samples),
        )
    )


    # ========================================================
    # BEST CHECKPOINT
    # ========================================================

    print(
        "Best checkpoint:",
        checkpoint_callback.best_model_path
    )


    return (
        task,
        trainer,
        checkpoint_callback.best_model_path,
    )

In [ ]:
# Frozen — Post S2

task, trainer, ckpt = train_prithvi(
    adaptation="frozen",
    temporal=False
)

In [ ]:
# Frozen — Pre/Post S2
task, trainer, ckpt = train_prithvi(
    adaptation="frozen",
    temporal=True
)

print("Saved checkpoint:", ckpt)

del task, trainer, ckpt
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# LoRA — Post S2

task, trainer, ckpt = train_prithvi(
    adaptation="lora",
    temporal=False
)

print("Saved checkpoint:", ckpt)

del task, trainer, ckpt
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Train the Prithvi LoRA pre/post configuration without deleting prior runs.
# Checkpoint compatibility and resume policy are handled by train_prithvi().
task, trainer, ckpt = train_prithvi(adaptation="lora", temporal=True)
print("Saved checkpoint:", ckpt)
del task, trainer, ckpt
import gc
gc.collect()


# 7. Accuracy metrics shared by all models

In [ ]:
# ============================================================
# COMMON ACCURACY METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    jaccard_score,
)


def calculate_segmentation_metrics(
    y_true,
    y_pred,
):

    # --------------------------------------------------------
    # Convert to arrays
    # --------------------------------------------------------

    y_true = np.asarray(
        y_true
    )

    y_pred = np.asarray(
        y_pred
    )


    # --------------------------------------------------------
    # Ensure identical shape
    # --------------------------------------------------------

    if y_true.shape != y_pred.shape:

        raise ValueError(
            f"Shape mismatch: "
            f"y_true={y_true.shape}, "
            f"y_pred={y_pred.shape}"
        )


    # --------------------------------------------------------
    # Keep only valid binary reference pixels
    #
    # Valid classes:
    #   0 = unburned
    #   1 = burned
    #
    # This excludes:
    #   - ignore_index = -1
    #   - NoData
    #   - pixels outside the evaluation domain
    # --------------------------------------------------------

    valid_mask = np.isin(
        y_true,
        [0, 1]
    )

    y_true = y_true[
        valid_mask
    ]

    y_pred = y_pred[
        valid_mask
    ]


    # --------------------------------------------------------
    # Remove invalid prediction values
    # --------------------------------------------------------

    valid_pred = np.isin(
        y_pred,
        [0, 1]
    )

    y_true = y_true[
        valid_pred
    ]

    y_pred = y_pred[
        valid_pred
    ]


    # --------------------------------------------------------
    # Flatten and convert to integer classes
    # --------------------------------------------------------

    y_true = (
        y_true
        .astype(
            np.uint8
        )
        .ravel()
    )

    y_pred = (
        y_pred
        .astype(
            np.uint8
        )
        .ravel()
    )


    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if len(
        y_true
    ) == 0:

        raise ValueError(
            "No valid evaluation pixels remain "
            "after masking."
        )


    # --------------------------------------------------------
    # Overall pixel accuracy
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )


    # --------------------------------------------------------
    # Burned-class segmentation metrics
    #
    # Positive class:
    #   1 = burned
    #
    # Negative class:
    #   0 = unburned
    # --------------------------------------------------------

    precision = precision_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0,
    )

    recall = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0,
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0,
    )

    iou = jaccard_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0,
    )


    # --------------------------------------------------------
    # Burned-area statistics
    # --------------------------------------------------------

    observed_pixels = int(
        np.sum(
            y_true == 1
        )
    )

    predicted_pixels = int(
        np.sum(
            y_pred == 1
        )
    )


    if observed_pixels > 0:

        area_error_percent = (
            (
                predicted_pixels
                -
                observed_pixels
            )
            /
            observed_pixels
            *
            100.0
        )

    else:

        area_error_percent = np.nan


    # --------------------------------------------------------
    # Absolute burned-area error
    # --------------------------------------------------------

    if np.isfinite(
        area_error_percent
    ):

        absolute_area_error_percent = (
            abs(
                area_error_percent
            )
        )

    else:

        absolute_area_error_percent = (
            np.nan
        )


    # --------------------------------------------------------
    # Number of evaluated pixels
    # --------------------------------------------------------

    evaluated_pixels = int(
        len(
            y_true
        )
    )


    # --------------------------------------------------------
    # Return unified metrics
    # --------------------------------------------------------

    return {

        "accuracy":
            float(
                accuracy
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1
            ),

        "iou":
            float(
                iou
            ),

        "observed_burned_pixels":
            observed_pixels,

        "predicted_burned_pixels":
            predicted_pixels,

        "burned_area_error_percent":
            (
                float(
                    area_error_percent
                )
                if np.isfinite(
                    area_error_percent
                )
                else np.nan
            ),

        "absolute_area_error_percent":
            (
                float(
                    absolute_area_error_percent
                )
                if np.isfinite(
                    absolute_area_error_percent
                )
                else np.nan
            ),

        "evaluated_pixels":
            evaluated_pixels,
    }


# ============================================================
# SAVE ACCURACY RESULT
# ============================================================

def save_accuracy_result(
    method,
    input_config,
    fire_id,
    metrics,
):

    record = {

        "method":
            method,

        "input_config":
            input_config,

        "fire_id":
            str(
                fire_id
            ),

        **metrics,
    }


    new_df = pd.DataFrame(
        [record]
    )


    # --------------------------------------------------------
    # Replace existing record for the same
    # method + input configuration + fire
    # instead of creating duplicate rows.
    # --------------------------------------------------------

    if RESULTS_CSV.exists():

        old_df = pd.read_csv(
            RESULTS_CSV
        )


        required_keys = {
            "method",
            "input_config",
            "fire_id",
        }


        if required_keys.issubset(
            old_df.columns
        ):

            old_df[
                "fire_id"
            ] = (
                old_df[
                    "fire_id"
                ]
                .astype(
                    str
                )
            )


            duplicate_mask = (

                (
                    old_df[
                        "method"
                    ]
                    ==
                    method
                )

                &

                (
                    old_df[
                        "input_config"
                    ]
                    ==
                    input_config
                )

                &

                (
                    old_df[
                        "fire_id"
                    ]
                    ==
                    str(
                        fire_id
                    )
                )
            )


            old_df = (
                old_df[
                    ~duplicate_mask
                ]
                .copy()
            )


        final_df = pd.concat(
            [
                old_df,
                new_df,
            ],
            ignore_index=True,
            sort=False,
        )


    else:

        final_df = (
            new_df
        )


    # --------------------------------------------------------
    # Save unified accuracy table
    # --------------------------------------------------------

    final_df.to_csv(
        RESULTS_CSV,
        index=False,
    )

# 8. Unified held-out-fire evaluation and mapping

This section evaluates saved predictions consistently on the two held-out fires (`fire_01` and `fire_12`). Each completed prediction is immediately displayed beside the rasterized reference burned area. The same reference geometry, 20 m grid, threshold (0.5 for neural networks), and accuracy definitions are used across methods.


### Held-out test prediction domain: predefined common study extent

For `fire_01` and `fire_12`, prediction and evaluation are performed over the full predefined `collection_extents` footprint for each event. The current-fire polygon is the burned reference class (1), whereas all other valid pixels inside the study extent are unburned (0). This prevents the test domain from being restricted to the burned polygon plus a narrow candidate-unburned/buffer region and therefore allows false-positive predictions across the surrounding landscape to be penalized. Training and validation sampling remain unchanged.

**Important:** prediction rasters produced by the earlier buffer-limited evaluation are obsolete for the final manuscript. Model retraining is not required, but full-extent inference must be rerun for RF, U-Net, Siamese, and Prithvi before rebuilding the final tables and figures.


In [ ]:
# ============================================================
# UNIFIED TEST-EVALUATION UTILITIES
# ============================================================

from pathlib import Path
import math
import gc
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
from rasterio.windows import Window

# Define missing directories and variables to prevent NameError
PROJECT_DIR = PROJECT_DIR
RESULT_DIR = PROJECT_DIR / "model_results"

TEST_FIRE_IDS = ["fire_01", "fire_12"]
TEST_PIXEL_SIZE_M = 20.0
NN_TILE_SIZE = 64
NN_THRESHOLD = 0.5

PREDICTION_DIR = RESULT_DIR / "predictions"
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

UNIFIED_ACCURACY_CSV = RESULT_DIR / "accuracy_results_unified.csv"
PREDICTION_REGISTRY_CSV = RESULT_DIR / "prediction_registry.csv"


def upsert_csv_row(path, record, keys):
    new = pd.DataFrame([record])
    if Path(path).exists():
        old = pd.read_csv(path)
        if not old.empty:
            match = pd.Series(True, index=old.index)
            for key in keys:
                if key not in old.columns:
                    match &= False
                    continue
                value = record.get(key)
                if value is None:
                    match &= old[key].isna()
                else:
                    match &= old[key].astype(str).eq(str(value))
            old = old.loc[~match].copy()
        out = pd.concat([old, new], ignore_index=True, sort=False)
    else:
        out = new
    out.to_csv(path, index=False)
    return out


# ============================================================
# TEST-STUDY EXTENT GEOMETRIES
#
# IMPORTANT:
# Training/sampling still uses get_reference_geometries(),
# where candidate_unburned is used to construct samples.
#
# TEST evaluation is different:
#   - the predefined collection_extents polygon is the common
#     spatial footprint for prediction and evaluation;
#   - the current-fire polygon is class 1 (burned);
#   - all other valid pixels inside the study extent are class 0.
#
# This avoids artificially constraining test prediction to the
# burned polygon + sampled/buffered unburned region.
# ============================================================

_test_extent_cache = {}


def get_test_study_geometries(
    fire_id,
    target_crs,
):
    """
    Return:
        fire_geom
            Current-fire reference polygon.

        study_extent_geom
            Predefined common study extent from collection_extents.

    The full study extent is used for held-out test prediction
    and evaluation. Previous-fire scars are not used to crop the
    test extent; if a model predicts them as the current fire,
    those pixels correctly contribute false positives.
    """

    fire_id = normalize_fire_id(
        fire_id
    )

    key = (
        fire_id,
        str(target_crs),
    )

    if key in _test_extent_cache:
        return _test_extent_cache[key]

    fire_geom = get_event_geometry(
        fire_polygons,
        fire_id,
        target_crs,
    )

    study_extent_geom = get_event_geometry(
        collection_extents,
        fire_id,
        target_crs,
    )

    if study_extent_geom is None or study_extent_geom.is_empty:
        raise ValueError(
            f"{fire_id}: predefined collection extent is empty."
        )

    # Keep the reference fire constrained to the study extent.
    fire_geom = fire_geom.intersection(
        study_extent_geom
    )

    if fire_geom.is_empty:
        raise ValueError(
            f"{fire_id}: current-fire polygon does not intersect "
            "the predefined study extent."
        )

    _test_extent_cache[key] = (
        fire_geom,
        study_extent_geom,
    )

    return (
        fire_geom,
        study_extent_geom,
    )


def get_complete_test_grid(
    fire_id,
    pixel_size=TEST_PIXEL_SIZE_M,
):
    """
    Build one canonical 20 m grid covering the COMPLETE
    predefined study extent for a held-out fire.

    The bounds come from collection_extents, NOT from
    candidate_unburned / buffer geometries.
    """

    fire_id = normalize_fire_id(
        fire_id
    )

    s2_path = get_product_paths(
        fire_id,
        "S2_post",
    )[0]

    with rasterio.open(
        s2_path
    ) as src:
        crs = src.crs

    (
        fire_geom,
        study_extent_geom,
    ) = get_test_study_geometries(
        fire_id,
        crs,
    )

    xmin, ymin, xmax, ymax = (
        study_extent_geom.bounds
    )

    # Snap bounds to the common 20 m grid.
    xmin = math.floor(
        xmin / pixel_size
    ) * pixel_size

    ymin = math.floor(
        ymin / pixel_size
    ) * pixel_size

    xmax = math.ceil(
        xmax / pixel_size
    ) * pixel_size

    ymax = math.ceil(
        ymax / pixel_size
    ) * pixel_size

    width = int(
        round(
            (xmax - xmin)
            /
            pixel_size
        )
    )

    height = int(
        round(
            (ymax - ymin)
            /
            pixel_size
        )
    )

    if width <= 0 or height <= 0:
        raise ValueError(
            f"{fire_id}: invalid test-grid size "
            f"{height} x {width}."
        )

    transform = from_origin(
        xmin,
        ymax,
        pixel_size,
        pixel_size,
    )

    return {
        "crs":
            crs,

        "transform":
            transform,

        "height":
            height,

        "width":
            width,

        "fire_geom":
            fire_geom,

        # For compatibility with all existing prediction,
        # accuracy and figure code, eligible_geom now means
        # the complete predefined study extent.
        "eligible_geom":
            study_extent_geom,

        "study_extent_geom":
            study_extent_geom,

        "evaluation_domain":
            "predefined_common_study_extent",
    }


def read_config_to_grid(fire_id, input_config, transform, height, width):
    arrays = []
    valid = np.ones((height, width), dtype=bool)

    for product, bands in INPUT_CONFIGS[input_config]:
        data, _ = read_product_to_grid(
            fire_id, product, bands, transform, height, width
        )
        data = normalize_product(data, product)
        valid &= np.isfinite(data).all(axis=0)
        data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
        arrays.append(data)

    x = np.concatenate(arrays, axis=0)
    x = np.moveaxis(x, 0, -1).astype(np.float32)
    return x, valid


def read_siamese_to_grid(fire_id, input_config, transform, height, width):
    pre_s2, _ = read_product_to_grid(
        fire_id, "S2_pre", S2_BANDS, transform, height, width
    )
    post_s2, _ = read_product_to_grid(
        fire_id, "S2_post", S2_BANDS, transform, height, width
    )
    pre_s2 = normalize_product(pre_s2, "S2_pre")
    post_s2 = normalize_product(post_s2, "S2_post")

    pre_arrays, post_arrays = [pre_s2], [post_s2]
    valid = np.isfinite(pre_s2).all(axis=0) & np.isfinite(post_s2).all(axis=0)

    if input_config == "prepost_s2s1":
        pre_s1, _ = read_product_to_grid(
            fire_id, "S1_pre", S1_BANDS, transform, height, width
        )
        post_s1, _ = read_product_to_grid(
            fire_id, "S1_post", S1_BANDS, transform, height, width
        )
        pre_s1 = normalize_product(pre_s1, "S1_pre")
        post_s1 = normalize_product(post_s1, "S1_post")
        valid &= np.isfinite(pre_s1).all(axis=0) & np.isfinite(post_s1).all(axis=0)
        pre_arrays.append(pre_s1)
        post_arrays.append(post_s1)

    pre = np.moveaxis(
        np.concatenate(pre_arrays, axis=0), 0, -1
    ).astype(np.float32)
    post = np.moveaxis(
        np.concatenate(post_arrays, axis=0), 0, -1
    ).astype(np.float32)

    return (
        np.nan_to_num(pre, nan=0.0, posinf=0.0, neginf=0.0),
        np.nan_to_num(post, nan=0.0, posinf=0.0, neginf=0.0),
        valid,
    )


def reference_masks_for_grid(grid):
    """
    Build the held-out test reference on the complete predefined
    common study extent.

    Reference classes:
        1 = current burned-area polygon
        0 = all other pixels inside the study extent

    Pixels outside the predefined extent are not evaluated.
    """

    shape = (
        grid["height"],
        grid["width"],
    )

    actual = rasterize(
        [
            (
                grid["fire_geom"],
                1,
            )
        ],
        out_shape=shape,
        transform=grid["transform"],
        fill=0,
        dtype="uint8",
    )

    study_extent_geom = grid.get(
        "study_extent_geom",
        grid["eligible_geom"],
    )

    eligible = rasterize(
        [
            (
                study_extent_geom,
                1,
            )
        ],
        out_shape=shape,
        transform=grid["transform"],
        fill=0,
        dtype="uint8",
    ).astype(bool)

    # Ensure the current-fire reference is only evaluated
    # inside the predefined study extent.
    actual[
        ~eligible
    ] = 0

    return (
        actual,
        eligible,
    )


def save_prediction_geotiff(prediction, evaluated_mask, grid, path):
    arr = np.full(prediction.shape, 255, dtype=np.uint8)
    arr[evaluated_mask] = prediction[evaluated_mask].astype(np.uint8)
    profile = {
        "driver": "GTiff",
        "height": grid["height"],
        "width": grid["width"],
        "count": 1,
        "dtype": "uint8",
        "crs": grid["crs"],
        "transform": grid["transform"],
        "nodata": 255,
        "compress": "deflate",
    }
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr, 1)
    return path


def metrics_from_maps(actual, prediction, evaluated_mask, pixel_size=TEST_PIXEL_SIZE_M):
    y_true = actual[evaluated_mask].astype(np.uint8)
    y_pred = prediction[evaluated_mask].astype(np.uint8)

    m = calculate_segmentation_metrics(y_true, y_pred)
    pixel_area_ha = (pixel_size * pixel_size) / 10000.0
    observed_area_ha = float(y_true.sum() * pixel_area_ha)
    predicted_area_ha = float(y_pred.sum() * pixel_area_ha)
    area_error = (
        (predicted_area_ha - observed_area_ha) / observed_area_ha * 100.0
        if observed_area_ha > 0 else np.nan
    )
    return {
        "accuracy": m["accuracy"],
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "iou": m["iou"],
        "observed_burned_area_ha": observed_area_ha,
        "predicted_burned_area_ha": predicted_area_ha,
        "burned_area_error_percent": area_error,
        "absolute_area_error_percent": abs(area_error) if np.isfinite(area_error) else np.nan,
        "evaluated_pixels": int(evaluated_mask.sum()),
        "n_evaluated_pixels": int(evaluated_mask.sum()),
    }


def plot_prediction_vs_actual(prediction_path, fire_id, method, input_config, save=True):
    """Immediately compare one predicted burned-area map with the actual fire mask."""
    prediction_path = Path(prediction_path)
    with rasterio.open(prediction_path) as src:
        pred = src.read(1)
        transform = src.transform
        crs = src.crs
        h, w = src.height, src.width
        nodata = src.nodata

    (
        fire_geom,
        study_extent_geom,
    ) = get_test_study_geometries(
        fire_id,
        crs,
    )

    actual = rasterize(
        [(fire_geom, 1)],
        out_shape=(h, w),
        transform=transform,
        fill=0,
        dtype="uint8",
    )

    eligible = rasterize(
        [(study_extent_geom, 1)],
        out_shape=(h, w),
        transform=transform,
        fill=0,
        dtype="uint8",
    ).astype(bool)

    pred_show = pred.astype(float)
    if nodata is not None:
        pred_show[pred == nodata] = np.nan
    actual_show = actual.astype(float)
    actual_show[~eligible] = np.nan

    fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
    axes[0].imshow(actual_show, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Actual burned area")
    axes[1].imshow(pred_show, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Predicted burned area")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{fire_id} — {method} — {input_config}")

    if save:
        out = RESULT_DIR / "comparison_maps" / f"{method}_{input_config}_{fire_id}_actual_vs_prediction.png"
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=300, bbox_inches="tight")
        print("Saved comparison map:", out)

    plt.show()
    plt.close(fig)


def register_prediction(method, input_config, fire_id, prediction_path, metrics):
    record = {
        "method": method,
        "input_config": input_config,
        "fire_id": fire_id,
        "prediction_raster": str(prediction_path),
        **metrics,
    }
    upsert_csv_row(
        UNIFIED_ACCURACY_CSV,
        record,
        keys=["method", "input_config", "fire_id"],
    )
    upsert_csv_row(
        PREDICTION_REGISTRY_CSV,
        {
            "method": method,
            "input_config": input_config,
            "fire_id": fire_id,
            "prediction_raster": str(prediction_path),
        },
        keys=["method", "input_config", "fire_id"],
    )
    return record

# ============================================================
# TEST-EXTENT CHECK
# ============================================================

def report_test_study_extent(
    fire_ids=TEST_FIRE_IDS,
):
    """
    Report the canonical common study extent used for held-out
    inference and evaluation.
    """

    records = []

    for fire_id in fire_ids:

        grid = get_complete_test_grid(
            fire_id
        )

        actual, eligible = (
            reference_masks_for_grid(
                grid
            )
        )

        pixel_area_ha = (
            TEST_PIXEL_SIZE_M
            *
            TEST_PIXEL_SIZE_M
            /
            10000.0
        )

        extent_pixels = int(
            eligible.sum()
        )

        burned_pixels = int(
            (
                (actual == 1)
                &
                eligible
            ).sum()
        )

        records.append(
            {
                "fire_id":
                    fire_id,

                "grid_height":
                    grid["height"],

                "grid_width":
                    grid["width"],

                "study_extent_pixels":
                    extent_pixels,

                "study_extent_ha":
                    extent_pixels
                    *
                    pixel_area_ha,

                "reference_burned_pixels":
                    burned_pixels,

                "reference_burned_ha":
                    burned_pixels
                    *
                    pixel_area_ha,

                "burned_fraction_percent":
                    (
                        100.0
                        *
                        burned_pixels
                        /
                        extent_pixels
                    )
                    if extent_pixels > 0
                    else np.nan,

                "evaluation_domain":
                    grid[
                        "evaluation_domain"
                    ],
            }
        )

    table = pd.DataFrame(
        records
    )

    display(
        table
    )

    return table


print(
    "Unified evaluation utilities ready — "
    "held-out prediction domain = predefined common study extent."
)

## 8.1 U-Net full-fire prediction and immediate map comparison

Saved U-Net checkpoints are evaluated on `fire_01` and `fire_12`. The prediction GeoTIFF is followed immediately by a side-by-side actual-versus-predicted figure.


In [ ]:
# ============================================================
# U-NET FULL-FIRE TEST EVALUATION
# ============================================================


def predict_unet_complete_fire(
    model,
    fire_id,
    input_config,
    tile_size=NN_TILE_SIZE,
):

    # --------------------------------------------------------
    # Full-fire evaluation grid and reference masks
    # --------------------------------------------------------

    grid = get_complete_test_grid(
        fire_id
    )

    actual, eligible = (
        reference_masks_for_grid(
            grid
        )
    )


    # --------------------------------------------------------
    # Output arrays
    # --------------------------------------------------------

    prediction = np.zeros(
        (
            grid["height"],
            grid["width"],
        ),
        dtype=np.uint8,
    )

    valid_all = np.zeros_like(
        eligible,
        dtype=bool,
    )


    # --------------------------------------------------------
    # Tiled inference
    #
    # U-Net was trained with 64 x 64 patches.
    # Edge tiles smaller than 64 x 64 are padded.
    # --------------------------------------------------------

    for row in range(
        0,
        grid["height"],
        tile_size,
    ):

        for col in range(
            0,
            grid["width"],
            tile_size,
        ):

            h = min(
                tile_size,
                grid["height"] - row,
            )

            w = min(
                tile_size,
                grid["width"] - col,
            )


            tile_transform = (
                rasterio.windows.transform(
                    Window(
                        col,
                        row,
                        w,
                        h,
                    ),
                    grid["transform"],
                )
            )


            x, valid = (
                read_config_to_grid(
                    fire_id,
                    input_config,
                    tile_transform,
                    h,
                    w,
                )
            )


            # ------------------------------------------------
            # Pad edge tiles to training dimensions
            # ------------------------------------------------

            x_pad = np.zeros(
                (
                    tile_size,
                    tile_size,
                    x.shape[-1],
                ),
                dtype=np.float32,
            )

            x_pad[
                :h,
                :w,
                :
            ] = x


            # ------------------------------------------------
            # Predict burned probability
            # ------------------------------------------------

            prob = model.predict(
                x_pad[
                    None,
                    ...
                ],
                verbose=0,
            )[
                0,
                :h,
                :w,
                0,
            ]


            # ------------------------------------------------
            # Convert probability to binary prediction
            # ------------------------------------------------

            prediction[
                row:row + h,
                col:col + w,
            ] = (
                prob
                >=
                NN_THRESHOLD
            ).astype(
                np.uint8
            )


            valid_all[
                row:row + h,
                col:col + w,
            ] = valid


    # --------------------------------------------------------
    # Common full-study-extent evaluation domain
    #
    # Pixel must be:
    #   1. eligible for evaluation
    #   2. valid in the satellite input
    # --------------------------------------------------------

    evaluated = (
        eligible
        &
        valid_all
    )


    # --------------------------------------------------------
    # COMMON METRICS
    #
    # Same metric function used for all models.
    # --------------------------------------------------------

    if not np.any(
        evaluated
    ):

        raise ValueError(
            f"No valid evaluation pixels "
            f"for fire {fire_id}, "
            f"configuration {input_config}."
        )


    metrics = (
        calculate_segmentation_metrics(
            actual[
                evaluated
            ],
            prediction[
                evaluated
            ],
        )
    )


    # Keep this field for efficiency calculations
    # and compatibility with downstream code.

    metrics[
        "n_evaluated_pixels"
    ] = int(
        np.sum(
            evaluated
        )
    )


    return (
        prediction,
        evaluated,
        grid,
        metrics,
    )


# ============================================================
# EVALUATE ONE U-NET CONFIGURATION / FIRE
# ============================================================

def evaluate_unet_test_fire(
    input_config,
    fire_id,
):

    model_path = (
        MODEL_DIR
        /
        f"UNet_{input_config}.keras"
    )


    # --------------------------------------------------------
    # Check trained model
    # --------------------------------------------------------

    if not model_path.exists():

        print(
            f"Skipping missing U-Net checkpoint: "
            f"{model_path}"
        )

        return None


    # --------------------------------------------------------
    # Clear TensorFlow state
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    gc.collect()


    # --------------------------------------------------------
    # Load best trained model
    # --------------------------------------------------------

    model = (
        tf.keras.models.load_model(
            model_path
        )
    )


    # --------------------------------------------------------
    # Measure inference efficiency
    # --------------------------------------------------------

    monitor = ResourceMonitor(
        interval_seconds=0.5
    )

    monitor.start()


    try:

        (
            prediction,
            evaluated,
            grid,
            metrics,
        ) = (
            predict_unet_complete_fire(
                model,
                fire_id,
                input_config,
            )
        )


    finally:

        efficiency = (
            monitor.stop()
        )


    # --------------------------------------------------------
    # Normalize inference time
    # --------------------------------------------------------

    n_pixels = metrics[
        "n_evaluated_pixels"
    ]


    if n_pixels > 0:

        efficiency[
            "inference_sec_per_million_pixels"
        ] = (
            efficiency[
                "elapsed_sec"
            ]
            /
            (
                n_pixels
                /
                1_000_000
            )
        )


    # --------------------------------------------------------
    # Save prediction raster
    # --------------------------------------------------------

    pred_path = (
        PREDICTION_DIR
        /
        f"UNet_{input_config}_{fire_id}.tif"
    )


    save_prediction_geotiff(
        prediction,
        evaluated,
        grid,
        pred_path,
    )


    # --------------------------------------------------------
    # Save computational-efficiency record
    # --------------------------------------------------------

    save_efficiency_record(
        method="U-Net",
        input_config=input_config,
        stage="prediction",
        metrics=efficiency,
        model=model,
        fire_id=fire_id,
        n_pixels=n_pixels,
        notes=(
            "Complete held-out fire; "
            "64x64 tiled inference."
        ),
    )


    # --------------------------------------------------------
    # Register accuracy result
    # --------------------------------------------------------

    record = register_prediction(
        "U-Net",
        input_config,
        fire_id,
        pred_path,
        metrics,
    )


    # --------------------------------------------------------
    # Report
    # --------------------------------------------------------

    print(
        "\n"
        +
        "=" * 70
    )

    print(
        f"U-Net | {input_config} | Fire {fire_id}"
    )

    print(
        "=" * 70
    )

    print(
        pd.Series(
            record
        )
    )


    # --------------------------------------------------------
    # Plot prediction against reference
    # --------------------------------------------------------

    plot_prediction_vs_actual(
        pred_path,
        fire_id,
        "U-Net",
        input_config,
    )


    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del (
        model,
        prediction,
        evaluated,
        grid,
    )

    tf.keras.backend.clear_session()

    gc.collect()


    return record


# ============================================================
# RUN ALL U-NET TEST CONFIGURATIONS
# ============================================================

for config in [
    "post_s2",
    "prepost_s2",
    "prepost_s2s1",
]:

    for fire_id in TEST_FIRE_IDS:

        evaluate_unet_test_fire(
            config,
            fire_id,
        )

## 8.2 Siamese full-fire prediction and immediate map comparison

The two Siamese configurations use identical full-fire grids and reference masks.


In [ ]:
# ============================================================
# SIAMESE FULL-FIRE TEST EVALUATION
# ============================================================

import gc
import numpy as np
import pandas as pd
import rasterio
import tensorflow as tf

from rasterio.windows import Window


# ============================================================
# CUSTOM LAYER REQUIRED TO LOAD SIAMESE MODELS
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="FireDetection"
)
class AbsoluteDifference(
    tf.keras.layers.Layer
):

    def call(
        self,
        inputs
    ):

        a, b = inputs

        return tf.abs(
            a - b
        )


    def compute_output_shape(
        self,
        input_shape
    ):

        return input_shape[0]


    def get_config(
        self
    ):

        return super().get_config()


# ============================================================
# PREDICT COMPLETE TEST FIRE
# ============================================================

def predict_siamese_complete_fire(
    model,
    fire_id,
    input_config,
    tile_size=NN_TILE_SIZE,
):

    # --------------------------------------------------------
    # Full-fire grid + reference masks
    # --------------------------------------------------------

    grid = get_complete_test_grid(
        fire_id
    )

    actual, eligible = (
        reference_masks_for_grid(
            grid
        )
    )


    # --------------------------------------------------------
    # Output arrays
    # --------------------------------------------------------

    prediction = np.zeros(
        (
            grid["height"],
            grid["width"],
        ),
        dtype=np.uint8,
    )

    valid_all = np.zeros_like(
        eligible,
        dtype=bool,
    )


    # --------------------------------------------------------
    # Tiled Siamese inference
    # --------------------------------------------------------

    for row in range(
        0,
        grid["height"],
        tile_size,
    ):

        for col in range(
            0,
            grid["width"],
            tile_size,
        ):

            h = min(
                tile_size,
                grid["height"] - row,
            )

            w = min(
                tile_size,
                grid["width"] - col,
            )


            tile_transform = (
                rasterio.windows.transform(
                    Window(
                        col,
                        row,
                        w,
                        h,
                    ),
                    grid["transform"],
                )
            )


            # ------------------------------------------------
            # Read paired pre/post inputs
            # ------------------------------------------------

            pre, post, valid = (
                read_siamese_to_grid(
                    fire_id,
                    input_config,
                    tile_transform,
                    h,
                    w,
                )
            )


            # ------------------------------------------------
            # Pad edge tiles to training size
            # ------------------------------------------------

            pre_pad = np.zeros(
                (
                    tile_size,
                    tile_size,
                    pre.shape[-1],
                ),
                dtype=np.float32,
            )

            post_pad = np.zeros(
                (
                    tile_size,
                    tile_size,
                    post.shape[-1],
                ),
                dtype=np.float32,
            )


            pre_pad[
                :h,
                :w,
                :
            ] = pre

            post_pad[
                :h,
                :w,
                :
            ] = post


            # ------------------------------------------------
            # Predict burned probability
            # ------------------------------------------------

            prob = model.predict(
                (
                    pre_pad[
                        None,
                        ...
                    ],
                    post_pad[
                        None,
                        ...
                    ],
                ),
                verbose=0,
            )[
                0,
                :h,
                :w,
                0,
            ]


            # ------------------------------------------------
            # Threshold probability
            # ------------------------------------------------

            prediction[
                row:row + h,
                col:col + w,
            ] = (
                prob
                >=
                NN_THRESHOLD
            ).astype(
                np.uint8
            )


            valid_all[
                row:row + h,
                col:col + w,
            ] = valid


    # --------------------------------------------------------
    # Common full-study-extent evaluation domain
    # --------------------------------------------------------

    evaluated = (
        eligible
        &
        valid_all
    )


    if not np.any(
        evaluated
    ):

        raise ValueError(
            f"No valid evaluation pixels "
            f"for fire {fire_id}, "
            f"configuration {input_config}."
        )


    # --------------------------------------------------------
    # Common metrics for all models
    # --------------------------------------------------------

    metrics = (
        calculate_segmentation_metrics(
            actual[
                evaluated
            ],
            prediction[
                evaluated
            ],
        )
    )


    metrics[
        "n_evaluated_pixels"
    ] = int(
        np.sum(
            evaluated
        )
    )


    return (
        prediction,
        evaluated,
        grid,
        metrics,
    )


# ============================================================
# EVALUATE ONE SIAMESE CONFIGURATION / FIRE
# ============================================================

def evaluate_siamese_test_fire(
    input_config,
    fire_id,
):

    model_path = (
        MODEL_DIR
        /
        f"Siamese_{input_config}.keras"
    )


    # --------------------------------------------------------
    # Check checkpoint
    # --------------------------------------------------------

    if not model_path.exists():

        print(
            f"Skipping missing Siamese checkpoint: "
            f"{model_path}"
        )

        return None


    # --------------------------------------------------------
    # Clear TF state
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    gc.collect()


    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    model = (
        tf.keras.models.load_model(
            model_path,
            custom_objects={
                "AbsoluteDifference":
                    AbsoluteDifference
            },
            compile=False,
        )
    )


    # --------------------------------------------------------
    # Measure inference efficiency
    # --------------------------------------------------------

    monitor = ResourceMonitor(
        interval_seconds=0.5
    )

    monitor.start()


    try:

        (
            prediction,
            evaluated,
            grid,
            metrics,
        ) = (
            predict_siamese_complete_fire(
                model,
                fire_id,
                input_config,
            )
        )


    finally:

        efficiency = (
            monitor.stop()
        )


    # --------------------------------------------------------
    # Normalize inference time
    # --------------------------------------------------------

    n_pixels = metrics[
        "n_evaluated_pixels"
    ]


    if n_pixels > 0:

        efficiency[
            "inference_sec_per_million_pixels"
        ] = (
            efficiency[
                "elapsed_sec"
            ]
            /
            (
                n_pixels
                /
                1_000_000
            )
        )


    # --------------------------------------------------------
    # Save prediction raster
    # --------------------------------------------------------

    pred_path = (
        PREDICTION_DIR
        /
        f"Siamese_{input_config}_{fire_id}.tif"
    )


    save_prediction_geotiff(
        prediction,
        evaluated,
        grid,
        pred_path,
    )


    # --------------------------------------------------------
    # Save efficiency metrics
    # --------------------------------------------------------

    save_efficiency_record(
        method="Siamese",
        input_config=input_config,
        stage="prediction",
        metrics=efficiency,
        model=model,
        fire_id=fire_id,
        n_pixels=n_pixels,
        notes=(
            "Complete held-out fire; "
            "64x64 tiled paired pre/post inference."
        ),
    )


    # --------------------------------------------------------
    # Register accuracy result
    # --------------------------------------------------------

    record = (
        register_prediction(
            "Siamese",
            input_config,
            fire_id,
            pred_path,
            metrics,
        )
    )


    # --------------------------------------------------------
    # Report
    # --------------------------------------------------------

    print(
        "\n"
        +
        "=" * 70
    )

    print(
        f"Siamese | {input_config} | Fire {fire_id}"
    )

    print(
        "=" * 70
    )

    print(
        pd.Series(
            record
        )
    )


    # --------------------------------------------------------
    # Comparison map
    # --------------------------------------------------------

    plot_prediction_vs_actual(
        pred_path,
        fire_id,
        "Siamese",
        input_config,
    )


    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del (
        model,
        prediction,
        evaluated,
        grid,
    )

    tf.keras.backend.clear_session()

    gc.collect()


    return record


# ============================================================
# RUN ALL SIAMESE TEST CONFIGURATIONS
# ============================================================

for config in [
    "prepost_s2",
    "prepost_s2s1",
]:

    for fire_id in TEST_FIRE_IDS:

        evaluate_siamese_test_fire(
            config,
            fire_id,
        )

## 8.3 Random Forest prediction maps

The RF inference block above already creates full-fire GeoTIFFs. This cell imports those results into the unified registry and displays each prediction beside the actual burned area, avoiding a second RF inference run.


In [ ]:
# ============================================================
# RANDOM FOREST — FULL COMMON-STUDY-EXTENT TEST INFERENCE
#
# IMPORTANT
# ---------
# This is the ONLY RF test/inference block that should be run.
#
# It DOES NOT retrain the RF models.
# It loads the existing trained RF checkpoints and predicts over
# the complete predefined collection_extents footprint for:
#
#   fire_01
#   fire_12
#
# Reference definition:
#   current fire polygon = burned (1)
#   all other valid pixels inside collection extent = unburned (0)
#   outside collection extent = not evaluated
#
# candidate_unburned is NOT used here.
# Previous-fire scars are NOT removed from the test domain.
#
# ============================================================


from pathlib import Path
import gc
import joblib
import numpy as np
import pandas as pd


# ============================================================
# 1. RF TEST CONFIGURATION
# ============================================================

RF_TEST_FIRE_IDS = [
    "fire_01",
    "fire_12",
]

RF_CONFIGS = [
    "post_s2",
    "prepost_s2",
    "prepost_s2s1",
]


RF_PREDICTION_DIR = (
    RESULT_DIR /
    "RF_predictions"
)

RF_PREDICTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. REQUIRED FUNCTION CHECK
# ============================================================

required_rf_functions = [

    "get_complete_test_grid",

    "reference_masks_for_grid",

    "read_config_to_grid",

    "metrics_from_maps",

    "save_prediction_geotiff",

    "register_prediction",

    "plot_prediction_vs_actual",

    "monitor_prediction",

]


missing_rf_functions = [

    name

    for name in required_rf_functions

    if name not in globals()
]


if missing_rf_functions:

    raise RuntimeError(

        "Missing required functions:\n"
        +
        "\n".join(
            missing_rf_functions
        )
        +
        "\n\nRun the unified evaluation utility cells first."
    )


# ============================================================
# 3. PREDICT ONE RF CONFIGURATION / FIRE
# ============================================================

def predict_rf_common_extent(
    model,
    fire_id,
    input_config,
):
    """
    Predict RF over the complete predefined common study extent.

    Returns
    -------
    prediction : uint8 array
        0/1 prediction for the canonical grid.

    evaluated : bool array
        Pixels included in evaluation:
        study extent AND valid EO inputs.

    metrics : dict
        Accuracy and burned-area metrics.

    grid : dict
        Canonical test-grid definition.
    """

    fire_id = normalize_fire_id(
        fire_id
    )


    # --------------------------------------------------------
    # Canonical predefined study extent
    #
    # IMPORTANT:
    # get_complete_test_grid() uses collection_extents,
    # NOT candidate_unburned.
    # --------------------------------------------------------

    grid = get_complete_test_grid(
        fire_id
    )


    print(
        "\n"
        + "=" * 75
    )

    print(
        "RANDOM FOREST — FULL COMMON STUDY EXTENT"
    )

    print(
        "=" * 75
    )

    print(
        "Fire:",
        fire_id
    )

    print(
        "Input:",
        input_config
    )

    print(
        "Grid:",
        grid["height"],
        "x",
        grid["width"]
    )

    print(
        "Evaluation domain:",
        grid.get(
            "evaluation_domain",
            "unknown"
        )
    )


    # --------------------------------------------------------
    # Reference:
    #
    # fire polygon                 -> 1
    # rest of collection extent    -> 0
    # outside collection extent    -> ignored
    # --------------------------------------------------------

    actual, study_extent_mask = (
        reference_masks_for_grid(
            grid
        )
    )


    # --------------------------------------------------------
    # Read complete RF input configuration to canonical grid
    # --------------------------------------------------------

    X_grid, valid_inputs = (
        read_config_to_grid(

            fire_id=
                fire_id,

            input_config=
                input_config,

            transform=
                grid["transform"],

            height=
                grid["height"],

            width=
                grid["width"],
        )
    )


    # --------------------------------------------------------
    # FINAL TEST MASK
    #
    # Every valid EO pixel inside the predefined collection
    # extent is evaluated.
    #
    # No candidate-unburned buffer restriction.
    # --------------------------------------------------------

    evaluated = (
        study_extent_mask
        &
        valid_inputs
    )


    n_extent_pixels = int(
        study_extent_mask.sum()
    )

    n_evaluated_pixels = int(
        evaluated.sum()
    )


    print(
        "Study-extent pixels:",
        f"{n_extent_pixels:,}"
    )

    print(
        "Valid evaluated pixels:",
        f"{n_evaluated_pixels:,}"
    )


    if n_evaluated_pixels == 0:

        raise RuntimeError(
            f"{fire_id} / {input_config}: "
            "no valid pixels available inside the study extent."
        )


    # --------------------------------------------------------
    # Allocate output
    # --------------------------------------------------------

    prediction = np.zeros(
        (
            grid["height"],
            grid["width"],
        ),
        dtype=np.uint8,
    )


    # --------------------------------------------------------
    # Extract only evaluated pixels for RF prediction
    #
    # X_grid shape:
    #     rows x cols x features
    # --------------------------------------------------------

    X = (
        X_grid[
            evaluated
        ]
        .astype(
            np.float32,
            copy=False
        )
    )


    # --------------------------------------------------------
    # RF prediction
    # --------------------------------------------------------

    y_pred = (
        model.predict(
            X
        )
        .astype(
            np.uint8
        )
    )


    prediction[
        evaluated
    ] = y_pred


    # --------------------------------------------------------
    # Metrics over complete valid common study extent
    # --------------------------------------------------------

    metrics = metrics_from_maps(

        actual=
            actual,

        prediction=
            prediction,

        evaluated_mask=
            evaluated,

    )


    # --------------------------------------------------------
    # Cleanup large feature arrays
    # --------------------------------------------------------

    del X

    del y_pred

    del X_grid

    del valid_inputs

    gc.collect()


    return (
        prediction,
        evaluated,
        metrics,
        grid,
    )


# ============================================================
# 4. RUN ONE RF TEST
# ============================================================

def run_rf_common_extent_test(
    input_config,
    fire_id,
):


    # --------------------------------------------------------
    # Existing trained RF model
    # --------------------------------------------------------

    model_path = (
        MODEL_DIR
        /
        f"RF_{input_config}.joblib"
    )


    if not model_path.exists():

        raise FileNotFoundError(
            f"RF model not found:\n{model_path}"
        )


    print(
        "\nLoading RF model:"
    )

    print(
        model_path
    )


    model = joblib.load(
        model_path
    )


    # --------------------------------------------------------
    # Get extent size before monitoring so inference efficiency
    # can be normalized consistently.
    # --------------------------------------------------------

    grid_for_monitor = (
        get_complete_test_grid(
            fire_id
        )
    )


    _, extent_mask_for_monitor = (
        reference_masks_for_grid(
            grid_for_monitor
        )
    )


    n_extent_pixels = int(
        extent_mask_for_monitor.sum()
    )


    del grid_for_monitor

    del extent_mask_for_monitor

    gc.collect()


    # --------------------------------------------------------
    # Run prediction + efficiency monitoring
    # --------------------------------------------------------

    def _predict():

        return predict_rf_common_extent(

            model=
                model,

            fire_id=
                fire_id,

            input_config=
                input_config,
        )


    result, efficiency_record = (
        monitor_prediction(

            predict_function=
                _predict,

            method=
                "Random Forest",

            input_config=
                input_config,

            model=
                model,

            fire_id=
                fire_id,

            n_pixels=
                n_extent_pixels,

            notes=
                "RF inference over complete predefined "
                "collection_extents study footprint.",
        )
    )


    (
        prediction,
        evaluated,
        metrics,
        grid,
    ) = result


    # --------------------------------------------------------
    # Save new prediction raster
    #
    # Existing same-name restricted rasters are intentionally
    # overwritten because they are no longer valid for the
    # manuscript evaluation.
    # --------------------------------------------------------

    prediction_path = (

        RF_PREDICTION_DIR
        /
        f"RF_{input_config}_{fire_id}.tif"
    )


    prediction_path = save_prediction_geotiff(

        prediction=
            prediction,

        evaluated_mask=
            evaluated,

        grid=
            grid,

        path=
            prediction_path,
    )


    # --------------------------------------------------------
    # Register in unified outputs
    # --------------------------------------------------------

    record = register_prediction(

        "Random Forest",

        input_config,

        fire_id,

        prediction_path,

        metrics,
    )


    # --------------------------------------------------------
    # Plot actual vs predicted
    # --------------------------------------------------------

    plot_prediction_vs_actual(

        prediction_path,

        fire_id,

        "Random Forest",

        input_config,
    )


    # --------------------------------------------------------
    # Print metrics
    # --------------------------------------------------------

    print(
        "\n"
        + "-" * 75
    )

    print(
        "RF TEST RESULTS"
    )

    print(
        "-" * 75
    )

    print(
        f"Accuracy                 : "
        f"{metrics['accuracy']:.4f}"
    )

    print(
        f"Precision                : "
        f"{metrics['precision']:.4f}"
    )

    print(
        f"Recall                   : "
        f"{metrics['recall']:.4f}"
    )

    print(
        f"F1                       : "
        f"{metrics['f1']:.4f}"
    )

    print(
        f"IoU                      : "
        f"{metrics['iou']:.4f}"
    )

    print(
        f"Observed burned area     : "
        f"{metrics['observed_burned_area_ha']:.2f} ha"
    )

    print(
        f"Predicted burned area    : "
        f"{metrics['predicted_burned_area_ha']:.2f} ha"
    )

    print(
        f"Area error               : "
        f"{metrics['burned_area_error_percent']:.2f}%"
    )

    print(
        f"Absolute area error      : "
        f"{metrics['absolute_area_error_percent']:.2f}%"
    )

    print(
        f"Evaluated pixels         : "
        f"{metrics['n_evaluated_pixels']:,}"
    )

    print(
        "\nPrediction saved:"
    )

    print(
        prediction_path
    )


    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del prediction

    del evaluated

    del model

    gc.collect()


    return (
        record,
        efficiency_record,
    )


# ============================================================
# 5. RUN ALL RF TESTS
#
# 3 input configurations × 2 held-out fires = 6 predictions
# ============================================================

rf_common_extent_records = []


for input_config in RF_CONFIGS:

    for fire_id in RF_TEST_FIRE_IDS:

        record, efficiency_record = (
            run_rf_common_extent_test(

                input_config=
                    input_config,

                fire_id=
                    fire_id,
            )
        )


        rf_common_extent_records.append(
            record
        )


        gc.collect()


# ============================================================
# 6. DISPLAY FINAL RF RESULTS
# ============================================================

rf_common_extent_results = pd.DataFrame(
    rf_common_extent_records
)


display_columns = [

    "input_config",

    "fire_id",

    "accuracy",

    "precision",

    "recall",

    "f1",

    "iou",

    "observed_burned_area_ha",

    "predicted_burned_area_ha",

    "burned_area_error_percent",

    "absolute_area_error_percent",

    "n_evaluated_pixels",

    "prediction_raster",
]


display_columns = [

    column

    for column in display_columns

    if column in rf_common_extent_results.columns
]


print(
    "\n"
    + "=" * 75
)

print(
    "RANDOM FOREST — FINAL COMMON-EXTENT TEST RESULTS"
)

print(
    "=" * 75
)


display(
    rf_common_extent_results[
        display_columns
    ]
)


print(
    "\nRF common-study-extent inference complete."
)

## 8.4 Prithvi full-fire test inference and prediction registration

The four trained Prithvi configurations (frozen-backbone and LoRA; post-fire S2 and pre/post S2) are loaded from their saved checkpoints and evaluated on the two held-out fires (`fire_01` and `fire_12`). Full-fire inference uses the same canonical 20 m evaluation grids as the other models. Each Prithvi prediction is saved as a GeoTIFF, registered in the unified accuracy/prediction tables, and its inference efficiency is recorded without retraining the model.


In [ ]:
# ============================================================
# PRITHVI FULL-FIRE INFERENCE — REQUIRED RASTER HELPERS
#
# PURPOSE
# -------
# Makes the Prithvi test-inference section restart-safe.
#
# This block:
#   1. Rebuilds the EO raster catalog if needed
#   2. Defines normalize_fire_id() if needed
#   3. Defines get_product_paths()
#   4. Checks all other dependencies before inference starts
#
# Run this block before the Prithvi full-fire inference block.
# ============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import rasterio


# ============================================================
# PROJECT / EO PATH
# ============================================================

PROJECT_DIR = PROJECT_DIR

EO_DIR = (
    PROJECT_DIR
    /
    "eo_data"
)


if not EO_DIR.exists():

    raise FileNotFoundError(
        f"EO data directory not found:\n"
        f"{EO_DIR}"
    )


# ============================================================
# PRODUCT NAMES
# ============================================================

PRODUCTS = [
    "S2_pre",
    "S2_post",
    "S1_pre",
    "S1_post",
]


# ============================================================
# NORMALIZE FIRE ID
# ============================================================

if "normalize_fire_id" not in globals():

    def normalize_fire_id(
        value
    ):

        value = (
            str(
                value
            )
            .strip()
            .lower()
        )


        match = re.search(
            r"fire[_\- ]*0*(\d+)",
            value,
            flags=re.IGNORECASE,
        )


        if match is None:

            match = re.search(
                r"(\d+)",
                value,
            )


        if match is None:

            return value


        number = int(
            match.group(
                1
            )
        )


        return (
            f"fire_{number:02d}"
        )


# ============================================================
# REBUILD RASTER CATALOG IF IT DOES NOT EXIST
# ============================================================

if (
    "catalog" not in globals()
    or
    not isinstance(
        catalog,
        dict,
    )
    or
    len(
        catalog
    ) == 0
):

    print(
        "Raster catalog not found in memory."
    )

    print(
        "Rebuilding catalog from:"
    )

    print(
        EO_DIR
    )


    pattern = re.compile(

        r"^(fire_\d+)_"
        r"(S2_pre|S2_post|S1_pre|S1_post)"
        r"_20m(?:-\d+-\d+)?\.tiff?$",

        re.IGNORECASE,
    )


    catalog = {}


    for path in sorted(
        EO_DIR.iterdir()
    ):

        if (
            path.suffix.lower()
            not in {
                ".tif",
                ".tiff",
            }
        ):

            continue


        match = pattern.match(
            path.name
        )


        if match is None:

            continue


        fire_id = (
            normalize_fire_id(
                match.group(
                    1
                )
            )
        )


        matched_product = (
            match.group(
                2
            )
        )


        product = next(

            p

            for p in PRODUCTS

            if (
                p.lower()
                ==
                matched_product.lower()
            )
        )


        catalog.setdefault(
            fire_id,
            {}
        )


        catalog[
            fire_id
        ].setdefault(
            product,
            []
        )


        catalog[
            fire_id
        ][
            product
        ].append(
            path
        )


    if len(
        catalog
    ) == 0:

        raise FileNotFoundError(
            "No EO rasters matching the expected "
            "naming convention were found."
        )


else:

    print(
        "Existing raster catalog found."
    )


# ============================================================
# NORMALIZE EXISTING CATALOG IDS
# ============================================================

catalog = {

    normalize_fire_id(
        fire_id
    ):
        products

    for fire_id, products
    in catalog.items()
}


# ============================================================
# GET PRODUCT PATHS
# ============================================================

def get_product_paths(
    fire_id,
    product,
):

    """
    Return all raster files associated with one fire/product.

    Supports catalog entries stored as:
      - list / tuple of paths
      - one Path/string
      - dictionary containing:
        paths, path, files, file, filepath
    """

    fire_id = (
        normalize_fire_id(
            fire_id
        )
    )


    # --------------------------------------------------------
    # Check fire
    # --------------------------------------------------------

    if fire_id not in catalog:

        raise KeyError(
            f"{fire_id} not found in raster catalog."
        )


    # --------------------------------------------------------
    # Check product
    # --------------------------------------------------------

    if product not in catalog[
        fire_id
    ]:

        raise KeyError(
            f"{product} not found for "
            f"{fire_id}."
        )


    item = (
        catalog[
            fire_id
        ][
            product
        ]
    )


    # --------------------------------------------------------
    # Single file
    # --------------------------------------------------------

    if isinstance(
        item,
        (
            str,
            Path,
        )
    ):

        paths = [
            item
        ]


    # --------------------------------------------------------
    # Dictionary
    # --------------------------------------------------------

    elif isinstance(
        item,
        dict,
    ):

        paths = None


        for key in [

            "paths",
            "path",
            "files",
            "file",
            "filepath",

        ]:

            if key in item:

                paths = (
                    item[
                        key
                    ]
                )

                break


        if paths is None:

            raise ValueError(
                "Cannot determine raster paths for "
                f"{fire_id}, {product}."
            )


        if isinstance(
            paths,
            (
                str,
                Path,
            )
        ):

            paths = [
                paths
            ]


    # --------------------------------------------------------
    # Iterable of files
    # --------------------------------------------------------

    else:

        paths = list(
            item
        )


    paths = [

        Path(
            path
        )

        for path in paths
    ]


    # --------------------------------------------------------
    # Validate
    # --------------------------------------------------------

    if len(
        paths
    ) == 0:

        raise FileNotFoundError(
            "No raster files found for "
            f"{fire_id}, {product}."
        )


    missing = [

        path

        for path in paths

        if not path.exists()
    ]


    if missing:

        raise FileNotFoundError(
            f"{fire_id}, {product}: "
            f"{len(missing)} raster file(s) "
            "do not exist:\n"
            +
            "\n".join(
                str(
                    path
                )
                for path in missing
            )
        )


    return paths


# ============================================================
# QUICK CATALOG CHECK
# ============================================================

print(
    "\nRaster catalog ready."
)

print(
    "Number of fires:",
    len(
        catalog
    )
)


for fire_id in [
    "fire_01",
    "fire_12",
]:

    fire_id = (
        normalize_fire_id(
            fire_id
        )
    )


    print(
        "\n",
        fire_id,
        sep="",
    )


    if fire_id not in catalog:

        print(
            "  MISSING FROM CATALOG"
        )

        continue


    for product in [
        "S2_pre",
        "S2_post",
    ]:

        try:

            paths = (
                get_product_paths(
                    fire_id,
                    product,
                )
            )


            print(
                f"  {product}: "
                f"{len(paths)} raster(s)"
            )


        except Exception as error:

            print(
                f"  {product}: ERROR"
            )

            print(
                "   ",
                error,
            )


# ============================================================
# CHECK ALL PRITHVI FULL-FIRE DEPENDENCIES
# ============================================================

required_dependencies = [

    # Reference/evaluation
    "get_reference_geometries",
    "get_test_study_geometries",
    "get_complete_test_grid",
    "reference_masks_for_grid",
    "collection_extents",

    # EO reading
    "read_product_to_grid",
    "normalize_product",

    # Metrics/output
    "calculate_segmentation_metrics",
    "metrics_from_maps",
    "save_prediction_geotiff",
    "register_prediction",

    # Efficiency
    "monitor_prediction",

    # Prithvi
    "build_prithvi_task",
    "PRITHVI_BANDS",

    # Paths/settings
    "MODEL_DIR",
    "RESULT_DIR",
    "TEST_PIXEL_SIZE_M",
]


missing_dependencies = [

    name

    for name in required_dependencies

    if name not in globals()
]


if missing_dependencies:

    raise RuntimeError(

        "The following prerequisite definitions "
        "have not been run in this Colab runtime:\n\n"

        +
        "\n".join(
            f"  - {name}"
            for name
            in missing_dependencies
        )

        +

        "\n\nRun the corresponding setup/helper cells "
        "above this section, then rerun this block."
    )


print(
    "\n"
    +
    "=" * 75
)

print(
    "PRITHVI FULL-FIRE PREREQUISITE CHECK PASSED"
)

print(
    "=" * 75
)

In [ ]:
# ============================================================
# PRITHVI FULL-FIRE TEST INFERENCE + REGISTRATION
#
# Evaluates the four trained Prithvi configurations on:
#   fire_01
#   fire_12
#
# Configurations:
#   1. Frozen backbone — Post S2
#   2. Frozen backbone — Pre/Post S2
#   3. LoRA — Post S2
#   4. LoRA — Pre/Post S2
#
# Outputs:
#   - full-fire prediction GeoTIFFs
#   - inference-efficiency records
#   - per-fire accuracy records
#   - unified prediction registry entries
#
# IMPORTANT:
#   No retraining is performed.
# ============================================================

from pathlib import Path
import gc

import numpy as np
import rasterio

import torch
import torch.nn.functional as F

from rasterio.windows import Window


# ============================================================
# SETTINGS
# ============================================================

PRITHVI_TEST_FIRE_IDS = [
    "fire_01",
    "fire_12",
]


# Training patches were 64 x 64 at 20 m.
# They were resized to 224 x 224 before entering Prithvi.
PRITHVI_TILE_SIZE = 64

PRITHVI_MODEL_SIZE = 224


PRITHVI_PREDICTION_DIR = (
    RESULT_DIR
    /
    "Prithvi_predictions"
)


PRITHVI_PREDICTION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# PRITHVI EXPERIMENTS
# ============================================================

PRITHVI_EXPERIMENTS = [

    # --------------------------------------------------------
    # Frozen backbone — Post S2
    # --------------------------------------------------------

    {
        "adaptation":
            "frozen",

        "input_config":
            "post_s2",

        "temporal":
            False,

        "n_frames":
            1,

        "checkpoint_dir":
            (
                MODEL_DIR
                /
                "Prithvi_frozen_post_s2"
            ),

        "preferred_checkpoint":
            "best.ckpt",
    },


    # --------------------------------------------------------
    # Frozen backbone — Pre/Post S2
    # --------------------------------------------------------

    {
        "adaptation":
            "frozen",

        "input_config":
            "prepost_s2",

        "temporal":
            True,

        "n_frames":
            2,

        "checkpoint_dir":
            (
                MODEL_DIR
                /
                "Prithvi_frozen_prepost_s2"
            ),

        "preferred_checkpoint":
            "best-v2.ckpt",
    },


    # --------------------------------------------------------
    # LoRA — Post S2
    # --------------------------------------------------------

    {
        "adaptation":
            "lora",

        "input_config":
            "post_s2",

        "temporal":
            False,

        "n_frames":
            1,

        "checkpoint_dir":
            (
                MODEL_DIR
                /
                "Prithvi_lora_post_s2"
            ),

        "preferred_checkpoint":
            "best.ckpt",
    },


    # --------------------------------------------------------
    # LoRA — Pre/Post S2
    # --------------------------------------------------------

    {
        "adaptation":
            "lora",

        "input_config":
            "prepost_s2",

        "temporal":
            True,

        "n_frames":
            2,

        "checkpoint_dir":
            (
                MODEL_DIR
                /
                "Prithvi_lora_prepost_s2"
            ),

        "preferred_checkpoint":
            "best.ckpt",
    },
]


# ============================================================
# FIND SAVED CHECKPOINT
# ============================================================

def find_prithvi_checkpoint(
    checkpoint_dir,
    preferred_checkpoint=None,
):

    checkpoint_dir = Path(
        checkpoint_dir
    )


    # --------------------------------------------------------
    # Preferred checkpoint
    # --------------------------------------------------------

    if preferred_checkpoint is not None:

        preferred_path = (
            checkpoint_dir
            /
            preferred_checkpoint
        )


        if preferred_path.exists():

            return preferred_path


    # --------------------------------------------------------
    # Fallback: any best*.ckpt
    # --------------------------------------------------------

    candidates = sorted(
        checkpoint_dir.glob(
            "best*.ckpt"
        )
    )


    # --------------------------------------------------------
    # Fallback: any checkpoint
    # --------------------------------------------------------

    if not candidates:

        candidates = sorted(
            checkpoint_dir.glob(
                "*.ckpt"
            )
        )


    if not candidates:

        raise FileNotFoundError(
            "No Prithvi checkpoint found in:\n"
            f"{checkpoint_dir}"
        )


    return candidates[-1]


# ============================================================
# LOAD TRAINED PRITHVI TASK
# ============================================================

def load_prithvi_task_from_checkpoint(
    adaptation,
    n_frames,
    checkpoint_path,
    device,
):

    checkpoint_path = Path(
        checkpoint_path
    )


    # --------------------------------------------------------
    # Rebuild exact architecture used during training
    # --------------------------------------------------------

    task = build_prithvi_task(
        adaptation=adaptation,
        n_frames=n_frames,
    )


    # --------------------------------------------------------
    # Load checkpoint
    # --------------------------------------------------------

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )


    # Lightning checkpoints normally contain state_dict
    if (
        isinstance(
            checkpoint,
            dict,
        )
        and
        "state_dict" in checkpoint
    ):

        state_dict = checkpoint[
            "state_dict"
        ]


    else:

        state_dict = checkpoint


    # --------------------------------------------------------
    # Exact checkpoint compatibility
    # --------------------------------------------------------

    task.load_state_dict(
        state_dict,
        strict=True,
    )


    task = task.to(
        device
    )


    task.eval()


    return task


# ============================================================
# EXTRACT PRITHVI SEGMENTATION LOGITS
#
# TerraTorch returns:
#
# terratorch.models.model.ModelOutput
#
# For segmentation, the actual tensor is normally:
#
#     output.output
#
# ============================================================

def extract_prithvi_logits(
    output,
):

    # --------------------------------------------------------
    # Direct tensor
    # --------------------------------------------------------

    if torch.is_tensor(
        output
    ):

        return output


    # --------------------------------------------------------
    # TerraTorch ModelOutput
    # --------------------------------------------------------

    if hasattr(
        output,
        "output",
    ):

        model_output = (
            output.output
        )


        if torch.is_tensor(
            model_output
        ):

            return model_output


        # Some wrappers could return a nested structure
        if isinstance(
            model_output,
            dict,
        ):

            for key in [

                "out",
                "output",
                "logits",
                "prediction",
                "pred",

            ]:

                if (
                    key in model_output
                    and
                    torch.is_tensor(
                        model_output[
                            key
                        ]
                    )
                ):

                    return model_output[
                        key
                    ]


        if isinstance(
            model_output,
            (
                list,
                tuple,
            ),
        ):

            for value in model_output:

                if torch.is_tensor(
                    value
                ):

                    return value


    # --------------------------------------------------------
    # Optional logits attribute
    # --------------------------------------------------------

    if hasattr(
        output,
        "logits",
    ):

        logits = (
            output.logits
        )


        if torch.is_tensor(
            logits
        ):

            return logits


    # --------------------------------------------------------
    # Dictionary
    # --------------------------------------------------------

    if isinstance(
        output,
        dict,
    ):

        for key in [

            "output",
            "out",
            "logits",
            "prediction",
            "pred",

        ]:

            if (
                key in output
                and
                torch.is_tensor(
                    output[
                        key
                    ]
                )
            ):

                return output[
                    key
                ]


    # --------------------------------------------------------
    # Tuple/list
    # --------------------------------------------------------

    if isinstance(
        output,
        (
            list,
            tuple,
        ),
    ):

        for value in output:

            if torch.is_tensor(
                value
            ):

                return value


            if hasattr(
                value,
                "output",
            ):

                nested = (
                    value.output
                )


                if torch.is_tensor(
                    nested
                ):

                    return nested


    # --------------------------------------------------------
    # Diagnostic if unsupported
    # --------------------------------------------------------

    available_attributes = [

        name

        for name
        in dir(
            output
        )

        if not name.startswith(
            "_"
        )
    ]


    raise TypeError(

        "Could not extract segmentation logits.\n"
        f"Output type: {type(output)}\n"
        f"Available attributes: "
        f"{available_attributes}"
    )


# ============================================================
# PRITHVI FORWARD PASS
# ============================================================

def prithvi_forward(
    task,
    image,
):

    # --------------------------------------------------------
    # TerraTorch segmentation model
    # --------------------------------------------------------

    output = task.model(
        image
    )


    logits = (
        extract_prithvi_logits(
            output
        )
    )


    # --------------------------------------------------------
    # Safety check
    #
    # Expected:
    # [batch, classes, height, width]
    # --------------------------------------------------------

    if logits.ndim != 4:

        raise ValueError(
            "Unexpected Prithvi logits shape:\n"
            f"{tuple(logits.shape)}"
        )


    return logits


# ============================================================
# READ ONE PRITHVI TILE
# ============================================================

def read_prithvi_tile(
    fire_id,
    grid,
    row_off,
    col_off,
    tile_height,
    tile_width,
    temporal,
):

    # --------------------------------------------------------
    # Window on canonical fire grid
    # --------------------------------------------------------

    window = Window(

        col_off=col_off,

        row_off=row_off,

        width=tile_width,

        height=tile_height,
    )


    tile_transform = (
        rasterio.windows.transform(

            window,

            grid[
                "transform"
            ],
        )
    )


    # ========================================================
    # POST-FIRE S2
    # ========================================================

    post, _ = (
        read_product_to_grid(

            fire_id,

            "S2_post",

            PRITHVI_BANDS,

            tile_transform,

            tile_height,

            tile_width,
        )
    )


    post = normalize_product(
        post,
        "S2_post",
    )


    # --------------------------------------------------------
    # Valid pixels
    # --------------------------------------------------------

    valid = (

        np.isfinite(
            post
        )

        .all(
            axis=0
        )
    )


    post = np.nan_to_num(

        post,

        nan=0.0,

        posinf=0.0,

        neginf=0.0,
    )


    # ========================================================
    # PRE-FIRE S2
    # ========================================================

    if temporal:

        pre, _ = (
            read_product_to_grid(

                fire_id,

                "S2_pre",

                PRITHVI_BANDS,

                tile_transform,

                tile_height,

                tile_width,
            )
        )


        pre = normalize_product(
            pre,
            "S2_pre",
        )


        valid &= (

            np.isfinite(
                pre
            )

            .all(
                axis=0
            )
        )


        pre = np.nan_to_num(

            pre,

            nan=0.0,

            posinf=0.0,

            neginf=0.0,
        )


    # ========================================================
    # CONVERT TO TENSORS
    # ========================================================

    post_tensor = (
        torch.from_numpy(
            post
        )
        .float()
    )


    # --------------------------------------------------------
    # Match training preprocessing:
    #
    # 64 x 64 -> 224 x 224
    # --------------------------------------------------------

    post_tensor = (

        F.interpolate(

            post_tensor.unsqueeze(
                0
            ),

            size=(

                PRITHVI_MODEL_SIZE,

                PRITHVI_MODEL_SIZE,
            ),

            mode=
                "bilinear",

            align_corners=
                False,
        )

        .squeeze(
            0
        )
    )


    # ========================================================
    # TEMPORAL INPUT
    # ========================================================

    if temporal:

        pre_tensor = (
            torch.from_numpy(
                pre
            )
            .float()
        )


        pre_tensor = (

            F.interpolate(

                pre_tensor.unsqueeze(
                    0
                ),

                size=(

                    PRITHVI_MODEL_SIZE,

                    PRITHVI_MODEL_SIZE,
                ),

                mode=
                    "bilinear",

                align_corners=
                    False,
            )

            .squeeze(
                0
            )
        )


        # ----------------------------------------------------
        # Shape:
        # [C, T, H, W]
        #
        # T=2:
        # pre, post
        # ----------------------------------------------------

        image = torch.stack(

            [

                pre_tensor,

                post_tensor,
            ],

            dim=1,
        )


    # ========================================================
    # SINGLE-DATE INPUT
    # ========================================================

    else:

        # ----------------------------------------------------
        # Shape:
        # [C, 1, H, W]
        # ----------------------------------------------------

        image = (
            post_tensor
            .unsqueeze(
                1
            )
        )


    return (
        image,
        valid,
    )


# ============================================================
# FULL-FIRE PRITHVI INFERENCE
# ============================================================

def predict_prithvi_complete_fire(
    task,
    fire_id,
    temporal,
    tile_size=PRITHVI_TILE_SIZE,
):

    # --------------------------------------------------------
    # Canonical test grid
    # --------------------------------------------------------

    grid = (
        get_complete_test_grid(
            fire_id
        )
    )


    # --------------------------------------------------------
    # Reference + predefined common study extent
    # --------------------------------------------------------

    actual, eligible = (
        reference_masks_for_grid(
            grid
        )
    )


    # --------------------------------------------------------
    # Output arrays
    # --------------------------------------------------------

    prediction = np.zeros(

        (

            grid[
                "height"
            ],

            grid[
                "width"
            ],
        ),

        dtype=
            np.uint8,
    )


    valid_all = np.zeros(

        prediction.shape,

        dtype=
            bool,
    )


    device = next(
        task.parameters()
    ).device


    # ========================================================
    # TILE THROUGH COMPLETE FIRE GRID
    # ========================================================

    for row_off in range(

        0,

        grid[
            "height"
        ],

        tile_size,
    ):

        for col_off in range(

            0,

            grid[
                "width"
            ],

            tile_size,
        ):

            # ------------------------------------------------
            # Edge tile dimensions
            # ------------------------------------------------

            tile_height = min(

                tile_size,

                grid[
                    "height"
                ]
                -
                row_off,
            )


            tile_width = min(

                tile_size,

                grid[
                    "width"
                ]
                -
                col_off,
            )


            # =================================================
            # READ TILE
            # =================================================

            image, valid = (
                read_prithvi_tile(

                    fire_id=
                        fire_id,

                    grid=
                        grid,

                    row_off=
                        row_off,

                    col_off=
                        col_off,

                    tile_height=
                        tile_height,

                    tile_width=
                        tile_width,

                    temporal=
                        temporal,
                )
            )


            # ------------------------------------------------
            # Add batch dimension
            #
            # [1, C, T, 224, 224]
            # ------------------------------------------------

            image = (

                image

                .unsqueeze(
                    0
                )

                .to(
                    device,
                    non_blocking=True,
                )
            )


            # =================================================
            # FORWARD PASS
            # =================================================

            with torch.inference_mode():

                if device.type == "cuda":

                    with torch.autocast(

                        device_type=
                            "cuda",

                        dtype=
                            torch.float16,
                    ):

                        logits = (
                            prithvi_forward(

                                task,

                                image,
                            )
                        )


                else:

                    logits = (
                        prithvi_forward(

                            task,

                            image,
                        )
                    )


            # ------------------------------------------------
            # Logits expected:
            #
            # [1, 2, H, W]
            # ------------------------------------------------

            if logits.ndim != 4:

                raise ValueError(
                    "Unexpected Prithvi logits shape: "
                    f"{tuple(logits.shape)}"
                )


            if logits.shape[
                1
            ] != 2:

                raise ValueError(
                    "Expected two Prithvi output classes, "
                    f"but got shape "
                    f"{tuple(logits.shape)}"
                )


            # =================================================
            # RESIZE BACK TO ORIGINAL 20 m TILE SIZE
            # =================================================

            logits = F.interpolate(

                logits.float(),

                size=(

                    tile_height,

                    tile_width,
                ),

                mode=
                    "bilinear",

                align_corners=
                    False,
            )


            # =================================================
            # CLASSIFICATION
            # =================================================

            tile_prediction = (

                torch.argmax(

                    logits,

                    dim=1,
                )

                .squeeze(
                    0
                )

                .detach()

                .cpu()

                .numpy()

                .astype(
                    np.uint8
                )
            )


            # =================================================
            # WRITE TILE TO COMPLETE FIRE ARRAY
            # =================================================

            row_slice = slice(

                row_off,

                row_off
                +
                tile_height,
            )


            col_slice = slice(

                col_off,

                col_off
                +
                tile_width,
            )


            prediction[

                row_slice,

                col_slice,

            ] = (
                tile_prediction
            )


            valid_all[

                row_slice,

                col_slice,

            ] = (
                valid
            )


            # ------------------------------------------------
            # Release tile tensors
            # ------------------------------------------------

            del image
            del logits
            del tile_prediction


    # ========================================================
    # FINAL COMMON STUDY-EXTENT EVALUATION DOMAIN
    # ========================================================

    evaluated = (

        eligible

        &

        valid_all
    )


    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if not np.any(
        evaluated
    ):

        raise ValueError(
            f"No valid evaluated pixels for "
            f"{fire_id}."
        )


    return (

        prediction,

        evaluated,

        actual,

        grid,
    )


# ============================================================
# CUDA CHECK
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU is required for "
        "Prithvi full-fire inference."
    )


device = torch.device(
    "cuda"
)


# ============================================================
# RUN ALL FOUR PRITHVI CONFIGURATIONS
# ============================================================

for experiment in PRITHVI_EXPERIMENTS:

    adaptation = (
        experiment[
            "adaptation"
        ]
    )


    input_config = (
        experiment[
            "input_config"
        ]
    )


    temporal = (
        experiment[
            "temporal"
        ]
    )


    n_frames = (
        experiment[
            "n_frames"
        ]
    )


    # ========================================================
    # FIND CHECKPOINT
    # ========================================================

    checkpoint_path = (
        find_prithvi_checkpoint(

            checkpoint_dir=
                experiment[
                    "checkpoint_dir"
                ],

            preferred_checkpoint=
                experiment[
                    "preferred_checkpoint"
                ],
        )
    )


    method_name = (
        f"Prithvi-{adaptation}"
    )


    print(
        "\n"
        +
        "=" * 90
    )


    print(
        f"{method_name} | "
        f"{input_config}"
    )


    print(
        "Checkpoint:",
        checkpoint_path,
    )


    print(
        "=" * 90
    )


    # ========================================================
    # CLEAN GPU MEMORY
    # ========================================================

    gc.collect()

    torch.cuda.empty_cache()


    # ========================================================
    # LOAD SAVED TRAINED MODEL
    # ========================================================

    task = (
        load_prithvi_task_from_checkpoint(

            adaptation=
                adaptation,

            n_frames=
                n_frames,

            checkpoint_path=
                checkpoint_path,

            device=
                device,
        )
    )


    # ========================================================
    # TEST BOTH HELD-OUT FIRES
    # ========================================================

    for fire_id in PRITHVI_TEST_FIRE_IDS:

        print(
            "\nTesting:",
            fire_id,
        )


        # ----------------------------------------------------
        # Canonical fire grid
        # ----------------------------------------------------

        canonical_grid = (
            get_complete_test_grid(
                fire_id
            )
        )


        total_pixels = int(

            canonical_grid[
                "height"
            ]

            *

            canonical_grid[
                "width"
            ]
        )


        # ====================================================
        # FULL-FIRE INFERENCE + EFFICIENCY MONITORING
        # ====================================================

        (
            inference_output,
            prediction_efficiency,

        ) = monitor_prediction(

            predict_function=

                lambda
                    task=task,
                    fire_id=fire_id,
                    temporal=temporal:

                    predict_prithvi_complete_fire(

                        task=
                            task,

                        fire_id=
                            fire_id,

                        temporal=
                            temporal,

                        tile_size=
                            PRITHVI_TILE_SIZE,
                    ),


            method=
                method_name,


            input_config=
                input_config,


            model=
                task,


            fire_id=
                fire_id,


            n_pixels=
                total_pixels,


            notes=(

                "Full-fire held-out inference; "

                f"checkpoint="
                f"{checkpoint_path.name}"
            ),
        )


        # ====================================================
        # UNPACK OUTPUT
        # ====================================================

        (
            prediction,
            evaluated,
            actual,
            grid,

        ) = inference_output


        # ====================================================
        # ACCURACY METRICS
        # ====================================================

        metrics = (
            metrics_from_maps(

                actual=
                    actual,

                prediction=
                    prediction,

                evaluated_mask=
                    evaluated,

                pixel_size=
                    TEST_PIXEL_SIZE_M,
            )
        )


        # ====================================================
        # SAVE PREDICTION GEOTIFF
        # ====================================================

        prediction_path = (

            PRITHVI_PREDICTION_DIR

            /

            (
                f"Prithvi_"
                f"{adaptation}_"
                f"{input_config}_"
                f"{fire_id}.tif"
            )
        )


        save_prediction_geotiff(

            prediction=
                prediction,

            evaluated_mask=
                evaluated,

            grid=
                grid,

            path=
                prediction_path,
        )


        # ====================================================
        # REGISTER RESULT
        #
        # Saves to:
        #   accuracy_results_unified.csv
        #   prediction_registry.csv
        # ====================================================

        register_prediction(

            method=
                method_name,

            input_config=
                input_config,

            fire_id=
                fire_id,

            prediction_path=
                prediction_path,

            metrics=
                metrics,
        )


        # ====================================================
        # REPORT
        # ====================================================

        print(
            f"Accuracy  = "
            f"{metrics['accuracy']:.4f}"
        )


        print(
            f"Precision = "
            f"{metrics['precision']:.4f}"
        )


        print(
            f"Recall    = "
            f"{metrics['recall']:.4f}"
        )


        print(
            f"F1        = "
            f"{metrics['f1']:.4f}"
        )


        print(
            f"IoU       = "
            f"{metrics['iou']:.4f}"
        )


        print(
            f"Area err. = "
            f"{metrics['burned_area_error_percent']:.2f}%"
        )


        print(
            f"Evaluated pixels = "
            f"{metrics['n_evaluated_pixels']:,}"
        )


        print(
            "Saved prediction:"
        )


        print(
            prediction_path
        )


        # ----------------------------------------------------
        # Cleanup between fires
        # ----------------------------------------------------

        gc.collect()

        torch.cuda.empty_cache()


    # ========================================================
    # CLEAN MODEL BEFORE NEXT CONFIGURATION
    # ========================================================

    del task

    gc.collect()

    torch.cuda.empty_cache()


# ============================================================
# FINAL REPORT
# ============================================================

print(
    "\n"
    +
    "=" * 90
)


print(
    "PRITHVI FULL-FIRE EVALUATION COMPLETE"
)


print(
    "=" * 90
)


print(
    "\nPrediction directory:"
)


print(
    PRITHVI_PREDICTION_DIR
)


print(
    "\nUnified accuracy CSV:"
)


print(
    UNIFIED_ACCURACY_CSV
)


print(
    "\nPrediction registry:"
)


print(
    PREDICTION_REGISTRY_CSV
)


print(
    "\nEfficiency CSV:"
)


print(
    EFFICIENCY_CSV
)

# 9. Publication-ready accuracy and performance tables

The following cells create manuscript-oriented summaries. Accuracy is summarized across the two held-out fires. Efficiency is summarized separately for training and full-fire prediction so computational cost is not mixed with mapping accuracy.


In [ ]:
# ============================================================
# PUBLICATION-READY TABLES
# ============================================================

TABLE_DIR = RESULT_DIR / "publication_tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)


def pretty_method_config(method, config):
    method_map = {
        "Random Forest": "RF",
        "U-Net": "U-Net",
        "Siamese": "Siamese",
        "Prithvi-frozen": "Prithvi (frozen)",
        "Prithvi-lora": "Prithvi (LoRA)",
        "Prithvi — Frozen": "Prithvi (frozen)",
        "Prithvi — LoRA": "Prithvi (LoRA)",
    }
    config_map = {
        "post_s2": "Post-fire S2",
        "prepost_s2": "Pre + post S2",
        "prepost_s2s1": "Pre + post S2 + S1",
    }
    return method_map.get(method, method), config_map.get(config, config)


# ------------------------------
# ACCURACY TABLE
# ------------------------------
if UNIFIED_ACCURACY_CSV.exists():
    acc = pd.read_csv(UNIFIED_ACCURACY_CSV)

    accuracy_summary = (
        acc.groupby(["method", "input_config"], as_index=False)
        .agg(
            Accuracy=("accuracy", "mean"),
            Precision=("precision", "mean"),
            Recall=("recall", "mean"),
            F1=("f1", "mean"),
            IoU=("iou", "mean"),
            Area_error_pct=("burned_area_error_percent", "mean"),
            Abs_area_error_pct=("absolute_area_error_percent", "mean"),
            N_test_fires=("fire_id", "nunique"),
        )
    )

    labels = accuracy_summary.apply(
        lambda r: pretty_method_config(r["method"], r["input_config"]), axis=1
    )
    accuracy_summary.insert(0, "Model", [x[0] for x in labels])
    accuracy_summary.insert(1, "Satellite input", [x[1] for x in labels])
    accuracy_summary = accuracy_summary.drop(columns=["method", "input_config"])

    for c in ["Accuracy", "Precision", "Recall", "F1", "IoU"]:
        accuracy_summary[c] = accuracy_summary[c].round(3)
    for c in ["Area_error_pct", "Abs_area_error_pct"]:
        accuracy_summary[c] = accuracy_summary[c].round(2)

    accuracy_summary.to_csv(
        TABLE_DIR / "Table_accuracy_publication.csv", index=False
    )
    accuracy_summary.to_excel(
        TABLE_DIR / "Table_accuracy_publication.xlsx", index=False
    )

    print("PUBLICATION TABLE — ACCURACY")
    display(accuracy_summary)
else:
    print("No unified accuracy results yet.")


# ------------------------------
# EFFICIENCY / PERFORMANCE TABLE
# ------------------------------
if EFFICIENCY_CSV.exists():
    eff = pd.read_csv(EFFICIENCY_CSV)

    # Keep columns that actually exist in the current monitor output.
    candidate = [
        "method", "input_config", "stage", "elapsed_sec",
        "inference_sec_per_million_pixels", "peak_process_ram_gb",
        "mean_cpu_system_percent", "peak_cpu_system_percent",
        "mean_gpu_util_percent", "peak_gpu_memory_gb",
        "trainable_parameters", "total_parameters"
    ]
    cols = [c for c in candidate if c in eff.columns]
    eff2 = eff[cols].copy()

    numeric_cols = [c for c in cols if c not in {"method", "input_config", "stage"}]
    perf_summary = (
        eff2.groupby(["method", "input_config", "stage"], as_index=False)[numeric_cols]
        .mean(numeric_only=True)
    )

    labels = perf_summary.apply(
        lambda r: pretty_method_config(r["method"], r["input_config"]), axis=1
    )
    perf_summary.insert(0, "Model", [x[0] for x in labels])
    perf_summary.insert(1, "Satellite input", [x[1] for x in labels])
    perf_summary = perf_summary.drop(columns=["method", "input_config"])

    perf_summary.to_csv(
        TABLE_DIR / "Table_efficiency_publication.csv", index=False
    )
    perf_summary.to_excel(
        TABLE_DIR / "Table_efficiency_publication.xlsx", index=False
    )

    print("PUBLICATION TABLE — COMPUTATIONAL PERFORMANCE")
    display(perf_summary)
else:
    print("No efficiency results yet.")

print("Publication tables directory:", TABLE_DIR)


# 10. Final prediction figures for the two held-out fires

One multi-panel figure is created for each test site. The first panel is the actual burned area; subsequent panels are every registered model/input prediction available for that fire. This directly supports a manuscript figure comparing spatial agreement among methods.


In [ ]:
# ============================================================
# FINAL MULTI-MODEL FIGURES — ONE FIGURE PER TEST FIRE
# ============================================================

FINAL_FIGURE_DIR = RESULT_DIR / "publication_figures"
FINAL_FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def make_all_predictions_figure(fire_id):
    if not PREDICTION_REGISTRY_CSV.exists():
        print("Prediction registry does not exist yet.")
        return None

    registry = pd.read_csv(PREDICTION_REGISTRY_CSV)
    subset = registry[registry["fire_id"].astype(str) == str(fire_id)].copy()
    subset = subset[subset["prediction_raster"].map(lambda p: Path(str(p)).exists())]

    if subset.empty:
        print(f"No registered predictions available for {fire_id}.")
        return None

    # Canonical predefined common study extent for this fire.
    grid = get_complete_test_grid(
        fire_id
    )

    transform = grid["transform"]
    crs = grid["crs"]
    h = grid["height"]
    w = grid["width"]

    actual, eligible = reference_masks_for_grid(
        grid
    )

    actual = actual.astype(
        float
    )

    actual[
        ~eligible
    ] = np.nan

    panels = [("Actual burned area", actual)]

    for _, row in subset.sort_values(["method", "input_config"]).iterrows():
        p = Path(row["prediction_raster"])
        with rasterio.open(p) as src:
            arr = src.read(1).astype(float)
            nodata = src.nodata
            if (src.height, src.width) != (h, w) or src.transform != transform or src.crs != crs:
                # Reproject prediction to first registered grid for display only.
                dst = np.full((h, w), np.nan, dtype=np.float32)
                from rasterio.warp import reproject, Resampling
                source = arr.astype(np.float32)
                if nodata is not None:
                    source[arr == nodata] = np.nan
                reproject(
                    source=source,
                    destination=dst,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=crs,
                    src_nodata=np.nan,
                    dst_nodata=np.nan,
                    resampling=Resampling.nearest,
                )
                arr = dst
            elif nodata is not None:
                arr[arr == nodata] = np.nan

        m, c = pretty_method_config(row["method"], row["input_config"])
        panels.append((f"{m}\n{c}", arr))

    n = len(panels)
    ncols = min(4, n)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(4.0 * ncols, 4.1 * nrows), constrained_layout=True
    )
    axes = np.atleast_1d(axes).ravel()

    for ax, (title, arr) in zip(axes, panels):
        ax.imshow(arr, cmap="gray", vmin=0, vmax=1)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    for ax in axes[len(panels):]:
        ax.axis("off")

    fig.suptitle(f"Held-out fire {fire_id}: actual and predicted burned area", fontsize=14)
    out_png = FINAL_FIGURE_DIR / f"Figure_all_predictions_{fire_id}.png"
    out_pdf = FINAL_FIGURE_DIR / f"Figure_all_predictions_{fire_id}.pdf"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("Saved:", out_png)
    print("Saved:", out_pdf)
    return out_png


SITE_1_FIGURE = make_all_predictions_figure("fire_01")
SITE_2_FIGURE = make_all_predictions_figure("fire_12")


In [ ]:
# ============================================================
# STANDARDIZE METHOD NAMES
# ============================================================

def nice_method_name(x):

    x = str(x).strip()
    xl = x.lower()

    if xl in [
        "rf",
        "random forest",
    ]:
        return "RF"

    if xl in [
        "u-net",
        "unet",
    ]:
        return "U-Net"

    if xl == "siamese":
        return "Siamese"

    if xl in [
        "prithvi (frozen)",
        "prithvi-frozen",
        "prithvi_frozen",
    ]:
        return "Prithvi — Frozen"

    if xl in [
        "prithvi (lora)",
        "prithvi-lora",
        "prithvi_lora",
    ]:
        return "Prithvi — LoRA"

    return x


# ============================================================
# STANDARDIZE SATELLITE INPUT NAMES
# ============================================================

def nice_input_name(x):

    x = str(x).strip()

    mapping = {

        # ----------------------------------------------------
        # Internal coding names
        # ----------------------------------------------------

        "post_s2":
            "Post S2",

        "prepost_s2":
            "Pre/Post S2",

        "prepost_s2s1":
            "Pre/Post S2 + S1",


        # ----------------------------------------------------
        # Efficiency-table names
        # ----------------------------------------------------

        "Post-fire S2":
            "Post S2",

        "Pre + post S2":
            "Pre/Post S2",

        "Pre + post S2 + S1":
            "Pre/Post S2 + S1",


        # ----------------------------------------------------
        # Already standardized names
        # ----------------------------------------------------

        "Post S2":
            "Post S2",

        "Pre/Post S2":
            "Pre/Post S2",

        "Pre/Post S2 + S1":
            "Pre/Post S2 + S1",
    }

    return mapping.get(
        x,
        x,
    )

In [ ]:
# ============================================================
# REBUILD TABLE 2
# ACCURACY vs OPERATIONAL EFFICIENCY
#
# Uses exact label mappings from the existing files
# ============================================================

# ------------------------------------------------------------
# Reload efficiency source cleanly
# ------------------------------------------------------------

import pandas as pd
import numpy as np

efficiency = pd.read_csv(
    EFFICIENCY_CSV
)

# ============================================================
# EXACT METHOD MAPPING
# ============================================================

EFF_METHOD_MAP = {
    "RF": "RF",
    "Random Forest": "RF",
    "U-Net": "U-Net",
    "Siamese": "Siamese",
    "Prithvi (frozen)": "Prithvi — Frozen",
    "Prithvi (LoRA)": "Prithvi — LoRA",
    "Prithvi-frozen": "Prithvi — Frozen",
    "Prithvi-lora": "Prithvi — LoRA",
    "Prithvi — Frozen": "Prithvi — Frozen",
    "Prithvi — LoRA": "Prithvi — LoRA"
}

ACC_METHOD_MAP = {
    "RF": "RF",
    "U-Net": "U-Net",
    "Siamese": "Siamese",
    "Prithvi (frozen)": "Prithvi — Frozen",
    "Prithvi (LoRA)": "Prithvi — LoRA",
    "Prithvi-frozen": "Prithvi — Frozen",
    "Prithvi-lora": "Prithvi — LoRA",
    "Prithvi — Frozen": "Prithvi — Frozen",
    "Prithvi — LoRA": "Prithvi — LoRA"
}

# ============================================================
# EXACT INPUT MAPPING
# ============================================================

EFF_INPUT_MAP = {
    "post_s2": "Post S2",
    "prepost_s2": "Pre/Post S2",
    "prepost_s2s1": "Pre/Post S2 + S1",
    "Post-fire S2": "Post S2",
    "Pre + post S2": "Pre/Post S2",
    "Pre + post S2 + S1": "Pre/Post S2 + S1",
}

ACC_INPUT_MAP = {
    "post_s2": "Post S2",
    "prepost_s2": "Pre/Post S2",
    "prepost_s2s1": "Pre/Post S2 + S1",
    "Post-fire S2": "Post S2",
    "Pre + post S2": "Pre/Post S2",
    "Pre + post S2 + S1": "Pre/Post S2 + S1",
}

# ============================================================
# STANDARDIZE ACCURACY KEYS
# ============================================================

accuracy_summary[
    "_method_key"
] = (
    accuracy_summary[
        "Model"
    ]
    .astype(str)
    .str.strip()
    .map(
        ACC_METHOD_MAP
    )
)

accuracy_summary[
    "_input_key"
] = (
    accuracy_summary[
        "Satellite input"
    ]
    .astype(str)
    .str.strip()
    .map(
        ACC_INPUT_MAP
    )
)

# ============================================================
# STANDARDIZE EFFICIENCY KEYS
# ============================================================

efficiency[
    "_method_key"
] = (
    efficiency[
        "method"
    ]
    .astype(str)
    .str.strip()
    .map(
        EFF_METHOD_MAP
    )
)

efficiency[
    "_input_key"
] = (
    efficiency[
        "input_config"
    ]
    .astype(str)
    .str.strip()
    .map(
        EFF_INPUT_MAP
    )
)

# ============================================================
# CHECK THAT KEYS NOW MATCH
# ============================================================

print(
    "\nAccuracy keys:"
)

print(
    accuracy_summary[
        [
            "_method_key",
            "_input_key",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "_method_key",
            "_input_key",
        ]
    )
    .to_string(
        index=False
    )
)

print(
    "\nEfficiency keys:"
)

print(
    efficiency[
        [
            "_method_key",
            "_input_key",
            "stage",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "_method_key",
            "_input_key",
            "stage",
        ]
    )
    .to_string(
        index=False
    )
)

# ============================================================
# TRAINING EFFICIENCY
# ============================================================

training_eff = (
    efficiency[
        efficiency[
            "stage"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            "training"
        )
    ]
    .copy()
)

training_summary = (
    training_eff
    .groupby(
        [
            "_method_key",
            "_input_key",
        ],
        as_index=False,
    )
    .agg(
        training_time_sec=(
            "elapsed_sec",
            "mean",
        ),
        peak_process_ram_gb=(
            "peak_process_ram_gb",
            "max",
        ),
        mean_cpu_system_percent=(
            "mean_cpu_system_percent",
            "mean",
        ),
        peak_cpu_system_percent=(
            "peak_cpu_system_percent",
            "max",
        ),
        mean_gpu_util_percent=(
            "mean_gpu_util_percent",
            "mean",
        ),
        peak_gpu_memory_gb=(
            "peak_gpu_memory_gb",
            "max",
        ),
        trainable_parameters=(
            "trainable_parameters",
            "max",
        ),
        total_parameters=(
            "total_parameters",
            "max",
        ),
    )
)

# ============================================================
# INFERENCE EFFICIENCY
# ============================================================

prediction_eff = (
    efficiency[
        efficiency[
            "stage"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            "prediction"
        )
    ]
    .copy()
)

prediction_summary = (
    prediction_eff
    .groupby(
        [
            "_method_key",
            "_input_key",
        ],
        as_index=False,
    )
    .agg(
        inference_time_sec=(
            "elapsed_sec",
            "mean",
        ),
        inference_sec_per_million_pixels=(
            "inference_sec_per_million_pixels",
            "mean",
        ),
        inference_peak_ram_gb=(
            "peak_process_ram_gb",
            "max",
        ),
        inference_mean_cpu_percent=(
            "mean_cpu_system_percent",
            "mean",
        ),
        inference_peak_cpu_percent=(
            "peak_cpu_system_percent",
            "max",
        ),
        inference_mean_gpu_util_percent=(
            "mean_gpu_util_percent",
            "mean",
        ),
        inference_peak_gpu_memory_gb=(
            "peak_gpu_memory_gb",
            "max",
        ),
    )
)

# ============================================================
# ACCURACY DATA FOR TABLE 2
# ============================================================

accuracy_eff = pd.DataFrame({
    "_method_key":
        accuracy_summary[
            "_method_key"
        ],
    "_input_key":
        accuracy_summary[
            "_input_key"
        ],
    "F1":
        accuracy_summary[
            "F1"
        ],
    "IoU":
        accuracy_summary[
            "IoU"
        ],
    "|Area error| (%)":
        accuracy_summary[
            "Abs_area_error_pct"
        ],
})

# ============================================================
# MERGE
# ============================================================

combined = (
    accuracy_eff
    .merge(
        training_summary,
        on=[
            "_method_key",
            "_input_key",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        prediction_summary,
        on=[
            "_method_key",
            "_input_key",
        ],
        how="left",
        validate="one_to_one",
    )
)

# ============================================================
# ARTICLE TABLE 2
# ============================================================

table_efficiency = pd.DataFrame({
    "Method":
        combined[
            "_method_key"
        ],
    "Input":
        combined[
            "_input_key"
        ],
    "F1":
        combined[
            "F1"
        ]
        .round(
            3
        ),
    "IoU":
        combined[
            "IoU"
        ]
        .round(
            3
        ),
    "|Area error| (%)":
        combined[
            "|Area error| (%)"
        ]
        .round(
            2
        ),
    "Training time (min)":
        (
            combined[
                "training_time_sec"
            ]
            /
            60
        )
        .round(
            2
        ),
    "Inference (s/Mpix)":
        combined[
            "inference_sec_per_million_pixels"
        ]
        .round(
            2
        ),
    "Peak RAM (GB)":
        combined[
            "peak_process_ram_gb"
        ]
        .round(
            2
        ),
    "Peak GPU memory (GB)":
        combined[
            "peak_gpu_memory_gb"
        ]
        .round(
            2
        ),
    "Trainable params (M)":
        (
            combined[
                "trainable_parameters"
            ]
            /
            1_000_000
        )
        .round(
            2
        ),
    "Total params (M)":
        (
            combined[
                "total_parameters"
            ]
            /
            1_000_000
        )
        .round(
            2
        ),
})

# ============================================================
# SORT BY IoU
# ============================================================

table_efficiency = (
    table_efficiency
    .sort_values(
        "IoU",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

# ============================================================
# DISPLAY
# ============================================================

print(
    "\n"
    +
    "=" * 120
)

print(
    "TABLE 2. ACCURACY VS OPERATIONAL EFFICIENCY"
)

print(
    "=" * 120
)

display(
    table_efficiency
)

# ============================================================
# SAVE
# ============================================================

TABLE2_CSV = (
    RESULT_DIR
    /
    "Table_2_Accuracy_vs_Operational_Efficiency.csv"
)

table_efficiency.to_csv(
    TABLE2_CSV,
    index=False,
)

EXCEL_PATH = RESULT_DIR / "Final_Article_Tables.xlsx"
table_accuracy = pd.read_csv(RESULT_DIR / "publication_tables/Table_accuracy_publication.csv")
accuracy = pd.read_csv(UNIFIED_ACCURACY_CSV)

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl",
) as writer:
    table_accuracy.to_excel(
        writer,
        sheet_name="Table1_Accuracy",
        index=False,
    )
    table_efficiency.to_excel(
        writer,
        sheet_name="Table2_Accuracy_Efficiency",
        index=False,
    )
    accuracy.to_excel(
        writer,
        sheet_name="Per_Fire_Accuracy",
        index=False,
    )
    accuracy_summary.to_excel(
        writer,
        sheet_name="Accuracy_Summary",
        index=False,
    )
    efficiency.to_excel(
        writer,
        sheet_name="Efficiency_Source",
        index=False,
    )
    combined.to_excel(
        writer,
        sheet_name="Combined_Numeric",
        index=False,
    )

print(
    "\nUpdated:"
)
print(
    TABLE2_CSV
)
print(
    EXCEL_PATH
)

In [ ]:
# ============================================================
# TABLE 2 — OPERATIONAL / COMPUTATIONAL EFFICIENCY
# ============================================================

# Define the missing OUTPUT_DIR pointing to your results folder
OUTPUT_DIR = RESULT_DIR

# Uses:
# efficiency
# _method_key
# _input_key
#
# These were already standardized in the previous block.
# ============================================================


# ------------------------------------------------------------
# TRAINING RECORDS
# ------------------------------------------------------------

training_eff = (
    efficiency[
        efficiency["stage"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("training")
    ]
    .copy()
)


training_summary = (

    training_eff

    .groupby(
        [
            "_method_key",
            "_input_key",
        ],
        as_index=False,
    )

    .agg(

        training_time_sec=(
            "elapsed_sec",
            "mean",
        ),

        peak_process_ram_gb=(
            "peak_process_ram_gb",
            "max",
        ),

        mean_cpu_system_percent=(
            "mean_cpu_system_percent",
            "mean",
        ),

        peak_cpu_system_percent=(
            "peak_cpu_system_percent",
            "max",
        ),

        mean_gpu_util_percent=(
            "mean_gpu_util_percent",
            "mean",
        ),

        peak_gpu_memory_gb=(
            "peak_gpu_memory_gb",
            "max",
        ),

        trainable_parameters=(
            "trainable_parameters",
            "max",
        ),

        total_parameters=(
            "total_parameters",
            "max",
        ),
    )
)


# ------------------------------------------------------------
# PREDICTION RECORDS
# ------------------------------------------------------------

prediction_eff = (
    efficiency[
        efficiency["stage"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("prediction")
    ]
    .copy()
)


prediction_summary = (

    prediction_eff

    .groupby(
        [
            "_method_key",
            "_input_key",
        ],
        as_index=False,
    )

    .agg(

        inference_time_sec=(
            "elapsed_sec",
            "mean",
        ),

        inference_sec_per_million_pixels=(
            "inference_sec_per_million_pixels",
            "mean",
        ),

        inference_peak_ram_gb=(
            "peak_process_ram_gb",
            "max",
        ),

        inference_mean_cpu_percent=(
            "mean_cpu_system_percent",
            "mean",
        ),

        inference_peak_gpu_memory_gb=(
            "peak_gpu_memory_gb",
            "max",
        ),
    )
)


# ============================================================
# COMBINE TRAINING + INFERENCE EFFICIENCY
# ============================================================

efficiency_summary = (

    training_summary

    .merge(
        prediction_summary,
        on=[
            "_method_key",
            "_input_key",
        ],
        how="outer",
    )
)


# ============================================================
# ARTICLE-STYLE TABLE 2
# ============================================================

table_efficiency = pd.DataFrame({

    "Method":
        efficiency_summary[
            "_method_key"
        ],

    "Input":
        efficiency_summary[
            "_input_key"
        ],

    "Training time (min)":
        (
            efficiency_summary[
                "training_time_sec"
            ]
            /
            60
        )
        .round(2),

    "Inference (s/Mpix)":
        efficiency_summary[
            "inference_sec_per_million_pixels"
        ]
        .round(2),

    "Peak RAM (GB)":
        efficiency_summary[
            "peak_process_ram_gb"
        ]
        .round(2),

    "Peak GPU memory (GB)":
        efficiency_summary[
            "peak_gpu_memory_gb"
        ]
        .round(2),

    "Trainable params (M)":
        (
            efficiency_summary[
                "trainable_parameters"
            ]
            /
            1_000_000
        )
        .round(2),

    "Total params (M)":
        (
            efficiency_summary[
                "total_parameters"
            ]
            /
            1_000_000
        )
        .round(2),
})


# ============================================================
# ORDER METHODS
# ============================================================

METHOD_ORDER = {

    "RF": 0,

    "U-Net": 1,

    "Siamese": 2,

    "Prithvi — Frozen": 3,

    "Prithvi — LoRA": 4,
}


INPUT_ORDER = {

    "Post S2": 0,

    "Pre/Post S2": 1,

    "Pre/Post S2 + S1": 2,
}


table_efficiency[
    "_method_order"
] = (
    table_efficiency[
        "Method"
    ]
    .map(METHOD_ORDER)
)


table_efficiency[
    "_input_order"
] = (
    table_efficiency[
        "Input"
    ]
    .map(INPUT_ORDER)
)


table_efficiency = (

    table_efficiency

    .sort_values(
        [
            "_method_order",
            "_input_order",
        ]
    )

    .drop(
        columns=[
            "_method_order",
            "_input_order",
        ]
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# SAVE THE TWO SEPARATE ARTICLE TABLES
# ============================================================

TABLE1_CSV = (
    OUTPUT_DIR
    /
    "Table_1_Accuracy.csv"
)

TABLE2_CSV = (
    OUTPUT_DIR
    /
    "Table_2_Efficiency.csv"
)


# table_accuracy was already generated correctly
table_accuracy.to_csv(
    TABLE1_CSV,
    index=False,
)


table_efficiency.to_csv(
    TABLE2_CSV,
    index=False,
)


# ============================================================
# SAVE BOTH TO ONE EXCEL WORKBOOK
# ============================================================

EXCEL_PATH = (
    OUTPUT_DIR
    /
    "Final_Article_Tables.xlsx"
)


with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl",
) as writer:

    table_accuracy.to_excel(
        writer,
        sheet_name="Table1_Accuracy",
        index=False,
    )

    table_efficiency.to_excel(
        writer,
        sheet_name="Table2_Efficiency",
        index=False,
    )


# ============================================================
# DISPLAY TABLE 1
# ============================================================

print(
    "\n"
    +
    "=" * 120
)

print(
    "TABLE 1. ACCURACY OF BURNED-AREA MAPPING MODELS"
)

print(
    "=" * 120
)

display(
    table_accuracy
)


# ============================================================
# DISPLAY TABLE 2
# ============================================================

print(
    "\n"
    +
    "=" * 120
)

print(
    "TABLE 2. OPERATIONAL EFFICIENCY OF BURNED-AREA MAPPING MODELS"
)

print(
    "=" * 120
)

display(
    table_efficiency
)


print(
    "\nSaved:"
)

print(TABLE1_CSV)
print(TABLE2_CSV)
print(EXCEL_PATH)

In [ ]:
"""Run after full-fire inference in the burned-area modelling notebook."""
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from affine import Affine
import rasterio
from rasterio.features import rasterize
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT

required = ["RESULT_DIR", "get_complete_test_grid"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run this code in the modelling notebook after its evaluation "
        f"utilities are defined. Missing: {missing}"
    )

MAP_OUTPUT_DIR = Path(RESULT_DIR) / "publication_figures"
MAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REGISTRY_PATH = Path(
    globals().get("PREDICTION_REGISTRY_CSV",
                  Path(RESULT_DIR) / "prediction_registry.csv")
)
MAP_TEST_FIRES = ["fire_01", "fire_12"]
MAX_DISPLAY_SIDE = 900  # Display only; original prediction files are unchanged.

METHOD_LABELS = {
    "RF": "Random Forest", "Random Forest": "Random Forest",
    "U-Net": "U-Net", "UNet": "U-Net",
    "Siamese": "Siamese",
    "Prithvi-frozen": "Prithvi (frozen)",
    "Prithvi (frozen)": "Prithvi (frozen)",
    "Prithvi — Frozen": "Prithvi (frozen)",
    "Prithvi-lora": "Prithvi (LoRA)",
    "Prithvi (LoRA)": "Prithvi (LoRA)",
    "Prithvi — LoRA": "Prithvi (LoRA)",
}
METHOD_ORDER = {
    name: i for i, name in enumerate(
        ["Random Forest", "U-Net", "Siamese",
         "Prithvi (frozen)", "Prithvi (LoRA)"]
    )
}
CONFIG_LABELS = {
    "post_s2": "Post-fire S2",
    "prepost_s2": "Pre + post S2",
    "prepost_s2s1": "Pre + post S2 + S1",
}
CONFIG_ORDER = {name: i for i, name in enumerate(CONFIG_LABELS)}


def plot_test_fire_maps(fire_id):
    if not REGISTRY_PATH.exists():
        raise FileNotFoundError(
            f"Prediction registry not found: {REGISTRY_PATH}. "
            "Run full-fire prediction and registration first."
        )

    registry = pd.read_csv(REGISTRY_PATH)
    subset = registry.loc[
        registry["fire_id"].astype(str).eq(str(fire_id))
    ].copy()
    if subset.empty:
        print(f"{fire_id}: no registered predictions.")
        return None

    exists = subset["prediction_raster"].map(lambda x: Path(str(x)).is_file())
    for path in subset.loc[~exists, "prediction_raster"]:
        print(f"Skipping missing prediction: {path}")
    subset = subset.loc[exists].copy()
    if subset.empty:
        return None

    subset["label"] = subset["method"].map(METHOD_LABELS).fillna(subset["method"])
    subset = subset.drop_duplicates(["label", "input_config"], keep="last")
    subset["method_order"] = subset["label"].map(METHOD_ORDER).fillna(99)
    subset["config_order"] = subset["input_config"].map(CONFIG_ORDER).fillna(99)
    subset = subset.sort_values(["method_order", "config_order"])

    # Common geographical extent for every panel of this fire.
    grid = get_complete_test_grid(fire_id)
    ratio = min(1.0, MAX_DISPLAY_SIDE / max(grid["height"], grid["width"]))
    height = max(1, round(grid["height"] * ratio))
    width = max(1, round(grid["width"] * ratio))
    transform = grid["transform"] * Affine.scale(
        grid["width"] / width, grid["height"] / height
    )
    shape = (height, width)

    actual = rasterize(
        [(grid["fire_geom"], 1)], out_shape=shape,
        transform=transform, fill=0, dtype="uint8"
    )
    eligible = rasterize(
        [(grid["eligible_geom"], 1)], out_shape=shape,
        transform=transform, fill=0, dtype="uint8"
    ).astype(bool)
    panels = [("Actual (reference)", np.ma.array(actual, mask=~eligible))]

    for row in subset.itertuples(index=False):
        # Read directly onto the display grid to limit memory consumption.
        with rasterio.open(row.prediction_raster) as src:
            with WarpedVRT(
                src, crs=grid["crs"], transform=transform,
                width=width, height=height, nodata=255,
                resampling=Resampling.nearest,
            ) as vrt:
                arr = vrt.read(1, masked=True)

        invalid = np.ma.getmaskarray(arr) | ~eligible
        invalid |= ~np.isin(arr.data, [0, 1])
        arr = np.ma.array(arr.data, mask=invalid)
        config = CONFIG_LABELS.get(row.input_config, row.input_config)
        panels.append((f"{row.label}\n{config}", arr))

    cmap = ListedColormap(["#ffffff", "#b30000"])
    cmap.set_bad("#dddddd")
    ncols = min(5, len(panels))
    nrows = math.ceil(len(panels) / ncols)
    fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(3.2 * ncols, 4.0 * nrows + 0.7),
    squeeze=False,
    layout="compressed",
    )

    fig.get_layout_engine().set(
    w_pad=0.05,
    h_pad=0.08,
    wspace=0.0,
    hspace=0.05,
    rect=(0, 0.05, 1, 0.93),  # Leave room for the legend
    )
    for ax, (title, arr) in zip(axes.ravel(), panels):
        ax.imshow(arr, cmap=cmap, vmin=0, vmax=1,
                  interpolation="nearest", origin="upper")
        ax.set_title(title, fontsize=10, pad=7)
        ax.axis("off")
    for ax in axes.ravel()[len(panels):]:
        ax.axis("off")

    event_label = str(fire_id)
    if "get_target_fire_row" in globals():
        record = get_target_fire_row(fire_id)
        event_label = (
            f"{record.get('area_name', fire_id)}, "
            f"{record.get('country', '')} ({fire_id})"
        )
    fig.suptitle(f"Actual and predicted burned areas: {event_label}", fontsize=14)
    fig.legend(
        handles=[
            Patch(facecolor="#b30000", label="Burned"),
            Patch(facecolor="#ffffff", edgecolor="#777777", label="Unburned"),
            Patch(facecolor="#dddddd", label="Excluded / no data"),
        ],
        loc="lower center", ncol=3, frameon=False,
        bbox_to_anchor=(0.5, 0.01),
    )


    png = MAP_OUTPUT_DIR / f"Actual_predicted_burned_areas_{fire_id}.png"
    pdf = MAP_OUTPUT_DIR / f"Actual_predicted_burned_areas_{fire_id}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {png}\nSaved: {pdf}")
    return png, pdf


for test_fire in MAP_TEST_FIRES:
    plot_test_fire_maps(test_fire)


# Trade off